<a href="https://colab.research.google.com/github/stilyank01-create/Media_AI/blob/main/08_transformative_mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Media AI — Transformative Mechanism

**Notebook:** `08_transformative_mechanism.ipynb`
**Stage:** Transformative mechanism construction and validation
**Pipeline position:** Notebook 08

This notebook implements and validates the **transformative mechanism** of the Media AI architecture.

The preceding notebooks construct the prepared corpus, supervision framework, learned representation spaces, and restored factual, psychological, and social representation model. Notebook 08 extends this architecture from **representation of individual media observations** to **sequential transformation across media exposure**.

The central objective is to model how an incoming media representation interacts with an accumulated prior state. Rather than treating each article or media observation as an isolated prediction problem, the transformative mechanism preserves a causal representation of preceding exposure and determines how new information modifies that representation.

The architecture maintains three distinct transformative pathways:

* **factual transformation**, operating in the factual representation space;
* **psychological transformation**, operating in the psychological representation space;
* **social transformation**, operating in the social representation space.

For each pathway, the current representation is combined with the corresponding recurrent state through a dedicated transformative mechanism. The resulting transformation is controlled by a pathway-specific transformation weight:

$$
TW_F,\qquad TW_P,\qquad TW_S
$$

and updates the corresponding recurrent state:

$$
H_F,\qquad H_P,\qquad H_S
$$

where (H_F), (H_P), and (H_S) represent the accumulated factual, psychological, and social states respectively.

The mechanism is explicitly **causal**. At article position (t), transformation may depend on the current representation and information accumulated from positions preceding (t), but it must not depend on future articles. Recurrent states are therefore propagated only in sequence order and are reset at defined article-sequence boundaries.

The notebook focuses on:

* restoring and validating the inherited transformative architecture;
* constructing the factual, psychological, and social transformation pathways;
* validating the dimensional contracts between representation spaces and recurrent states;
* computing and inspecting the pathway-specific transformation weights (TW_F), (TW_P), and (TW_S);
* updating and propagating the recurrent states (H_F), (H_P), and (H_S);
* enforcing causal sequence processing and article-boundary reset behaviour;
* verifying that future information cannot influence earlier transformative states;
* distinguishing representation outputs from state transformations;
* validating deterministic and controlled forward behaviour before optimisation;
* defining the transformation outputs required by subsequent training and evaluation stages.

The transformative mechanism does not replace the learned factual, psychological, or social representations. These remain the observation-level representations produced by the preceding model architecture. Notebook 08 introduces an additional temporal layer in which those representations can modify persistent pathway-specific states.

Conceptually, the transition can be expressed as:

$$
R_t^{(k)}, H_{t-1}^{(k)}
;\longrightarrow;
T_t^{(k)}, TW_t^{(k)}
;\longrightarrow;
H_t^{(k)},
\qquad
k \in {F,P,S}
$$

where (R_t^{(k)}) is the current representation in pathway (k), (H_{t-1}^{(k)}) is the inherited state from preceding exposure, (T_t^{(k)}) is the pathway-specific transformation, (TW_t^{(k)}) controls its contribution, and (H_t^{(k)}) is the updated recurrent state.

This separation is fundamental to the Media AI design. A representation describes what a media observation contains or expresses; a transformation describes how that observation changes an accumulated state.

All model restoration, parameter-state changes, recurrent-state updates, sequence operations, checkpoint loading, and other computationally significant actions remain explicit and auditable. No future-context information is permitted to enter the causal transformation path.


## Block 1 — Environment, Imports and Deterministic Execution Contract

This block establishes the computational environment required for the transformative-mechanism stage of the Media AI pipeline.

Notebook 08 inherits the factual, psychological, and social representation architecture prepared and validated in the preceding notebook. Before any model checkpoint is restored or any recurrent transformation is executed, the notebook defines the common software dependencies, deterministic execution settings, numerical conventions, and runtime assumptions used throughout the transformative analysis.

The block is intentionally restricted to **environment initialisation**. It does not load model parameters, construct transformative modules, initialise recurrent states, process article sequences, or modify any inherited artefact.

The block performs the following operations:

* imports the Python, numerical, tabular, and PyTorch dependencies required by the notebook;
* establishes the project-wide random seed and deterministic random-number generators;
* configures deterministic PyTorch behaviour where supported;
* resolves the available computation device without altering model state;
* defines common numerical and tensor-display conventions used for subsequent validation;
* records the relevant runtime and library versions required for reproducibility;
* verifies that the execution environment is suitable for the later transformative-mechanism blocks.

Deterministic initialisation is particularly important in this notebook because later validation must distinguish genuine state transformation from variation introduced by stochastic model behaviour. Repeated evaluation under an unchanged model state, identical inputs, and identical inherited recurrent state should therefore produce reproducible outputs within the numerical guarantees of the selected execution backend.

No factual, psychological, or social transformation is performed in this block. In particular, the pathway-specific transformation weights

$$
TW_F,\qquad TW_P,\qquad TW_S
$$

and recurrent states

$$
H_F,\qquad H_P,\qquad H_S
$$

are not yet instantiated or updated.

This separation ensures that environment configuration remains independent from model restoration and from the causal state-transition logic introduced in the subsequent blocks.


In [1]:
# =============================================================================
# Media AI — Notebook 08, Block 1
# Environment, Imports and Deterministic Execution Contract
# =============================================================================

import os
import sys
import platform
import random

import numpy as np
import pandas as pd
import torch


# -----------------------------------------------------------------------------
# Deterministic execution
# -----------------------------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
    DETERMINISTIC_ALGORITHMS = True
except (RuntimeError, NotImplementedError):
    DETERMINISTIC_ALGORITHMS = False


# -----------------------------------------------------------------------------
# Runtime device
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")


# -----------------------------------------------------------------------------
# Numerical and display conventions
# -----------------------------------------------------------------------------

DEFAULT_DTYPE = torch.float32

torch.set_default_dtype(DEFAULT_DTYPE)
torch.set_printoptions(
    precision=6,
    sci_mode=False,
    linewidth=120,
)

np.set_printoptions(
    precision=6,
    suppress=True,
    linewidth=120,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


# -----------------------------------------------------------------------------
# Environment validation
# -----------------------------------------------------------------------------

print("=" * 72)
print("Media AI — Notebook 08, Block 1")
print("Environment, Imports and Deterministic Execution Contract")
print("=" * 72)

print(f"Python version              : {sys.version.split()[0]}")
print(f"Platform                    : {platform.platform()}")
print(f"NumPy version               : {np.__version__}")
print(f"Pandas version              : {pd.__version__}")
print(f"PyTorch version             : {torch.__version__}")
print(f"Execution device            : {DEVICE}")
print(f"Default tensor dtype        : {torch.get_default_dtype()}")
print(f"Project seed                : {SEED}")
print(f"Deterministic algorithms    : {DETERMINISTIC_ALGORITHMS}")

if DEVICE.type == "cuda":
    print(f"CUDA version                : {torch.version.cuda}")
    print(f"CUDA device                 : {torch.cuda.get_device_name(0)}")

print("-" * 72)
print("Environment initialisation completed successfully.")
print("No model, checkpoint, recurrent state, or transformation was loaded.")
print("=" * 72)

Media AI — Notebook 08, Block 1
Environment, Imports and Deterministic Execution Contract
Python version              : 3.13.15
Platform                    : Linux-6.6.122+-x86_64-with-glibc2.35
NumPy version               : 2.1.3
Pandas version              : 2.2.3
PyTorch version             : 2.11.0+cpu
Execution device            : cpu
Default tensor dtype        : torch.float32
Project seed                : 42
Deterministic algorithms    : True
------------------------------------------------------------------------
Environment initialisation completed successfully.
No model, checkpoint, recurrent state, or transformation was loaded.


## Block 2 — Inherited Artefact and Path Contract

This block establishes the filesystem and artefact contract connecting Notebook 08 to the completed representation-model stage.

Notebook 08 does not reconstruct or retrain the factual, psychological, and social representation architecture from first principles. Instead, it inherits the validated model state and associated metadata produced by the preceding notebook and uses those artefacts as the fixed starting point for construction and validation of the transformative mechanism.

The block therefore resolves the required project paths and verifies the availability of the inherited artefacts before any model restoration or transformative computation is permitted.

The block performs the following operations:

* resolves the project, data, model, checkpoint, and Notebook 08 output directories;
* identifies the canonical inherited checkpoint and associated metadata produced by Notebook 07;
* verifies that every required inherited artefact exists and is accessible;
* records the resolved artefact locations used by subsequent blocks;
* creates Notebook 08 output directories where required;
* establishes explicit input/output boundaries between the representation and transformative stages;
* fails explicitly if a required inherited artefact cannot be located.

The inherited checkpoint is treated as an **immutable upstream artefact**. This block may inspect its location and filesystem-level properties, but it does not load model parameters, alter checkpoint contents, instantiate the inherited architecture, or change any learned parameter state.

Similarly, no recurrent transformative state is created in this block. The pathway-specific states

$$
H_F,\qquad H_P,\qquad H_S
$$

and their corresponding transformation weights

$$
TW_F,\qquad TW_P,\qquad TW_S
$$

remain undefined at runtime until the transformative architecture is explicitly introduced.

This separation provides a reproducible handover boundary:

$$
\text{Notebook 07 validated artefacts}
;\longrightarrow;
\text{Notebook 08 inherited artefact contract}
;\longrightarrow;
\text{controlled model restoration}
$$

Successful completion of this block confirms only that the required upstream artefacts and downstream storage locations are available. It does **not** establish that the inherited checkpoint is architecturally compatible, that its parameters have been restored, or that the transformative mechanism is operational. Those conditions are validated explicitly in subsequent blocks.


In [2]:
# =============================================================================
# Media AI — Notebook 08, Block 2
# Inherited Artefact and Drive Handover Contract
# =============================================================================

from datetime import datetime, timezone

import io
import json

import torch

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_2 = 2

NOTEBOOK_08_BLOCK_2_NAME = (
    "Inherited Artefact and Drive Handover Contract"
)

NOTEBOOK_08_BLOCK_2_VERSION = "1.0"

BLOCK_2_EXECUTED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# =============================================================================
# Block 1 dependency contract
# =============================================================================

required_block_2_objects = [
    "SEED",
    "DEVICE",
    "DETERMINISTIC_ALGORITHMS",
]


missing_block_2_objects = [
    object_name
    for object_name
    in required_block_2_objects
    if object_name not in globals()
]


if missing_block_2_objects:

    raise NameError(
        "Notebook 08 Block 2 requires the completed "
        "Block 1 environment contract. "
        f"Missing: {missing_block_2_objects}"
    )


# =============================================================================
# Canonical Notebook 07 -> Notebook 08 handover references
# =============================================================================
#
# These Drive file IDs were created and read back successfully by
# Notebook 07 Block 10.
# =============================================================================

NOTEBOOK_08_REPRESENTATION_CHECKPOINT_DRIVE_FILE_ID = (
    "19vnqvir-mQ385_6zjjuqwMv924IAlYdG"
)

NOTEBOOK_08_REPRESENTATION_METADATA_DRIVE_FILE_ID = (
    "1Dx7Un82PBvYQoAERuHXNpWcpbTO_OtZp"
)


NOTEBOOK_08_REPRESENTATION_CHECKPOINT_FILENAME = (
    "notebook_07_final_checkpoint.pt"
)

NOTEBOOK_08_REPRESENTATION_METADATA_FILENAME = (
    "notebook_07_final_metadata.json"
)


# =============================================================================
# Expected inherited contract
# =============================================================================

BLOCK_2_EXPECTED_CHECKPOINT_TYPE = (
    "validated_representation_model_handover"
)

BLOCK_2_EXPECTED_METADATA_TYPE = (
    "media_ai_notebook_07_to_08_representation_handover"
)

BLOCK_2_EXPECTED_SOURCE_NOTEBOOK = (
    "07_social_path"
)

BLOCK_2_EXPECTED_TARGET_NOTEBOOK = (
    "08_transformative_mechanism"
)

BLOCK_2_EXPECTED_TOTAL_PARAMETER_COUNT = (
    894003
)

BLOCK_2_EXPECTED_STATE_DICT_KEYS = {
    "backbone",
    "global_confluent",
    "psychological_head",
    "factual_head",
    "social_head",
}

BLOCK_2_EXPECTED_REPRESENTATION_SPACES = {
    "psychological",
    "factual",
    "social",
}


# =============================================================================
# Restricted Google Drive API access
# =============================================================================
#
# This uses the Google Drive API rather than mounting the user's Drive
# filesystem. No drive.mount() call is made.
# =============================================================================

try:

    auth.authenticate_user()

    NOTEBOOK_08_DRIVE_SERVICE = build(
        "drive",
        "v3",
        cache_discovery=False,
    )


except Exception as error:

    raise RuntimeError(
        "Notebook 08 could not initialise the Google Drive API service "
        "required for the persisted Notebook 07 handover."
    ) from error


if NOTEBOOK_08_DRIVE_SERVICE is None:

    raise RuntimeError(
        "Notebook 08 Google Drive API service was not created."
    )


# =============================================================================
# Drive metadata helper
# =============================================================================

def block_2_get_drive_metadata(
    drive_service,
    file_id,
):

    return (
        drive_service.files()
        .get(
            fileId=
                file_id,

            fields=
                "id,name,mimeType,size,modifiedTime,parents",
        )
        .execute()
    )


# =============================================================================
# Drive download helper
# =============================================================================

def block_2_download_drive_bytes(
    drive_service,
    file_id,
):

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    complete = False


    while not complete:

        _, complete = (
            downloader.next_chunk()
        )


    buffer.seek(
        0
    )


    return buffer.read()


# =============================================================================
# Resolve persisted Drive metadata
# =============================================================================

BLOCK_2_CHECKPOINT_DRIVE_METADATA = (
    block_2_get_drive_metadata(
        NOTEBOOK_08_DRIVE_SERVICE,
        NOTEBOOK_08_REPRESENTATION_CHECKPOINT_DRIVE_FILE_ID,
    )
)


BLOCK_2_METADATA_DRIVE_METADATA = (
    block_2_get_drive_metadata(
        NOTEBOOK_08_DRIVE_SERVICE,
        NOTEBOOK_08_REPRESENTATION_METADATA_DRIVE_FILE_ID,
    )
)


# =============================================================================
# Drive filename validation
# =============================================================================

BLOCK_2_CHECKPOINT_FILENAME_VALID = (
    BLOCK_2_CHECKPOINT_DRIVE_METADATA.get(
        "name"
    )
    ==
    NOTEBOOK_08_REPRESENTATION_CHECKPOINT_FILENAME
)


BLOCK_2_METADATA_FILENAME_VALID = (
    BLOCK_2_METADATA_DRIVE_METADATA.get(
        "name"
    )
    ==
    NOTEBOOK_08_REPRESENTATION_METADATA_FILENAME
)


if not all(
    [
        BLOCK_2_CHECKPOINT_FILENAME_VALID,
        BLOCK_2_METADATA_FILENAME_VALID,
    ]
):

    raise RuntimeError(
        "Notebook 07 handover Drive file names do not match "
        "the canonical Notebook 08 inheritance contract."
    )


# =============================================================================
# Download inherited checkpoint
# =============================================================================

BLOCK_2_CHECKPOINT_BYTES = (
    block_2_download_drive_bytes(
        NOTEBOOK_08_DRIVE_SERVICE,
        NOTEBOOK_08_REPRESENTATION_CHECKPOINT_DRIVE_FILE_ID,
    )
)


if not BLOCK_2_CHECKPOINT_BYTES:

    raise RuntimeError(
        "Persisted Notebook 07 representation checkpoint is empty."
    )


try:

    BLOCK_2_INHERITED_CHECKPOINT = torch.load(
        io.BytesIO(
            BLOCK_2_CHECKPOINT_BYTES
        ),
        map_location=
            "cpu",
        weights_only=
            False,
    )


except TypeError:

    BLOCK_2_INHERITED_CHECKPOINT = torch.load(
        io.BytesIO(
            BLOCK_2_CHECKPOINT_BYTES
        ),
        map_location=
            "cpu",
    )


if not isinstance(
    BLOCK_2_INHERITED_CHECKPOINT,
    dict,
):

    raise TypeError(
        "Persisted Notebook 07 checkpoint must be dictionary-like."
    )


# =============================================================================
# Download inherited metadata
# =============================================================================

BLOCK_2_METADATA_BYTES = (
    block_2_download_drive_bytes(
        NOTEBOOK_08_DRIVE_SERVICE,
        NOTEBOOK_08_REPRESENTATION_METADATA_DRIVE_FILE_ID,
    )
)


if not BLOCK_2_METADATA_BYTES:

    raise RuntimeError(
        "Persisted Notebook 07 handover metadata is empty."
    )


try:

    BLOCK_2_INHERITED_METADATA = json.loads(
        BLOCK_2_METADATA_BYTES.decode(
            "utf-8"
        )
    )


except (
    UnicodeDecodeError,
    json.JSONDecodeError,
) as error:

    raise RuntimeError(
        "Persisted Notebook 07 metadata is not valid UTF-8 JSON."
    ) from error


if not isinstance(
    BLOCK_2_INHERITED_METADATA,
    dict,
):

    raise TypeError(
        "Persisted Notebook 07 metadata must be a JSON object."
    )


# =============================================================================
# Checkpoint contract validation
# =============================================================================

BLOCK_2_CHECKPOINT_TYPE_VALID = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "checkpoint_type"
    )
    ==
    BLOCK_2_EXPECTED_CHECKPOINT_TYPE
)


BLOCK_2_CHECKPOINT_SOURCE_VALID = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "source_notebook"
    )
    ==
    BLOCK_2_EXPECTED_SOURCE_NOTEBOOK
)


BLOCK_2_CHECKPOINT_TARGET_VALID = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "target_notebook"
    )
    ==
    BLOCK_2_EXPECTED_TARGET_NOTEBOOK
)


BLOCK_2_CHECKPOINT_SEED_VALID = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "seed"
    )
    ==
    SEED
)


BLOCK_2_CHECKPOINT_PARAMETER_COUNT_VALID = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "total_parameter_count"
    )
    ==
    BLOCK_2_EXPECTED_TOTAL_PARAMETER_COUNT
)


BLOCK_2_CHECKPOINT_STATE_DICT_KEYS = set(
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "state_dicts",
        {},
    ).keys()
)


BLOCK_2_CHECKPOINT_STATE_DICT_KEYS_VALID = (
    BLOCK_2_CHECKPOINT_STATE_DICT_KEYS
    ==
    BLOCK_2_EXPECTED_STATE_DICT_KEYS
)


BLOCK_2_CHECKPOINT_REPRESENTATION_SPACES_VALID = (
    set(
        BLOCK_2_INHERITED_CHECKPOINT.get(
            "representation_spaces",
            [],
        )
    )
    ==
    BLOCK_2_EXPECTED_REPRESENTATION_SPACES
)


BLOCK_2_CHECKPOINT_ARCHITECTURAL_BOUNDARY = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "architectural_boundary",
        {},
    )
)


BLOCK_2_CHECKPOINT_TRANSFORMATIVE_STATE_ABSENT = all(
    [
        (
            BLOCK_2_CHECKPOINT_ARCHITECTURAL_BOUNDARY.get(
                "transformative_mechanism_instantiated"
            )
            is False
        ),

        (
            BLOCK_2_CHECKPOINT_ARCHITECTURAL_BOUNDARY.get(
                "transformation_weights_instantiated"
            )
            is False
        ),

        (
            BLOCK_2_CHECKPOINT_ARCHITECTURAL_BOUNDARY.get(
                "recurrent_states_instantiated"
            )
            is False
        ),

        (
            BLOCK_2_CHECKPOINT_ARCHITECTURAL_BOUNDARY.get(
                "future_context_permitted"
            )
            is False
        ),
    ]
)


BLOCK_2_CHECKPOINT_CONTRACT_VALID = all(
    [
        BLOCK_2_CHECKPOINT_TYPE_VALID,
        BLOCK_2_CHECKPOINT_SOURCE_VALID,
        BLOCK_2_CHECKPOINT_TARGET_VALID,
        BLOCK_2_CHECKPOINT_SEED_VALID,
        BLOCK_2_CHECKPOINT_PARAMETER_COUNT_VALID,
        BLOCK_2_CHECKPOINT_STATE_DICT_KEYS_VALID,
        BLOCK_2_CHECKPOINT_REPRESENTATION_SPACES_VALID,
        BLOCK_2_CHECKPOINT_TRANSFORMATIVE_STATE_ABSENT,
    ]
)


if not BLOCK_2_CHECKPOINT_CONTRACT_VALID:

    raise RuntimeError(
        "Persisted Notebook 07 checkpoint failed the "
        "Notebook 08 inherited checkpoint contract."
    )


# =============================================================================
# Metadata contract validation
# =============================================================================

BLOCK_2_METADATA_TYPE_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "metadata_type"
    )
    ==
    BLOCK_2_EXPECTED_METADATA_TYPE
)


BLOCK_2_METADATA_SOURCE_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "source_notebook"
    )
    ==
    BLOCK_2_EXPECTED_SOURCE_NOTEBOOK
)


BLOCK_2_METADATA_TARGET_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "target_notebook"
    )
    ==
    BLOCK_2_EXPECTED_TARGET_NOTEBOOK
)


BLOCK_2_METADATA_CHECKPOINT_FILENAME_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "checkpoint_filename"
    )
    ==
    NOTEBOOK_08_REPRESENTATION_CHECKPOINT_FILENAME
)


BLOCK_2_METADATA_CHECKPOINT_TYPE_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "checkpoint_type"
    )
    ==
    BLOCK_2_EXPECTED_CHECKPOINT_TYPE
)


BLOCK_2_METADATA_SEED_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "seed"
    )
    ==
    SEED
)


BLOCK_2_METADATA_PARAMETER_COUNT_VALID = (
    BLOCK_2_INHERITED_METADATA.get(
        "total_parameter_count"
    )
    ==
    BLOCK_2_EXPECTED_TOTAL_PARAMETER_COUNT
)


BLOCK_2_METADATA_REPRESENTATION_SPACES_VALID = (
    set(
        BLOCK_2_INHERITED_METADATA.get(
            "representation_spaces",
            [],
        )
    )
    ==
    BLOCK_2_EXPECTED_REPRESENTATION_SPACES
)


BLOCK_2_NOTEBOOK_08_STARTING_STATE = (
    BLOCK_2_INHERITED_METADATA.get(
        "notebook_08_starting_state",
        {},
    )
)


BLOCK_2_REPRESENTATION_MODEL_AVAILABLE = (
    BLOCK_2_NOTEBOOK_08_STARTING_STATE.get(
        "representation_model_available"
    )
    is True
)


BLOCK_2_NOTEBOOK_08_READY = (
    BLOCK_2_NOTEBOOK_08_STARTING_STATE.get(
        "ready_to_initialise"
    )
    is True
)


BLOCK_2_METADATA_TRANSFORMATIVE_STATE_ABSENT = all(
    [
        (
            BLOCK_2_NOTEBOOK_08_STARTING_STATE.get(
                "transformative_mechanism_instantiated"
            )
            is False
        ),

        (
            BLOCK_2_NOTEBOOK_08_STARTING_STATE.get(
                "transformation_weights_instantiated"
            )
            is False
        ),

        (
            BLOCK_2_NOTEBOOK_08_STARTING_STATE.get(
                "recurrent_states_instantiated"
            )
            is False
        ),

        (
            BLOCK_2_NOTEBOOK_08_STARTING_STATE.get(
                "future_context_permitted"
            )
            is False
        ),
    ]
)


BLOCK_2_METADATA_CONTRACT_VALID = all(
    [
        BLOCK_2_METADATA_TYPE_VALID,
        BLOCK_2_METADATA_SOURCE_VALID,
        BLOCK_2_METADATA_TARGET_VALID,
        BLOCK_2_METADATA_CHECKPOINT_FILENAME_VALID,
        BLOCK_2_METADATA_CHECKPOINT_TYPE_VALID,
        BLOCK_2_METADATA_SEED_VALID,
        BLOCK_2_METADATA_PARAMETER_COUNT_VALID,
        BLOCK_2_METADATA_REPRESENTATION_SPACES_VALID,
        BLOCK_2_REPRESENTATION_MODEL_AVAILABLE,
        BLOCK_2_NOTEBOOK_08_READY,
        BLOCK_2_METADATA_TRANSFORMATIVE_STATE_ABSENT,
    ]
)


if not BLOCK_2_METADATA_CONTRACT_VALID:

    raise RuntimeError(
        "Persisted Notebook 07 metadata failed the "
        "Notebook 08 inherited metadata contract."
    )


# =============================================================================
# Cross-artefact consistency validation
# =============================================================================

BLOCK_2_PARAMETER_COUNTS_MATCH = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "parameter_counts"
    )
    ==
    {
        component_name:
            component_metadata.get(
                "parameters"
            )

        for (
            component_name,
            component_metadata,
        ) in BLOCK_2_INHERITED_METADATA.get(
            "components",
            {},
        ).items()
    }
)


BLOCK_2_ARCHITECTURE_CLASSES_MATCH = (
    BLOCK_2_INHERITED_CHECKPOINT.get(
        "architecture"
    )
    ==
    {
        component_name:
            component_metadata.get(
                "class"
            )

        for (
            component_name,
            component_metadata,
        ) in BLOCK_2_INHERITED_METADATA.get(
            "components",
            {},
        ).items()
    }
)


BLOCK_2_CROSS_ARTEFACT_CONTRACT_VALID = all(
    [
        BLOCK_2_PARAMETER_COUNTS_MATCH,
        BLOCK_2_ARCHITECTURE_CLASSES_MATCH,
    ]
)


if not BLOCK_2_CROSS_ARTEFACT_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 07 checkpoint and metadata are mutually inconsistent."
    )


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_08_BLOCK_2_MODEL_INSTANTIATED = False

NOTEBOOK_08_BLOCK_2_STATE_DICT_LOADED_INTO_MODEL = False

NOTEBOOK_08_BLOCK_2_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_2_TRANSFORMATIVE_MECHANISM_INSTANTIATED = False

NOTEBOOK_08_BLOCK_2_TRANSFORMATION_WEIGHTS_INSTANTIATED = False

NOTEBOOK_08_BLOCK_2_RECURRENT_STATE_INSTANTIATED = False

NOTEBOOK_08_BLOCK_2_PARAMETER_UPDATE_EXECUTED = False


# =============================================================================
# Notebook 08 inherited exports
# =============================================================================

NOTEBOOK_08_INHERITED_CHECKPOINT = (
    BLOCK_2_INHERITED_CHECKPOINT
)

NOTEBOOK_08_INHERITED_METADATA = (
    BLOCK_2_INHERITED_METADATA
)

NOTEBOOK_08_INHERITED_STATE_DICTS = (
    NOTEBOOK_08_INHERITED_CHECKPOINT[
        "state_dicts"
    ]
)

NOTEBOOK_08_INHERITED_ARCHITECTURE = (
    NOTEBOOK_08_INHERITED_CHECKPOINT[
        "architecture"
    ]
)

NOTEBOOK_08_INHERITED_PARAMETER_COUNTS = (
    NOTEBOOK_08_INHERITED_CHECKPOINT[
        "parameter_counts"
    ]
)

NOTEBOOK_08_INHERITED_TOTAL_PARAMETER_COUNT = (
    NOTEBOOK_08_INHERITED_CHECKPOINT[
        "total_parameter_count"
    ]
)


NOTEBOOK_08_REPRESENTATION_HANDOVER_READY = all(
    [
        BLOCK_2_CHECKPOINT_CONTRACT_VALID,
        BLOCK_2_METADATA_CONTRACT_VALID,
        BLOCK_2_CROSS_ARTEFACT_CONTRACT_VALID,
        BLOCK_2_NOTEBOOK_08_READY,
    ]
)


# =============================================================================
# Final Block 2 validation
# =============================================================================

NOTEBOOK_08_BLOCK_2_VALID = all(
    [
        BLOCK_2_CHECKPOINT_FILENAME_VALID,
        BLOCK_2_METADATA_FILENAME_VALID,
        BLOCK_2_CHECKPOINT_CONTRACT_VALID,
        BLOCK_2_METADATA_CONTRACT_VALID,
        BLOCK_2_CROSS_ARTEFACT_CONTRACT_VALID,
        NOTEBOOK_08_REPRESENTATION_HANDOVER_READY,

        not NOTEBOOK_08_BLOCK_2_MODEL_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_2_STATE_DICT_LOADED_INTO_MODEL,
        not NOTEBOOK_08_BLOCK_2_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_2_TRANSFORMATIVE_MECHANISM_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_2_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_2_RECURRENT_STATE_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_2_PARAMETER_UPDATE_EXECUTED,
    ]
)


if not NOTEBOOK_08_BLOCK_2_VALID:

    raise RuntimeError(
        "Notebook 08 Block 2 inherited artefact validation failed."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_2_COMPLETE = True


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 2: "
    "Inherited Artefact and Drive Handover Contract"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_2_VERSION}"
)

print("-" * 72)

print(
    f"Checkpoint Drive file ID    : "
    f"{NOTEBOOK_08_REPRESENTATION_CHECKPOINT_DRIVE_FILE_ID}"
)

print(
    f"Checkpoint filename         : "
    f"{BLOCK_2_CHECKPOINT_DRIVE_METADATA.get('name')}"
)

print(
    f"Checkpoint bytes            : "
    f"{len(BLOCK_2_CHECKPOINT_BYTES):,}"
)

print("-" * 72)

print(
    f"Metadata Drive file ID      : "
    f"{NOTEBOOK_08_REPRESENTATION_METADATA_DRIVE_FILE_ID}"
)

print(
    f"Metadata filename           : "
    f"{BLOCK_2_METADATA_DRIVE_METADATA.get('name')}"
)

print(
    f"Metadata bytes              : "
    f"{len(BLOCK_2_METADATA_BYTES):,}"
)

print("-" * 72)

print(
    f"Source notebook             : "
    f"{BLOCK_2_INHERITED_CHECKPOINT.get('source_notebook')}"
)

print(
    f"Target notebook             : "
    f"{BLOCK_2_INHERITED_CHECKPOINT.get('target_notebook')}"
)

print(
    f"Checkpoint type             : "
    f"{BLOCK_2_INHERITED_CHECKPOINT.get('checkpoint_type')}"
)

print(
    f"Total inherited parameters  : "
    f"{NOTEBOOK_08_INHERITED_TOTAL_PARAMETER_COUNT:,}"
)

print(
    f"Representation spaces       : "
    f"{sorted(BLOCK_2_EXPECTED_REPRESENTATION_SPACES)}"
)

print("-" * 72)

print(
    f"Checkpoint contract valid   : "
    f"{BLOCK_2_CHECKPOINT_CONTRACT_VALID}"
)

print(
    f"Metadata contract valid     : "
    f"{BLOCK_2_METADATA_CONTRACT_VALID}"
)

print(
    f"Cross-artefact valid        : "
    f"{BLOCK_2_CROSS_ARTEFACT_CONTRACT_VALID}"
)

print(
    f"Representation model avail. : "
    f"{BLOCK_2_REPRESENTATION_MODEL_AVAILABLE}"
)

print(
    f"Notebook 08 ready           : "
    f"{BLOCK_2_NOTEBOOK_08_READY}"
)

print(
    f"Transformative state absent : "
    f"{BLOCK_2_METADATA_TRANSFORMATIVE_STATE_ABSENT}"
)

print("-" * 72)

print(
    f"Model instantiated          : "
    f"{NOTEBOOK_08_BLOCK_2_MODEL_INSTANTIATED}"
)

print(
    f"State loaded into model     : "
    f"{NOTEBOOK_08_BLOCK_2_STATE_DICT_LOADED_INTO_MODEL}"
)

print(
    f"Forward pass executed       : "
    f"{NOTEBOOK_08_BLOCK_2_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transformative instantiated : "
    f"{NOTEBOOK_08_BLOCK_2_TRANSFORMATIVE_MECHANISM_INSTANTIATED}"
)

print(
    f"Recurrent state instantiated: "
    f"{NOTEBOOK_08_BLOCK_2_RECURRENT_STATE_INSTANTIATED}"
)

print("-" * 72)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_2_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_2_COMPLETE}"
)

print("=" * 72)

print(
    "Notebook 07 representation-model handover was retrieved "
    "and validated successfully through the Google Drive API."
)

print(
    "No Google Drive filesystem mount was used."
)

print(
    "No model module was instantiated and no persisted parameter "
    "state was loaded into a runtime model."
)

print(
    "No transformative mechanism, transformation weight, "
    "recurrent state, forward pass, or parameter update was executed."
)

print(
    "Notebook 08 may proceed to controlled inherited-architecture "
    "restoration in the next block."
)

print("=" * 72)

Media AI — Notebook 08, Block 2: Inherited Artefact and Drive Handover Contract
Block version               : 1.0
------------------------------------------------------------------------
Checkpoint Drive file ID    : 19vnqvir-mQ385_6zjjuqwMv924IAlYdG
Checkpoint filename         : notebook_07_final_checkpoint.pt
Checkpoint bytes            : 3,585,179
------------------------------------------------------------------------
Metadata Drive file ID      : 1Dx7Un82PBvYQoAERuHXNpWcpbTO_OtZp
Metadata filename           : notebook_07_final_metadata.json
Metadata bytes              : 1,483
------------------------------------------------------------------------
Source notebook             : 07_social_path
Target notebook             : 08_transformative_mechanism
Checkpoint type             : validated_representation_model_handover
Total inherited parameters  : 894,003
Representation spaces       : ['factual', 'psychological', 'social']
-----------------------------------------------------------

## Block 3 — Controlled Inherited-Architecture Restoration and State-Dict Validation

This block restores the validated representation-model architecture inherited from Notebook 07 and verifies that the persisted parameter state can be reproduced exactly within the Notebook 08 runtime.

The restoration is deliberately restricted to the representation system established upstream. No transformative mechanism, recurrent transformation state, transformation weight, or temporal update rule is introduced at this stage.

The inherited representation model consists of five validated components:

- the semantic backbone;
- the global confluent projection;
- the psychological representation head;
- the factual representation head;
- the social representation head.

Together, these components contain **894,003 persisted parameters** and define the three inherited representation spaces available to the transformative mechanism:

$$
\mathbf{r}^{(\mathrm{psych})}_t \in \mathbb{R}^{34},
\qquad
\mathbf{r}^{(\mathrm{fact})}_t \in \mathbb{R}^{10},
\qquad
\mathbf{r}^{(\mathrm{social})}_t \in \mathbb{R}^{7}.
$$

The block reconstructs the corresponding Notebook 07 module classes using the validated architectural dimensions encoded by the inherited handover contract. The persisted state dictionaries retrieved in Block 2 are then loaded into these runtime modules under strict parameter-key matching.

For each inherited component, restoration must preserve:

- the validated module class and layer structure;
- the complete persisted state-dict key set;
- tensor shapes and parameter dimensions;
- the total parameter count;
- deterministic numerical configuration;
- the absence of missing or unexpected persisted parameters.

After restoration, all inherited representation-model parameters are placed on the execution device established in Block 1 and frozen:

$$
\mathrm{requires\_grad}(\theta_{\mathrm{repr}})=\mathrm{False}.
$$

The restored modules are placed in evaluation mode so that their behaviour remains consistent with the validated Notebook 07 handover state. This is particularly important for components containing dropout or other training-dependent behaviour.

The restoration contract is therefore:

$$
\theta_{\mathrm{repr}}^{(08)}
=
\theta_{\mathrm{repr}}^{(07)},
$$

where $\theta_{\mathrm{repr}}^{(07)}$ denotes the persisted validated representation-model parameters inherited from Notebook 07 and $\theta_{\mathrm{repr}}^{(08)}$ denotes their restored Notebook 08 runtime state.

Exact restoration is verified at the state-dict level before the inherited model is exposed to subsequent transformative computation.

This block does **not** execute a representation forward pass. It therefore does not yet produce psychological, factual, or social representation vectors from input data. Forward-path validation remains separate from parameter restoration so that architectural reconstruction and numerical execution can be audited independently.

At completion, Notebook 08 must therefore contain a validated and frozen runtime copy of the complete inherited representation model while maintaining the following boundary conditions:

- no transformative mechanism has been instantiated;
- no transformation weights have been created;
- no recurrent transformation state exists;
- no forward transformation has been executed;
- no inherited parameter has been updated;
- no future-context information has been introduced.

The resulting frozen representation model forms the controlled upstream basis from which the transformative mechanism can subsequently be defined and validated.

In [3]:
# =============================================================================
# Media AI — Notebook 08, Block 3
# Controlled Inherited-Architecture Restoration and State-Dict Validation
# =============================================================================

import torch
import torch.nn as nn


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_3 = 3

NOTEBOOK_08_BLOCK_3_NAME = (
    "Controlled Inherited-Architecture Restoration "
    "and State-Dict Validation"
)

NOTEBOOK_08_BLOCK_3_VERSION = "1.0"


# =============================================================================
# Dependency checks
# =============================================================================

required_block_3_objects = [
    # -------------------------------------------------------------------------
    # Block 1 — deterministic runtime
    # -------------------------------------------------------------------------
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Block 2 — validated Notebook 07 handover
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_2_COMPLETE",
    "NOTEBOOK_08_BLOCK_2_VALID",
    "NOTEBOOK_08_REPRESENTATION_HANDOVER_READY",
    "NOTEBOOK_08_INHERITED_CHECKPOINT",
    "NOTEBOOK_08_INHERITED_METADATA",
    "NOTEBOOK_08_INHERITED_STATE_DICTS",
    "NOTEBOOK_08_INHERITED_ARCHITECTURE",
    "NOTEBOOK_08_INHERITED_PARAMETER_COUNTS",
    "NOTEBOOK_08_INHERITED_TOTAL_PARAMETER_COUNT",
]


missing_block_3_objects = [
    object_name
    for object_name
    in required_block_3_objects
    if object_name not in globals()
]


if missing_block_3_objects:

    raise NameError(
        "Notebook 08 Block 3 prerequisites are not initialised. "
        f"Missing: {missing_block_3_objects}"
    )


if not NOTEBOOK_08_BLOCK_2_COMPLETE:

    raise RuntimeError(
        "Notebook 08 Block 2 must be complete before Block 3."
    )


if not NOTEBOOK_08_BLOCK_2_VALID:

    raise RuntimeError(
        "Notebook 08 Block 2 must be valid before Block 3."
    )


if not NOTEBOOK_08_REPRESENTATION_HANDOVER_READY:

    raise RuntimeError(
        "Notebook 08 inherited representation handover is not ready."
    )


# =============================================================================
# Deterministic inherited architecture contract
# =============================================================================

NOTEBOOK_08_TEXTUAL_REPRESENTATION_DIM = 768

NOTEBOOK_08_BACKBONE_INTERFACE_DIM = 768

NOTEBOOK_08_GLOBAL_CONFLUENT_DIM = 256

NOTEBOOK_08_REPRESENTATION_HIDDEN_DIM = 128

NOTEBOOK_08_PSYCHOLOGICAL_REPRESENTATION_DIM = 34

NOTEBOOK_08_FACTUAL_REPRESENTATION_DIM = 10

NOTEBOOK_08_SOCIAL_REPRESENTATION_DIM = 7

NOTEBOOK_08_GLOBAL_CONFLUENT_DROPOUT = 0.10

NOTEBOOK_08_REPRESENTATION_HEAD_DROPOUT = 0.10


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_08_BLOCK_3_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_3_LOSS_CALCULATED = False

NOTEBOOK_08_BLOCK_3_OPTIMIZER_CREATED = False

NOTEBOOK_08_BLOCK_3_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_3_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_08_BLOCK_3_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_08_BLOCK_3_TRANSFORMATIVE_MECHANISM_INSTANTIATED = False

NOTEBOOK_08_BLOCK_3_TRANSFORMATION_WEIGHTS_INSTANTIATED = False

NOTEBOOK_08_BLOCK_3_RECURRENT_STATE_INSTANTIATED = False


# =============================================================================
# Deterministic inherited architecture reconstruction
# =============================================================================
#
# These definitions reproduce the validated Notebook 07 architecture and
# preserve the original state-dict key topology exactly.
# =============================================================================

class Notebook08Backbone(
    nn.Module
):

    def __init__(
        self,
        input_dim=768,
        output_dim=768,
    ):

        super().__init__()


        self.projection = nn.Linear(
            input_dim,
            output_dim,
        )


    def forward(
        self,
        x,
    ):

        return self.projection(
            x
        )


class Notebook08GlobalConfluent(
    nn.Module
):

    def __init__(
        self,
        input_dim=768,
        output_dim=256,
        dropout=0.10,
    ):

        super().__init__()


        self.global_projection = nn.Sequential(
            nn.Linear(
                input_dim,
                output_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.LayerNorm(
                output_dim
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.global_projection(
            x
        )


class Notebook08RepresentationCore(
    nn.Module
):

    def __init__(
        self,
        input_dim=256,
        hidden_dim=128,
        dropout=0.10,
    ):

        super().__init__()


        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.LayerNorm(
                hidden_dim
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.network(
            x
        )


class Notebook08RepresentationHead(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        dropout=0.10,
    ):

        super().__init__()


        self.core = Notebook08RepresentationCore(
            input_dim=
                input_dim,

            hidden_dim=
                hidden_dim,

            dropout=
                dropout,
        )


        self.output_layer = nn.Linear(
            hidden_dim,
            output_dim,
        )


    def forward(
        self,
        x,
    ):

        return self.output_layer(
            self.core(
                x
            )
        )


# =============================================================================
# Explicit architecture validation
# =============================================================================

BLOCK_3_BACKBONE_ARCHITECTURE_PROBE = (
    Notebook08Backbone()
)


BLOCK_3_BACKBONE_ARCHITECTURE_VALID = all(
    [
        isinstance(
            BLOCK_3_BACKBONE_ARCHITECTURE_PROBE.projection,
            nn.Linear,
        ),

        (
            BLOCK_3_BACKBONE_ARCHITECTURE_PROBE.projection.in_features
            ==
            NOTEBOOK_08_TEXTUAL_REPRESENTATION_DIM
        ),

        (
            BLOCK_3_BACKBONE_ARCHITECTURE_PROBE.projection.out_features
            ==
            NOTEBOOK_08_BACKBONE_INTERFACE_DIM
        ),
    ]
)


BLOCK_3_GLOBAL_ARCHITECTURE_PROBE = (
    Notebook08GlobalConfluent()
)


BLOCK_3_GLOBAL_ARCHITECTURE_VALID = all(
    [
        isinstance(
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE.global_projection[0],
            nn.Linear,
        ),

        isinstance(
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE.global_projection[1],
            nn.GELU,
        ),

        isinstance(
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE.global_projection[2],
            nn.Dropout,
        ),

        isinstance(
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE.global_projection[3],
            nn.LayerNorm,
        ),

        (
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE
            .global_projection[0]
            .in_features
            ==
            NOTEBOOK_08_BACKBONE_INTERFACE_DIM
        ),

        (
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE
            .global_projection[0]
            .out_features
            ==
            NOTEBOOK_08_GLOBAL_CONFLUENT_DIM
        ),

        (
            BLOCK_3_GLOBAL_ARCHITECTURE_PROBE
            .global_projection[2]
            .p
            ==
            NOTEBOOK_08_GLOBAL_CONFLUENT_DROPOUT
        ),
    ]
)


BLOCK_3_CORE_ARCHITECTURE_PROBE = (
    Notebook08RepresentationCore()
)


BLOCK_3_CORE_ARCHITECTURE_VALID = all(
    [
        isinstance(
            BLOCK_3_CORE_ARCHITECTURE_PROBE.network[0],
            nn.Linear,
        ),

        isinstance(
            BLOCK_3_CORE_ARCHITECTURE_PROBE.network[1],
            nn.GELU,
        ),

        isinstance(
            BLOCK_3_CORE_ARCHITECTURE_PROBE.network[2],
            nn.Dropout,
        ),

        isinstance(
            BLOCK_3_CORE_ARCHITECTURE_PROBE.network[3],
            nn.LayerNorm,
        ),

        (
            BLOCK_3_CORE_ARCHITECTURE_PROBE
            .network[0]
            .in_features
            ==
            NOTEBOOK_08_GLOBAL_CONFLUENT_DIM
        ),

        (
            BLOCK_3_CORE_ARCHITECTURE_PROBE
            .network[0]
            .out_features
            ==
            NOTEBOOK_08_REPRESENTATION_HIDDEN_DIM
        ),

        (
            BLOCK_3_CORE_ARCHITECTURE_PROBE
            .network[2]
            .p
            ==
            NOTEBOOK_08_REPRESENTATION_HEAD_DROPOUT
        ),
    ]
)


BLOCK_3_EXPLICIT_ARCHITECTURE_VALID = all(
    [
        BLOCK_3_BACKBONE_ARCHITECTURE_VALID,
        BLOCK_3_GLOBAL_ARCHITECTURE_VALID,
        BLOCK_3_CORE_ARCHITECTURE_VALID,
    ]
)


if not BLOCK_3_EXPLICIT_ARCHITECTURE_VALID:

    raise RuntimeError(
        "Notebook 08 inherited architecture reconstruction is invalid."
    )


# =============================================================================
# Inherited architecture-class contract
# =============================================================================
#
# Notebook 08 uses locally reconstructed class names, but the inherited
# metadata must still identify the validated Notebook 07 source classes.
# =============================================================================

BLOCK_3_EXPECTED_INHERITED_ARCHITECTURE = {
    "backbone":
        "Notebook07Backbone",

    "global_confluent":
        "Notebook07GlobalConfluent",

    "psychological_head":
        "Notebook07RepresentationHead",

    "factual_head":
        "Notebook07RepresentationHead",

    "social_head":
        "Notebook07RepresentationHead",
}


BLOCK_3_INHERITED_ARCHITECTURE_VALID = (
    NOTEBOOK_08_INHERITED_ARCHITECTURE
    ==
    BLOCK_3_EXPECTED_INHERITED_ARCHITECTURE
)


if not BLOCK_3_INHERITED_ARCHITECTURE_VALID:

    raise RuntimeError(
        "Notebook 07 inherited architecture-class contract "
        "does not match the expected validated architecture."
    )


# =============================================================================
# Instantiate Notebook 08 inherited runtime modules
# =============================================================================

NOTEBOOK_08_RESTORED_BACKBONE = (
    Notebook08Backbone(
        input_dim=
            NOTEBOOK_08_TEXTUAL_REPRESENTATION_DIM,

        output_dim=
            NOTEBOOK_08_BACKBONE_INTERFACE_DIM,
    )
)


NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT = (
    Notebook08GlobalConfluent(
        input_dim=
            NOTEBOOK_08_BACKBONE_INTERFACE_DIM,

        output_dim=
            NOTEBOOK_08_GLOBAL_CONFLUENT_DIM,

        dropout=
            NOTEBOOK_08_GLOBAL_CONFLUENT_DROPOUT,
    )
)


NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD = (
    Notebook08RepresentationHead(
        input_dim=
            NOTEBOOK_08_GLOBAL_CONFLUENT_DIM,

        hidden_dim=
            NOTEBOOK_08_REPRESENTATION_HIDDEN_DIM,

        output_dim=
            NOTEBOOK_08_PSYCHOLOGICAL_REPRESENTATION_DIM,

        dropout=
            NOTEBOOK_08_REPRESENTATION_HEAD_DROPOUT,
    )
)


NOTEBOOK_08_RESTORED_FACTUAL_HEAD = (
    Notebook08RepresentationHead(
        input_dim=
            NOTEBOOK_08_GLOBAL_CONFLUENT_DIM,

        hidden_dim=
            NOTEBOOK_08_REPRESENTATION_HIDDEN_DIM,

        output_dim=
            NOTEBOOK_08_FACTUAL_REPRESENTATION_DIM,

        dropout=
            NOTEBOOK_08_REPRESENTATION_HEAD_DROPOUT,
    )
)


NOTEBOOK_08_RESTORED_SOCIAL_HEAD = (
    Notebook08RepresentationHead(
        input_dim=
            NOTEBOOK_08_GLOBAL_CONFLUENT_DIM,

        hidden_dim=
            NOTEBOOK_08_REPRESENTATION_HIDDEN_DIM,

        output_dim=
            NOTEBOOK_08_SOCIAL_REPRESENTATION_DIM,

        dropout=
            NOTEBOOK_08_REPRESENTATION_HEAD_DROPOUT,
    )
)


BLOCK_3_RESTORED_MODULES = {
    "backbone":
        NOTEBOOK_08_RESTORED_BACKBONE,

    "global_confluent":
        NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,

    "psychological_head":
        NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,

    "factual_head":
        NOTEBOOK_08_RESTORED_FACTUAL_HEAD,

    "social_head":
        NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
}


# =============================================================================
# Pre-restoration state-dict topology validation
# =============================================================================

BLOCK_3_STATE_DICT_TOPOLOGY_VALID = True

BLOCK_3_STATE_DICT_TOPOLOGY_DETAILS = {}


for (
    module_name,
    module,
) in BLOCK_3_RESTORED_MODULES.items():

    runtime_state_dict = (
        module.state_dict()
    )

    inherited_state_dict = (
        NOTEBOOK_08_INHERITED_STATE_DICTS[
            module_name
        ]
    )


    key_match = (
        tuple(
            runtime_state_dict.keys()
        )
        ==
        tuple(
            inherited_state_dict.keys()
        )
    )


    shape_match = (
        key_match
        and
        all(
            runtime_state_dict[
                parameter_name
            ].shape
            ==
            inherited_state_dict[
                parameter_name
            ].shape

            for parameter_name
            in runtime_state_dict
        )
    )


    BLOCK_3_STATE_DICT_TOPOLOGY_DETAILS[
        module_name
    ] = {
        "keys_match":
            key_match,

        "shapes_match":
            shape_match,
    }


    if not all(
        [
            key_match,
            shape_match,
        ]
    ):

        BLOCK_3_STATE_DICT_TOPOLOGY_VALID = False


if not BLOCK_3_STATE_DICT_TOPOLOGY_VALID:

    raise RuntimeError(
        "Notebook 08 reconstructed module topology does not match "
        "the inherited Notebook 07 state dictionaries."
    )


# =============================================================================
# Strict inherited-state restoration
# =============================================================================

BLOCK_3_LOAD_RESULTS = {}


for (
    module_name,
    module,
) in BLOCK_3_RESTORED_MODULES.items():

    load_result = module.load_state_dict(
        NOTEBOOK_08_INHERITED_STATE_DICTS[
            module_name
        ],
        strict=True,
    )


    BLOCK_3_LOAD_RESULTS[
        module_name
    ] = load_result


NOTEBOOK_08_BLOCK_3_STATE_DICT_LOADED_INTO_MODEL = True


# =============================================================================
# Exact restoration helper
# =============================================================================

def block_3_state_dict_exact_match(
    module,
    expected_state_dict,
):

    observed_state_dict = (
        module.state_dict()
    )


    if (
        tuple(
            observed_state_dict.keys()
        )
        !=
        tuple(
            expected_state_dict.keys()
        )
    ):

        return False


    return all(
        torch.equal(
            observed_state_dict[
                parameter_name
            ]
            .detach()
            .cpu(),

            expected_state_dict[
                parameter_name
            ]
            .detach()
            .cpu(),
        )

        for parameter_name
        in observed_state_dict
    )


# =============================================================================
# Exact post-restoration validation
# =============================================================================

BLOCK_3_EXACT_STATE_MATCH = {
    module_name:
        block_3_state_dict_exact_match(
            module,
            NOTEBOOK_08_INHERITED_STATE_DICTS[
                module_name
            ],
        )

    for (
        module_name,
        module,
    ) in BLOCK_3_RESTORED_MODULES.items()
}


BLOCK_3_ALL_STATES_EXACT = all(
    BLOCK_3_EXACT_STATE_MATCH.values()
)


if not BLOCK_3_ALL_STATES_EXACT:

    raise RuntimeError(
        "One or more Notebook 08 modules do not exactly reproduce "
        "the persisted Notebook 07 parameter state."
    )


# =============================================================================
# Move restored modules to execution device
# =============================================================================

for module in BLOCK_3_RESTORED_MODULES.values():

    module.to(
        DEVICE
    )


# =============================================================================
# Freeze inherited representation model
# =============================================================================

for module in BLOCK_3_RESTORED_MODULES.values():

    for parameter in module.parameters():

        parameter.requires_grad = False


# =============================================================================
# Evaluation-mode contract
# =============================================================================

for module in BLOCK_3_RESTORED_MODULES.values():

    module.eval()


# =============================================================================
# Freezing validation
# =============================================================================

BLOCK_3_MODULES_FROZEN = {
    module_name:
        all(
            not parameter.requires_grad

            for parameter
            in module.parameters()
        )

    for (
        module_name,
        module,
    ) in BLOCK_3_RESTORED_MODULES.items()
}


BLOCK_3_ALL_MODULES_FROZEN = all(
    BLOCK_3_MODULES_FROZEN.values()
)


if not BLOCK_3_ALL_MODULES_FROZEN:

    raise RuntimeError(
        "One or more inherited Notebook 08 representation modules "
        "remain trainable."
    )


# =============================================================================
# Evaluation-mode validation
# =============================================================================

BLOCK_3_MODULES_IN_EVAL_MODE = {
    module_name:
        not module.training

    for (
        module_name,
        module,
    ) in BLOCK_3_RESTORED_MODULES.items()
}


BLOCK_3_ALL_MODULES_IN_EVAL_MODE = all(
    BLOCK_3_MODULES_IN_EVAL_MODE.values()
)


if not BLOCK_3_ALL_MODULES_IN_EVAL_MODE:

    raise RuntimeError(
        "One or more inherited representation modules "
        "are not in evaluation mode."
    )


# =============================================================================
# Execution-device validation
# =============================================================================

def block_3_module_device(
    module,
):

    parameter_devices = {
        parameter.device
        for parameter
        in module.parameters()
    }


    if len(
        parameter_devices
    ) != 1:

        return None


    return next(
        iter(
            parameter_devices
        )
    )


BLOCK_3_MODULE_DEVICES = {
    module_name:
        block_3_module_device(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_3_RESTORED_MODULES.items()
}


BLOCK_3_DEVICE_PLACEMENT_VALID = all(
    module_device == DEVICE

    for module_device
    in BLOCK_3_MODULE_DEVICES.values()
)


if not BLOCK_3_DEVICE_PLACEMENT_VALID:

    raise RuntimeError(
        "One or more inherited representation modules "
        "were not placed on the configured execution device."
    )


# =============================================================================
# Parameter-count validation
# =============================================================================

BLOCK_3_RESTORED_PARAMETER_COUNTS = {
    module_name:
        sum(
            parameter.numel()

            for parameter
            in module.parameters()
        )

    for (
        module_name,
        module,
    ) in BLOCK_3_RESTORED_MODULES.items()
}


BLOCK_3_PARAMETER_COUNTS_MATCH_HANDOVER = (
    BLOCK_3_RESTORED_PARAMETER_COUNTS
    ==
    NOTEBOOK_08_INHERITED_PARAMETER_COUNTS
)


BLOCK_3_RESTORED_TOTAL_PARAMETER_COUNT = sum(
    BLOCK_3_RESTORED_PARAMETER_COUNTS.values()
)


BLOCK_3_TOTAL_PARAMETER_COUNT_VALID = (
    BLOCK_3_RESTORED_TOTAL_PARAMETER_COUNT
    ==
    NOTEBOOK_08_INHERITED_TOTAL_PARAMETER_COUNT
    ==
    894003
)


if not all(
    [
        BLOCK_3_PARAMETER_COUNTS_MATCH_HANDOVER,
        BLOCK_3_TOTAL_PARAMETER_COUNT_VALID,
    ]
):

    raise RuntimeError(
        "Notebook 08 restored parameter counts do not match "
        "the validated Notebook 07 handover."
    )


# =============================================================================
# Post-device exact-state validation
# =============================================================================
#
# State equality is checked on CPU copies so that the validation remains
# valid if a future runtime uses CUDA.
# =============================================================================

BLOCK_3_POST_DEVICE_STATE_MATCH = {
    module_name:
        block_3_state_dict_exact_match(
            module,
            NOTEBOOK_08_INHERITED_STATE_DICTS[
                module_name
            ],
        )

    for (
        module_name,
        module,
    ) in BLOCK_3_RESTORED_MODULES.items()
}


BLOCK_3_POST_DEVICE_STATES_EXACT = all(
    BLOCK_3_POST_DEVICE_STATE_MATCH.values()
)


if not BLOCK_3_POST_DEVICE_STATES_EXACT:

    raise RuntimeError(
        "Moving the restored representation model to the execution "
        "device altered one or more inherited parameter tensors."
    )


# =============================================================================
# Runtime restoration contract
# =============================================================================

NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED = all(
    [
        BLOCK_3_EXPLICIT_ARCHITECTURE_VALID,
        BLOCK_3_INHERITED_ARCHITECTURE_VALID,
        BLOCK_3_STATE_DICT_TOPOLOGY_VALID,
        BLOCK_3_ALL_STATES_EXACT,
        BLOCK_3_POST_DEVICE_STATES_EXACT,
        BLOCK_3_ALL_MODULES_FROZEN,
        BLOCK_3_ALL_MODULES_IN_EVAL_MODE,
        BLOCK_3_DEVICE_PLACEMENT_VALID,
        BLOCK_3_PARAMETER_COUNTS_MATCH_HANDOVER,
        BLOCK_3_TOTAL_PARAMETER_COUNT_VALID,
    ]
)


# =============================================================================
# Final Block 3 validation
# =============================================================================

NOTEBOOK_08_BLOCK_3_VALID = all(
    [
        NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED,

        NOTEBOOK_08_BLOCK_3_STATE_DICT_LOADED_INTO_MODEL,

        not NOTEBOOK_08_BLOCK_3_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_3_LOSS_CALCULATED,
        not NOTEBOOK_08_BLOCK_3_OPTIMIZER_CREATED,
        not NOTEBOOK_08_BLOCK_3_BACKWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_3_OPTIMIZER_STEP_EXECUTED,
        not NOTEBOOK_08_BLOCK_3_PARAMETER_UPDATE_EXECUTED,
        not NOTEBOOK_08_BLOCK_3_TRANSFORMATIVE_MECHANISM_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_3_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_3_RECURRENT_STATE_INSTANTIATED,
    ]
)


if not NOTEBOOK_08_BLOCK_3_VALID:

    raise RuntimeError(
        "Notebook 08 Block 3 inherited-architecture restoration "
        "validation failed."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_3_COMPLETE = True


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 3: "
    "Controlled Inherited-Architecture Restoration "
    "and State-Dict Validation"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_3_VERSION}"
)

print("-" * 72)

print(
    f"Backbone architecture       : "
    f"Linear(768 -> 768)"
)

print(
    f"Global confluent            : "
    f"Linear(768 -> 256) -> GELU -> "
    f"Dropout(0.10) -> LayerNorm(256)"
)

print(
    f"Representation core         : "
    f"Linear(256 -> 128) -> GELU -> "
    f"Dropout(0.10) -> LayerNorm(128)"
)

print(
    f"Psychological output        : "
    f"Linear(128 -> 34)"
)

print(
    f"Factual output              : "
    f"Linear(128 -> 10)"
)

print(
    f"Social output               : "
    f"Linear(128 -> 7)"
)

print("-" * 72)

for (
    module_name,
    parameter_count,
) in BLOCK_3_RESTORED_PARAMETER_COUNTS.items():

    print(
        f"{module_name:<28}: "
        f"{parameter_count:,} parameters"
    )

print("-" * 72)

print(
    f"Total restored parameters   : "
    f"{BLOCK_3_RESTORED_TOTAL_PARAMETER_COUNT:,}"
)

print(
    f"Parameter counts valid      : "
    f"{BLOCK_3_TOTAL_PARAMETER_COUNT_VALID}"
)

print(
    f"State topology valid        : "
    f"{BLOCK_3_STATE_DICT_TOPOLOGY_VALID}"
)

print(
    f"Exact state restoration     : "
    f"{BLOCK_3_ALL_STATES_EXACT}"
)

print(
    f"Post-device state exact     : "
    f"{BLOCK_3_POST_DEVICE_STATES_EXACT}"
)

print("-" * 72)

print(
    f"Execution device            : "
    f"{DEVICE}"
)

print(
    f"Device placement valid      : "
    f"{BLOCK_3_DEVICE_PLACEMENT_VALID}"
)

print(
    f"All modules frozen          : "
    f"{BLOCK_3_ALL_MODULES_FROZEN}"
)

print(
    f"All modules in eval mode    : "
    f"{BLOCK_3_ALL_MODULES_IN_EVAL_MODE}"
)

print("-" * 72)

print(
    f"Forward pass executed       : "
    f"{NOTEBOOK_08_BLOCK_3_FORWARD_PASS_EXECUTED}"
)

print(
    f"Optimizer created           : "
    f"{NOTEBOOK_08_BLOCK_3_OPTIMIZER_CREATED}"
)

print(
    f"Parameter update executed   : "
    f"{NOTEBOOK_08_BLOCK_3_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Transformative instantiated : "
    f"{NOTEBOOK_08_BLOCK_3_TRANSFORMATIVE_MECHANISM_INSTANTIATED}"
)

print(
    f"Transform weights created   : "
    f"{NOTEBOOK_08_BLOCK_3_TRANSFORMATION_WEIGHTS_INSTANTIATED}"
)

print(
    f"Recurrent state created     : "
    f"{NOTEBOOK_08_BLOCK_3_RECURRENT_STATE_INSTANTIATED}"
)

print("-" * 72)

print(
    f"Representation model restored: "
    f"{NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_3_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_3_COMPLETE}"
)

print("=" * 72)

print(
    "The complete Notebook 07 representation architecture was "
    "restored exactly from the persisted handover."
)

print(
    "All 894,003 inherited parameters were reproduced, moved to the "
    "configured execution device, frozen, and placed in evaluation mode."
)

print(
    "No representation forward pass, loss calculation, optimiser "
    "operation, or parameter update was executed."
)

print(
    "No transformative mechanism, transformation weight, or recurrent "
    "state has yet been instantiated."
)

print(
    "Notebook 08 may now proceed from verified representation-model "
    "restoration to the controlled transformative-mechanism stage."
)

print("=" * 72)

Media AI — Notebook 08, Block 3: Controlled Inherited-Architecture Restoration and State-Dict Validation
Block version               : 1.0
------------------------------------------------------------------------
Backbone architecture       : Linear(768 -> 768)
Global confluent            : Linear(768 -> 256) -> GELU -> Dropout(0.10) -> LayerNorm(256)
Representation core         : Linear(256 -> 128) -> GELU -> Dropout(0.10) -> LayerNorm(128)
Psychological output        : Linear(128 -> 34)
Factual output              : Linear(128 -> 10)
Social output               : Linear(128 -> 7)
------------------------------------------------------------------------
backbone                    : 590,592 parameters
global_confluent            : 197,376 parameters
psychological_head          : 37,538 parameters
factual_head                : 34,442 parameters
social_head                 : 34,055 parameters
------------------------------------------------------------------------
Total restored parameter

## Block 4 — Transformative-Mechanism Architectural Contract and Module Construction

This block introduces the core **transformative mechanism** of the Media AI architecture.

The preceding blocks established a validated and frozen representation system capable of producing three distinct descriptions of an incoming media observation:

$$
\mathbf{r}^{(F)}_t \in \mathbb{R}^{10},
\qquad
\mathbf{r}^{(P)}_t \in \mathbb{R}^{34},
\qquad
\mathbf{r}^{(S)}_t \in \mathbb{R}^{7},
$$

where the superscripts (F), (P), and (S) denote the factual, psychological, and social representation spaces respectively.

These vectors describe the **current media observation**. They do not, by themselves, describe how that observation changes an accumulated state.

The purpose of the transformative mechanism is to introduce that distinction.

### From representation to transformation

A representation answers a question of the form:

> **What is present in this media observation?**

A transformation answers a different question:

> **Given what has already been accumulated, what change does this new observation imply?**

The transformative mechanism therefore does not replace the representation model. Instead, it operates **on top of the representation model** and converts a current representation, together with the relevant preceding state, into a pathway-specific **candidate transformation**.

For pathway (k),

$$
k \in {F,P,S},
$$

the conceptual operation is

$$
\mathcal{T}^{(k)}
:
\left(
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(k)}_t,
$$

where:

* (\mathbf{r}^{(k)}_t) is the representation of the current observation;
* (\mathbf{h}^{(k)}_{t-1}) is the accumulated pathway-specific state inherited from preceding observations;
* (\mathcal{T}^{(k)}) is the learned transformative mapping;
* (\boldsymbol{\tau}^{(k)}_t) is the resulting **candidate transformation**.

The candidate transformation is deliberately distinguished from the next recurrent state.

It describes **what the current observation proposes to change**, given the state that existed before that observation was encountered.

It does not yet determine how strongly that proposed transformation should be applied.

### Three independent transformative pathways

The Media AI architecture preserves the separation between factual, psychological, and social information throughout the transformative stage.

The factual mechanism operates in the factual space:

$$
\mathcal{T}^{(F)}
:
\left(
\mathbf{r}^{(F)}*t,
\mathbf{h}^{(F)}*{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(F)}_t.
$$

The psychological mechanism operates independently in the psychological space:

$$
\mathcal{T}^{(P)}
:
\left(
\mathbf{r}^{(P)}*t,
\mathbf{h}^{(P)}*{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(P)}_t.
$$

The social mechanism similarly operates in the social space:

$$
\mathcal{T}^{(S)}
:
\left(
\mathbf{r}^{(S)}*t,
\mathbf{h}^{(S)}*{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(S)}_t.
$$

The three mechanisms are structurally independent. A factual representation is not projected through the psychological transformative mechanism, and a social transformation does not implicitly overwrite factual or psychological state.

This preserves the semantic meaning of the three representation spaces while allowing each pathway to develop its own temporal dynamics.

### Why the preceding state is required

If the transformative mechanism depended only on the current representation,

$$
\boldsymbol{\tau}^{(k)}_t
=========================

\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}_t
\right),
$$

then the same observation would always imply the same transformation regardless of what had been observed previously.

That would reduce the model to another observation-level projection.

The transformative mechanism instead conditions the candidate transformation on both the current representation and the preceding accumulated state:

$$
\boldsymbol{\tau}^{(k)}_t
=========================

\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1}
\right).
$$

Consequently, an identical or similar media representation may imply a different transformation depending on the state into which it arrives.

This provides the architectural basis for modelling effects such as accumulation, reinforcement, contradiction, attenuation, saturation, and context-dependent response without requiring the representation vector itself to encode the entire previous exposure history.

### Transformation is not yet state update

A central architectural distinction is maintained in this notebook.

The output

$$
\boldsymbol{\tau}^{(k)}_t
$$

is a **candidate transformation**, not the final updated state.

The architecture subsequently introduces a pathway-specific transformative weight,

$$
TW^{(k)}_t,
$$

which determines how strongly the candidate transformation contributes to the recurrent update.

The complete conceptual sequence is therefore:

$$
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1}
;\longrightarrow;
\boldsymbol{\tau}^{(k)}_t
;\longrightarrow;
TW^{(k)}_t
;\longrightarrow;
\mathbf{h}^{(k)}_t.
$$

Block 4 is concerned only with the first transition:

$$
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1}
;\longrightarrow;
\boldsymbol{\tau}^{(k)}_t.
$$

The transformative weights and the explicit recurrent-state update rule remain outside this block.

This separation prevents three conceptually different operations from being collapsed into a single opaque recurrent computation:

1. **representation** — what the current media observation contains;
2. **transformation** — what change that observation proposes in the context of prior state;
3. **state update** — how much of that proposed change is incorporated into the accumulated state.

### Dimensional preservation

Each transformative pathway operates in its own semantic space and returns a candidate transformation with the same dimensionality as that space.

Accordingly,

$$
\boldsymbol{\tau}^{(F)}_t
\in
\mathbb{R}^{10},
$$

$$
\boldsymbol{\tau}^{(P)}_t
\in
\mathbb{R}^{34},
$$

and

$$
\boldsymbol{\tau}^{(S)}_t
\in
\mathbb{R}^{7}.
$$

The associated state vectors obey the same pathway-specific dimensional contracts:

$$
\mathbf{h}^{(F)}_{t-1}
\in
\mathbb{R}^{10},
$$

$$
\mathbf{h}^{(P)}_{t-1}
\in
\mathbb{R}^{34},
$$

and

$$
\mathbf{h}^{(S)}_{t-1}
\in
\mathbb{R}^{7}.
$$

Dimensional preservation is intentional. It ensures that a factual transformation remains interpretable as a change within the factual state space, a psychological transformation remains a change within the psychological state space, and a social transformation remains a change within the social state space.

### Controlled contextual interaction

For each pathway, the transformative module receives two semantically different inputs:

$$
\mathbf{r}^{(k)}*t
\quad\text{and}\quad
\mathbf{h}^{(k)}*{t-1}.
$$

The first describes the **incoming observation**.

The second describes the **accumulated condition before that observation**.

The module must therefore learn an interaction between new information and prior state rather than treating the two as interchangeable quantities.

Conceptually, the module implements

$$
\boldsymbol{\tau}^{(k)}_t
=========================

f^{(k)}
\left(
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1};
\theta^{(k)}_T
\right),
$$

where (\theta^{(k)}_T) denotes the trainable parameters of the transformative mechanism for pathway (k).

The three parameter sets remain independent:

$$
\theta^{(F)}_T,
\qquad
\theta^{(P)}_T,
\qquad
\theta^{(S)}_T.
$$

This allows the factual, psychological, and social pathways to learn different transformation functions even when they are exposed to the same sequence of media observations.

### Causal boundary

The transformative mechanism is explicitly causal.

At position (t), it may use:

$$
\mathbf{r}^{(k)}_t
$$

and the state accumulated strictly before the current observation,

$$
\mathbf{h}^{(k)}_{t-1}.
$$

It must not depend on:

$$
\mathbf{r}^{(k)}*{t+1},
\mathbf{r}^{(k)}*{t+2},
\ldots
$$

or on any state constructed using future observations.

The allowed information flow is therefore

$$
{\mathbf{r}*1,\ldots,\mathbf{r}*{t-1}}
\longrightarrow
\mathbf{h}_{t-1}
\longrightarrow
\mathcal{T}
\left(
\mathbf{r}*t,
\mathbf{h}*{t-1}
\right),
$$

while future-to-past information flow is prohibited.

This causal restriction is fundamental to the interpretation of the model as a sequential media-exposure mechanism.

### Architectural role of Block 4

This block establishes and instantiates the three transformative modules and validates their structural contracts.

It therefore:

* defines the factual, psychological, and social transformative module classes;
* preserves separate parameter sets for the three pathways;
* enforces pathway-specific input, state, and output dimensions;
* establishes the interface between current representations and preceding recurrent states;
* constructs candidate transformations without applying transformative weights;
* verifies module parameterisation and deterministic initialisation;
* confirms that the inherited representation model remains frozen and unchanged;
* establishes the trainability boundary between the frozen representation system and the newly introduced transformative modules.

At the end of this block, the architecture has progressed from a static representation system to a system that contains an explicit mechanism capable of modelling state-dependent change.

However, the mechanism is **structurally available rather than yet temporally executed**.

No article sequence is propagated through it in this block.

No transformative weight

$$
TW_F,\qquad TW_P,\qquad TW_S
$$

is calculated or applied.

No recurrent state

$$
H_F,\qquad H_P,\qquad H_S
$$

is updated.

No optimiser step or parameter update is performed.

The resulting boundary is therefore:

$$
\boxed{
\text{validated representation}
+
\text{constructed transformation function}
}
$$

rather than

$$
\boxed{
\text{completed recurrent transformation system}
}.
$$

This distinction keeps construction of the transformative mechanism independently auditable before transformation weighting and recurrent propagation are introduced in subsequent stages.


In [4]:
# =============================================================================
# Media AI — Notebook 08, Block 4
# Transformative-Mechanism Architectural Contract and Module Construction
# =============================================================================

import torch
import torch.nn as nn


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_4 = 4

NOTEBOOK_08_BLOCK_4_NAME = (
    "Transformative-Mechanism Architectural Contract "
    "and Module Construction"
)

NOTEBOOK_08_BLOCK_4_VERSION = "1.0"


# =============================================================================
# Dependency checks
# =============================================================================

required_block_4_objects = [
    # -------------------------------------------------------------------------
    # Block 1 — deterministic runtime
    # -------------------------------------------------------------------------
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Block 2 — inherited handover
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_2_COMPLETE",
    "NOTEBOOK_08_BLOCK_2_VALID",
    "NOTEBOOK_08_REPRESENTATION_HANDOVER_READY",

    # -------------------------------------------------------------------------
    # Block 3 — restored representation model
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_3_COMPLETE",
    "NOTEBOOK_08_BLOCK_3_VALID",
    "NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED",

    "NOTEBOOK_08_RESTORED_BACKBONE",
    "NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT",
    "NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD",
    "NOTEBOOK_08_RESTORED_FACTUAL_HEAD",
    "NOTEBOOK_08_RESTORED_SOCIAL_HEAD",

    "NOTEBOOK_08_PSYCHOLOGICAL_REPRESENTATION_DIM",
    "NOTEBOOK_08_FACTUAL_REPRESENTATION_DIM",
    "NOTEBOOK_08_SOCIAL_REPRESENTATION_DIM",
]


missing_block_4_objects = [
    object_name
    for object_name
    in required_block_4_objects
    if object_name not in globals()
]


if missing_block_4_objects:

    raise NameError(
        "Notebook 08 Block 4 prerequisites are not initialised. "
        f"Missing: {missing_block_4_objects}"
    )


if not all(
    [
        NOTEBOOK_08_BLOCK_2_COMPLETE,
        NOTEBOOK_08_BLOCK_2_VALID,
        NOTEBOOK_08_REPRESENTATION_HANDOVER_READY,
        NOTEBOOK_08_BLOCK_3_COMPLETE,
        NOTEBOOK_08_BLOCK_3_VALID,
        NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED,
    ]
):

    raise RuntimeError(
        "Notebook 08 Blocks 2 and 3 must be complete and valid "
        "before constructing the transformative mechanism."
    )


# =============================================================================
# Transformative-mechanism dimensional contract
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_HIDDEN_MULTIPLIER = 2

NOTEBOOK_08_TRANSFORMATIVE_DROPOUT = 0.10


NOTEBOOK_08_FACTUAL_TRANSFORM_DIM = (
    NOTEBOOK_08_FACTUAL_REPRESENTATION_DIM
)

NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM = (
    NOTEBOOK_08_PSYCHOLOGICAL_REPRESENTATION_DIM
)

NOTEBOOK_08_SOCIAL_TRANSFORM_DIM = (
    NOTEBOOK_08_SOCIAL_REPRESENTATION_DIM
)


NOTEBOOK_08_FACTUAL_TRANSFORM_INPUT_DIM = (
    2
    *
    NOTEBOOK_08_FACTUAL_TRANSFORM_DIM
)

NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_INPUT_DIM = (
    2
    *
    NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM
)

NOTEBOOK_08_SOCIAL_TRANSFORM_INPUT_DIM = (
    2
    *
    NOTEBOOK_08_SOCIAL_TRANSFORM_DIM
)


NOTEBOOK_08_FACTUAL_TRANSFORM_HIDDEN_DIM = (
    NOTEBOOK_08_TRANSFORMATIVE_HIDDEN_MULTIPLIER
    *
    NOTEBOOK_08_FACTUAL_TRANSFORM_DIM
)

NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_HIDDEN_DIM = (
    NOTEBOOK_08_TRANSFORMATIVE_HIDDEN_MULTIPLIER
    *
    NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM
)

NOTEBOOK_08_SOCIAL_TRANSFORM_HIDDEN_DIM = (
    NOTEBOOK_08_TRANSFORMATIVE_HIDDEN_MULTIPLIER
    *
    NOTEBOOK_08_SOCIAL_TRANSFORM_DIM
)


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_08_BLOCK_4_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_4_TRANSFORM_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_4_TRANSFORMATION_WEIGHTS_INSTANTIATED = False

NOTEBOOK_08_BLOCK_4_TRANSFORMATION_WEIGHTS_EXECUTED = False

NOTEBOOK_08_BLOCK_4_RECURRENT_STATE_INSTANTIATED = False

NOTEBOOK_08_BLOCK_4_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_08_BLOCK_4_LOSS_CALCULATED = False

NOTEBOOK_08_BLOCK_4_OPTIMIZER_CREATED = False

NOTEBOOK_08_BLOCK_4_ZERO_GRAD_EXECUTED = False

NOTEBOOK_08_BLOCK_4_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_4_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_08_BLOCK_4_PARAMETER_UPDATE_EXECUTED = False


# =============================================================================
# Snapshot inherited representation-model state
# =============================================================================

BLOCK_4_INHERITED_REPRESENTATION_MODULES = {
    "backbone":
        NOTEBOOK_08_RESTORED_BACKBONE,

    "global_confluent":
        NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,

    "psychological_head":
        NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,

    "factual_head":
        NOTEBOOK_08_RESTORED_FACTUAL_HEAD,

    "social_head":
        NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
}


def block_4_snapshot_module(
    module,
):

    return {
        parameter_name:
            parameter.detach()
            .cpu()
            .clone()

        for (
            parameter_name,
            parameter,
        ) in module.named_parameters()
    }


BLOCK_4_REPRESENTATION_STATE_BEFORE = {
    module_name:
        block_4_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_4_INHERITED_REPRESENTATION_MODULES.items()
}


# =============================================================================
# Transformative mechanism
# =============================================================================
#
# For pathway k:
#
#   r_t^(k)      : current representation
#   h_(t-1)^(k)  : preceding pathway-specific state
#
# The two vectors are concatenated:
#
#   [r_t^(k) ; h_(t-1)^(k)]
#
# and mapped to a candidate transformation tau_t^(k) with the same
# dimensionality as the pathway representation/state space.
#
# No Transformative Weight is applied in this module.
# No recurrent state update is performed in this module.
# =============================================================================

class Notebook08TransformativeMechanism(
    nn.Module
):

    def __init__(
        self,
        representation_dim,
        hidden_dim,
        dropout=0.10,
    ):

        super().__init__()


        self.representation_dim = (
            representation_dim
        )

        self.state_dim = (
            representation_dim
        )

        self.input_dim = (
            representation_dim
            +
            self.state_dim
        )

        self.hidden_dim = (
            hidden_dim
        )

        self.output_dim = (
            representation_dim
        )


        self.transform_network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                self.hidden_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.LayerNorm(
                self.hidden_dim
            ),

            nn.Linear(
                self.hidden_dim,
                self.output_dim,
            ),
        )


    def forward(
        self,
        representation,
        previous_state,
    ):

        if (
            representation.shape[-1]
            !=
            self.representation_dim
        ):

            raise ValueError(
                "Representation dimension does not match "
                "the transformative-mechanism contract."
            )


        if (
            previous_state.shape[-1]
            !=
            self.state_dim
        ):

            raise ValueError(
                "Previous-state dimension does not match "
                "the transformative-mechanism contract."
            )


        if (
            representation.shape[:-1]
            !=
            previous_state.shape[:-1]
        ):

            raise ValueError(
                "Representation and previous state must have "
                "matching leading dimensions."
            )


        combined_input = torch.cat(
            [
                representation,
                previous_state,
            ],
            dim=-1,
        )


        candidate_transformation = (
            self.transform_network(
                combined_input
            )
        )


        return candidate_transformation


# =============================================================================
# Construct pathway-specific transformative modules
# =============================================================================

NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM = (
    Notebook08TransformativeMechanism(
        representation_dim=
            NOTEBOOK_08_FACTUAL_TRANSFORM_DIM,

        hidden_dim=
            NOTEBOOK_08_FACTUAL_TRANSFORM_HIDDEN_DIM,

        dropout=
            NOTEBOOK_08_TRANSFORMATIVE_DROPOUT,
    )
    .to(
        DEVICE
    )
)


NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM = (
    Notebook08TransformativeMechanism(
        representation_dim=
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM,

        hidden_dim=
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_HIDDEN_DIM,

        dropout=
            NOTEBOOK_08_TRANSFORMATIVE_DROPOUT,
    )
    .to(
        DEVICE
    )
)


NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM = (
    Notebook08TransformativeMechanism(
        representation_dim=
            NOTEBOOK_08_SOCIAL_TRANSFORM_DIM,

        hidden_dim=
            NOTEBOOK_08_SOCIAL_TRANSFORM_HIDDEN_DIM,

        dropout=
            NOTEBOOK_08_TRANSFORMATIVE_DROPOUT,
    )
    .to(
        DEVICE
    )
)


NOTEBOOK_08_TRANSFORMATIVE_MODULES = {
    "factual":
        NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM,

    "psychological":
        NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,

    "social":
        NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM,
}


NOTEBOOK_08_BLOCK_4_TRANSFORMATIVE_MECHANISM_INSTANTIATED = True


# =============================================================================
# Pathway independence validation
# =============================================================================

BLOCK_4_PATHWAY_OBJECTS_DISTINCT = (
    len(
        {
            id(
                module
            )
            for module
            in NOTEBOOK_08_TRANSFORMATIVE_MODULES.values()
        }
    )
    ==
    3
)


BLOCK_4_PARAMETER_OBJECT_IDS = {
    pathway_name:
        {
            id(
                parameter
            )
            for parameter
            in module.parameters()
        }

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_08_TRANSFORMATIVE_MODULES.items()
}


BLOCK_4_FACTUAL_PSYCHOLOGICAL_PARAMETERS_DISTINCT = (
    BLOCK_4_PARAMETER_OBJECT_IDS[
        "factual"
    ].isdisjoint(
        BLOCK_4_PARAMETER_OBJECT_IDS[
            "psychological"
        ]
    )
)


BLOCK_4_FACTUAL_SOCIAL_PARAMETERS_DISTINCT = (
    BLOCK_4_PARAMETER_OBJECT_IDS[
        "factual"
    ].isdisjoint(
        BLOCK_4_PARAMETER_OBJECT_IDS[
            "social"
        ]
    )
)


BLOCK_4_PSYCHOLOGICAL_SOCIAL_PARAMETERS_DISTINCT = (
    BLOCK_4_PARAMETER_OBJECT_IDS[
        "psychological"
    ].isdisjoint(
        BLOCK_4_PARAMETER_OBJECT_IDS[
            "social"
        ]
    )
)


BLOCK_4_PATHWAY_PARAMETERS_DISTINCT = all(
    [
        BLOCK_4_FACTUAL_PSYCHOLOGICAL_PARAMETERS_DISTINCT,
        BLOCK_4_FACTUAL_SOCIAL_PARAMETERS_DISTINCT,
        BLOCK_4_PSYCHOLOGICAL_SOCIAL_PARAMETERS_DISTINCT,
    ]
)


# =============================================================================
# Pathway dimensional validation
# =============================================================================

BLOCK_4_EXPECTED_DIMENSIONS = {
    "factual":
        {
            "representation":
                10,

            "state":
                10,

            "combined_input":
                20,

            "hidden":
                20,

            "output":
                10,
        },

    "psychological":
        {
            "representation":
                34,

            "state":
                34,

            "combined_input":
                68,

            "hidden":
                68,

            "output":
                34,
        },

    "social":
        {
            "representation":
                7,

            "state":
                7,

            "combined_input":
                14,

            "hidden":
                14,

            "output":
                7,
        },
}


BLOCK_4_PATHWAY_DIMENSIONS_VALID = {}


for (
    pathway_name,
    module,
) in NOTEBOOK_08_TRANSFORMATIVE_MODULES.items():

    expected = (
        BLOCK_4_EXPECTED_DIMENSIONS[
            pathway_name
        ]
    )


    BLOCK_4_PATHWAY_DIMENSIONS_VALID[
        pathway_name
    ] = all(
        [
            (
                module.representation_dim
                ==
                expected[
                    "representation"
                ]
            ),

            (
                module.state_dim
                ==
                expected[
                    "state"
                ]
            ),

            (
                module.input_dim
                ==
                expected[
                    "combined_input"
                ]
            ),

            (
                module.hidden_dim
                ==
                expected[
                    "hidden"
                ]
            ),

            (
                module.output_dim
                ==
                expected[
                    "output"
                ]
            ),

            isinstance(
                module.transform_network[0],
                nn.Linear,
            ),

            (
                module.transform_network[0].in_features
                ==
                expected[
                    "combined_input"
                ]
            ),

            (
                module.transform_network[0].out_features
                ==
                expected[
                    "hidden"
                ]
            ),

            isinstance(
                module.transform_network[1],
                nn.GELU,
            ),

            isinstance(
                module.transform_network[2],
                nn.Dropout,
            ),

            (
                module.transform_network[2].p
                ==
                NOTEBOOK_08_TRANSFORMATIVE_DROPOUT
            ),

            isinstance(
                module.transform_network[3],
                nn.LayerNorm,
            ),

            isinstance(
                module.transform_network[4],
                nn.Linear,
            ),

            (
                module.transform_network[4].in_features
                ==
                expected[
                    "hidden"
                ]
            ),

            (
                module.transform_network[4].out_features
                ==
                expected[
                    "output"
                ]
            ),
        ]
    )


BLOCK_4_ALL_PATHWAY_DIMENSIONS_VALID = all(
    BLOCK_4_PATHWAY_DIMENSIONS_VALID.values()
)


if not BLOCK_4_ALL_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "One or more transformative pathways violate "
        "the dimensional architecture contract."
    )


# =============================================================================
# Transformative-module device validation
# =============================================================================

def block_4_module_device(
    module,
):

    parameter_devices = {
        parameter.device

        for parameter
        in module.parameters()
    }


    if len(
        parameter_devices
    ) != 1:

        return None


    return next(
        iter(
            parameter_devices
        )
    )


BLOCK_4_TRANSFORMATIVE_MODULE_DEVICES = {
    pathway_name:
        block_4_module_device(
            module
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_08_TRANSFORMATIVE_MODULES.items()
}


BLOCK_4_TRANSFORMATIVE_DEVICE_VALID = all(
    module_device
    ==
    DEVICE

    for module_device
    in BLOCK_4_TRANSFORMATIVE_MODULE_DEVICES.values()
)


if not BLOCK_4_TRANSFORMATIVE_DEVICE_VALID:

    raise RuntimeError(
        "One or more transformative modules were not placed "
        "on the configured execution device."
    )


# =============================================================================
# Trainability boundary
# =============================================================================
#
# The inherited representation model must remain frozen.
# The newly introduced transformative mechanisms are trainable by design.
# =============================================================================

BLOCK_4_TRANSFORMATIVE_MODULES_TRAINABLE = {
    pathway_name:
        all(
            parameter.requires_grad

            for parameter
            in module.parameters()
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_08_TRANSFORMATIVE_MODULES.items()
}


BLOCK_4_ALL_TRANSFORMATIVE_MODULES_TRAINABLE = all(
    BLOCK_4_TRANSFORMATIVE_MODULES_TRAINABLE.values()
)


BLOCK_4_REPRESENTATION_MODULES_FROZEN = {
    module_name:
        all(
            not parameter.requires_grad

            for parameter
            in module.parameters()
        )

    for (
        module_name,
        module,
    ) in BLOCK_4_INHERITED_REPRESENTATION_MODULES.items()
}


BLOCK_4_ALL_REPRESENTATION_MODULES_FROZEN = all(
    BLOCK_4_REPRESENTATION_MODULES_FROZEN.values()
)


if not BLOCK_4_ALL_TRANSFORMATIVE_MODULES_TRAINABLE:

    raise RuntimeError(
        "One or more transformative modules are not trainable."
    )


if not BLOCK_4_ALL_REPRESENTATION_MODULES_FROZEN:

    raise RuntimeError(
        "The inherited representation model must remain frozen "
        "during transformative-mechanism construction."
    )


# =============================================================================
# Parameter accounting
# =============================================================================

BLOCK_4_TRANSFORMATIVE_PARAMETER_COUNTS = {
    pathway_name:
        sum(
            parameter.numel()

            for parameter
            in module.parameters()
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_08_TRANSFORMATIVE_MODULES.items()
}


BLOCK_4_TOTAL_TRANSFORMATIVE_PARAMETERS = sum(
    BLOCK_4_TRANSFORMATIVE_PARAMETER_COUNTS.values()
)


BLOCK_4_ALL_PARAMETER_COUNTS_POSITIVE = all(
    parameter_count > 0

    for parameter_count
    in BLOCK_4_TRANSFORMATIVE_PARAMETER_COUNTS.values()
)


if not BLOCK_4_ALL_PARAMETER_COUNTS_POSITIVE:

    raise RuntimeError(
        "One or more transformative mechanisms contain "
        "no trainable parameters."
    )


# =============================================================================
# Verify inherited representation state remains unchanged
# =============================================================================

BLOCK_4_REPRESENTATION_STATE_AFTER = {
    module_name:
        block_4_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_4_INHERITED_REPRESENTATION_MODULES.items()
}


def block_4_states_equal(
    before_state,
    after_state,
):

    if (
        tuple(
            before_state.keys()
        )
        !=
        tuple(
            after_state.keys()
        )
    ):

        return False


    return all(
        torch.equal(
            before_state[
                parameter_name
            ],
            after_state[
                parameter_name
            ],
        )

        for parameter_name
        in before_state
    )


BLOCK_4_REPRESENTATION_MODULES_UNCHANGED = {
    module_name:
        block_4_states_equal(
            BLOCK_4_REPRESENTATION_STATE_BEFORE[
                module_name
            ],

            BLOCK_4_REPRESENTATION_STATE_AFTER[
                module_name
            ],
        )

    for module_name
    in BLOCK_4_INHERITED_REPRESENTATION_MODULES
}


BLOCK_4_ALL_REPRESENTATION_MODULES_UNCHANGED = all(
    BLOCK_4_REPRESENTATION_MODULES_UNCHANGED.values()
)


if not BLOCK_4_ALL_REPRESENTATION_MODULES_UNCHANGED:

    raise RuntimeError(
        "One or more inherited representation-model parameters "
        "changed during transformative-mechanism construction."
    )


# =============================================================================
# Architectural separation validation
# =============================================================================

BLOCK_4_TRANSFORMATION_OUTPUT_DIM_PRESERVED = all(
    [
        (
            NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM.output_dim
            ==
            NOTEBOOK_08_FACTUAL_REPRESENTATION_DIM
        ),

        (
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM.output_dim
            ==
            NOTEBOOK_08_PSYCHOLOGICAL_REPRESENTATION_DIM
        ),

        (
            NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM.output_dim
            ==
            NOTEBOOK_08_SOCIAL_REPRESENTATION_DIM
        ),
    ]
)


BLOCK_4_STATE_DIM_PRESERVED = all(
    [
        (
            NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM.state_dim
            ==
            NOTEBOOK_08_FACTUAL_REPRESENTATION_DIM
        ),

        (
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM.state_dim
            ==
            NOTEBOOK_08_PSYCHOLOGICAL_REPRESENTATION_DIM
        ),

        (
            NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM.state_dim
            ==
            NOTEBOOK_08_SOCIAL_REPRESENTATION_DIM
        ),
    ]
)


BLOCK_4_ARCHITECTURAL_SEPARATION_VALID = all(
    [
        BLOCK_4_PATHWAY_OBJECTS_DISTINCT,
        BLOCK_4_PATHWAY_PARAMETERS_DISTINCT,
        BLOCK_4_TRANSFORMATION_OUTPUT_DIM_PRESERVED,
        BLOCK_4_STATE_DIM_PRESERVED,
    ]
)


if not BLOCK_4_ARCHITECTURAL_SEPARATION_VALID:

    raise RuntimeError(
        "Transformative pathway independence or dimensional "
        "preservation contract is invalid."
    )


# =============================================================================
# Transformative architecture readiness
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY = all(
    [
        NOTEBOOK_08_BLOCK_4_TRANSFORMATIVE_MECHANISM_INSTANTIATED,
        BLOCK_4_ALL_PATHWAY_DIMENSIONS_VALID,
        BLOCK_4_TRANSFORMATIVE_DEVICE_VALID,
        BLOCK_4_ALL_TRANSFORMATIVE_MODULES_TRAINABLE,
        BLOCK_4_ALL_REPRESENTATION_MODULES_FROZEN,
        BLOCK_4_ALL_REPRESENTATION_MODULES_UNCHANGED,
        BLOCK_4_ARCHITECTURAL_SEPARATION_VALID,
        BLOCK_4_ALL_PARAMETER_COUNTS_POSITIVE,
    ]
)


# =============================================================================
# Final Block 4 validation
# =============================================================================

NOTEBOOK_08_BLOCK_4_VALID = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY,

        not NOTEBOOK_08_BLOCK_4_REPRESENTATION_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_TRANSFORM_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_4_TRANSFORMATION_WEIGHTS_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_RECURRENT_STATE_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_4_RECURRENT_UPDATE_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_LOSS_CALCULATED,
        not NOTEBOOK_08_BLOCK_4_OPTIMIZER_CREATED,
        not NOTEBOOK_08_BLOCK_4_ZERO_GRAD_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_BACKWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_OPTIMIZER_STEP_EXECUTED,
        not NOTEBOOK_08_BLOCK_4_PARAMETER_UPDATE_EXECUTED,
    ]
)


if not NOTEBOOK_08_BLOCK_4_VALID:

    raise RuntimeError(
        "Notebook 08 Block 4 transformative-mechanism "
        "architecture validation failed."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_4_COMPLETE = True


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 4: "
    "Transformative-Mechanism Architectural Contract "
    "and Module Construction"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_4_VERSION}"
)

print("-" * 72)

print(
    "Transformative architecture"
)

print(
    f"Factual pathway             : "
    f"20 -> 20 -> 10"
)

print(
    f"Psychological pathway       : "
    f"68 -> 68 -> 34"
)

print(
    f"Social pathway              : "
    f"14 -> 14 -> 7"
)

print(
    f"Activation                  : "
    f"GELU"
)

print(
    f"Dropout                     : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_DROPOUT:.2f}"
)

print(
    f"Normalisation               : "
    f"LayerNorm(hidden_dim)"
)

print("-" * 72)

print(
    "Pathway semantic dimensions"
)

print(
    f"Factual representation      : "
    f"{NOTEBOOK_08_FACTUAL_TRANSFORM_DIM}"
)

print(
    f"Factual preceding state     : "
    f"{NOTEBOOK_08_FACTUAL_TRANSFORM_DIM}"
)

print(
    f"Factual candidate transform : "
    f"{NOTEBOOK_08_FACTUAL_TRANSFORM_DIM}"
)

print(
    f"Psychological representation: "
    f"{NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM}"
)

print(
    f"Psychological preceding state: "
    f"{NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM}"
)

print(
    f"Psychological candidate     : "
    f"{NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM}"
)

print(
    f"Social representation       : "
    f"{NOTEBOOK_08_SOCIAL_TRANSFORM_DIM}"
)

print(
    f"Social preceding state      : "
    f"{NOTEBOOK_08_SOCIAL_TRANSFORM_DIM}"
)

print(
    f"Social candidate transform  : "
    f"{NOTEBOOK_08_SOCIAL_TRANSFORM_DIM}"
)

print("-" * 72)

print(
    "Transformative parameters"
)

for (
    pathway_name,
    parameter_count,
) in BLOCK_4_TRANSFORMATIVE_PARAMETER_COUNTS.items():

    print(
        f"{pathway_name:<28}: "
        f"{parameter_count:,} parameters"
    )

print(
    f"Total transformative params : "
    f"{BLOCK_4_TOTAL_TRANSFORMATIVE_PARAMETERS:,}"
)

print("-" * 72)

print(
    f"Pathways structurally distinct: "
    f"{BLOCK_4_PATHWAY_OBJECTS_DISTINCT}"
)

print(
    f"Parameter sets independent  : "
    f"{BLOCK_4_PATHWAY_PARAMETERS_DISTINCT}"
)

print(
    f"Pathway dimensions valid    : "
    f"{BLOCK_4_ALL_PATHWAY_DIMENSIONS_VALID}"
)

print(
    f"Output dimensions preserved : "
    f"{BLOCK_4_TRANSFORMATION_OUTPUT_DIM_PRESERVED}"
)

print(
    f"State dimensions preserved  : "
    f"{BLOCK_4_STATE_DIM_PRESERVED}"
)

print(
    f"Device placement valid      : "
    f"{BLOCK_4_TRANSFORMATIVE_DEVICE_VALID}"
)

print("-" * 72)

print(
    f"Representation model frozen : "
    f"{BLOCK_4_ALL_REPRESENTATION_MODULES_FROZEN}"
)

print(
    f"Representation state unchanged: "
    f"{BLOCK_4_ALL_REPRESENTATION_MODULES_UNCHANGED}"
)

print(
    f"Transform modules trainable : "
    f"{BLOCK_4_ALL_TRANSFORMATIVE_MODULES_TRAINABLE}"
)

print("-" * 72)

print(
    f"Representation forward pass : "
    f"{NOTEBOOK_08_BLOCK_4_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform forward pass      : "
    f"{NOTEBOOK_08_BLOCK_4_TRANSFORM_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform weights created   : "
    f"{NOTEBOOK_08_BLOCK_4_TRANSFORMATION_WEIGHTS_INSTANTIATED}"
)

print(
    f"Recurrent state created     : "
    f"{NOTEBOOK_08_BLOCK_4_RECURRENT_STATE_INSTANTIATED}"
)

print(
    f"Optimizer created           : "
    f"{NOTEBOOK_08_BLOCK_4_OPTIMIZER_CREATED}"
)

print(
    f"Parameter update executed   : "
    f"{NOTEBOOK_08_BLOCK_4_PARAMETER_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Transform architecture ready: "
    f"{NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_4_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_4_COMPLETE}"
)

print("=" * 72)

print(
    "Independent factual, psychological and social transformative "
    "mechanisms were constructed successfully."
)

print(
    "Each mechanism combines the current pathway representation with "
    "a same-dimensional preceding state and produces a same-dimensional "
    "candidate transformation."
)

print(
    "Candidate transformations remain distinct from Transformative "
    "Weights and from recurrent state updates."
)

print(
    "The inherited 894,003-parameter representation model remained "
    "frozen and exactly unchanged."
)

print(
    "No representation or transformative forward pass was executed."
)

print(
    "No Transformative Weight, recurrent state, loss, optimiser, "
    "backward pass or parameter update was introduced."
)

print(
    "Notebook 08 may now proceed to controlled forward-path validation "
    "of the newly constructed transformative mechanisms."
)

print("=" * 72)

Media AI — Notebook 08, Block 4: Transformative-Mechanism Architectural Contract and Module Construction
Block version               : 1.0
------------------------------------------------------------------------
Transformative architecture
Factual pathway             : 20 -> 20 -> 10
Psychological pathway       : 68 -> 68 -> 34
Social pathway              : 14 -> 14 -> 7
Activation                  : GELU
Dropout                     : 0.10
Normalisation               : LayerNorm(hidden_dim)
------------------------------------------------------------------------
Pathway semantic dimensions
Factual representation      : 10
Factual preceding state     : 10
Factual candidate transform : 10
Psychological representation: 34
Psychological preceding state: 34
Psychological candidate     : 34
Social representation       : 7
Social preceding state      : 7
Social candidate transform  : 7
------------------------------------------------------------------------
Transformative parameters
factual  

## Block 5 — Controlled Transformative Forward-Path Validation

This block performs the first controlled numerical execution of the transformative mechanisms constructed in Block 4.

The objective is not to train the mechanisms or to propagate an actual media sequence. Instead, the block verifies that each pathway implements the intended transformation contract numerically and that the resulting candidate transformations respond correctly to both components of their input:

1. the representation of the current observation; and
2. the state that precedes that observation.

The validation therefore tests the transformative mechanism as a function before any Transformative Weight or recurrent update mechanism is introduced.

### Transformative forward-path contract

For each pathway

$$
k \in \{F,P,S\},
$$

Block 4 established a learned mapping

$$
\mathcal{T}^{(k)}
:
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(k)}_t.
$$

The corresponding numerical forward operation is

$$
\boldsymbol{\tau}^{(k)}_t
=
\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1};
\theta_T^{(k)}
\right),
$$

where $\theta_T^{(k)}$ denotes the parameters of the pathway-specific transformative mechanism.

The current representation and preceding state are concatenated internally:

$$
\mathbf{x}^{(k)}_t
=
\left[
\mathbf{r}^{(k)}_t
\mathbin{\|}
\mathbf{h}^{(k)}_{t-1}
\right].
$$

The resulting candidate transformation remains in the same semantic space:

$$
\boldsymbol{\tau}^{(F)}_t \in \mathbb{R}^{10},
\qquad
\boldsymbol{\tau}^{(P)}_t \in \mathbb{R}^{34},
\qquad
\boldsymbol{\tau}^{(S)}_t \in \mathbb{R}^{7}.
$$

No dimensional expansion introduced internally by the transformation network is therefore exposed beyond the module boundary.

### Deterministic probe construction

The forward path is validated with controlled deterministic probe tensors rather than with corpus observations.

For each pathway, the block constructs a current-representation probe

$$
\tilde{\mathbf{r}}^{(k)}
$$

and a preceding-state probe

$$
\tilde{\mathbf{h}}^{(k)}.
$$

These probes have exactly the dimensions required by the corresponding semantic space.

Their purpose is architectural validation rather than substantive interpretation. The resulting candidate transformations must therefore not be interpreted as factual, psychological, or social predictions about real media content.

Using controlled probes isolates the transformative mechanism from data loading, representation generation, recurrent propagation, and supervision. A failure can consequently be attributed directly to the transformative forward path rather than to an upstream or downstream component.

### Baseline forward-path validation

For each pathway, the primary probe pair is evaluated as

$$
\tilde{\boldsymbol{\tau}}^{(k)}
=
\mathcal{T}^{(k)}
\left(
\tilde{\mathbf{r}}^{(k)},
\tilde{\mathbf{h}}^{(k)}
\right).
$$

The resulting tensor must satisfy the pathway-specific dimensional contract:

$$
\operatorname{dim}
\left(
\tilde{\boldsymbol{\tau}}^{(k)}
\right)
=
\operatorname{dim}
\left(
\tilde{\mathbf{r}}^{(k)}
\right)
=
\operatorname{dim}
\left(
\tilde{\mathbf{h}}^{(k)}
\right).
$$

All output values must additionally be finite:

$$
\tilde{\boldsymbol{\tau}}^{(k)}
\in
\mathbb{R}^{d_k},
$$

with no `NaN` or infinite values.

This verifies that each constructed transformative pathway can execute a numerically valid candidate-transformation forward pass.

### Dependence on the current representation

A genuine transformative mechanism must respond to the current observation.

To test this property, the preceding state is held fixed while the current representation is changed:

$$
\boldsymbol{\tau}^{(k)}_a
=
\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}_a,
\mathbf{h}^{(k)}
\right),
$$

$$
\boldsymbol{\tau}^{(k)}_b
=
\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}_b,
\mathbf{h}^{(k)}
\right).
$$

For distinct representation probes,

$$
\mathbf{r}^{(k)}_a
\neq
\mathbf{r}^{(k)}_b,
$$

the mechanism should produce a corresponding numerical response:

$$
\boldsymbol{\tau}^{(k)}_a
\neq
\boldsymbol{\tau}^{(k)}_b.
$$

This test verifies that the current representation participates materially in the candidate-transformation function.

### Dependence on preceding state

The complementary test verifies that the mechanism is genuinely state-conditioned.

The current representation is held fixed while the preceding state is changed:

$$
\boldsymbol{\tau}^{(k)}_a
=
\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)},
\mathbf{h}^{(k)}_a
\right),
$$

$$
\boldsymbol{\tau}^{(k)}_b
=
\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)},
\mathbf{h}^{(k)}_b
\right).
$$

For distinct preceding-state probes,

$$
\mathbf{h}^{(k)}_a
\neq
\mathbf{h}^{(k)}_b,
$$

the resulting candidate transformations should differ:

$$
\boldsymbol{\tau}^{(k)}_a
\neq
\boldsymbol{\tau}^{(k)}_b.
$$

This validation is particularly important.

Without state dependence, the architecture would effectively collapse to an additional representation-level projection:

$$
\boldsymbol{\tau}^{(k)}_t
\approx
f
\left(
\mathbf{r}^{(k)}_t
\right),
$$

rather than implementing the intended contextual transformation:

$$
\boldsymbol{\tau}^{(k)}_t
=
f
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1}
\right).
$$

The state-sensitivity test therefore provides direct numerical evidence that the candidate transformation depends on the condition into which the current observation arrives.

### Pathway independence

The factual, psychological, and social transformative mechanisms remain separate during forward validation.

The block evaluates

$$
\mathcal{T}^{(F)},
\qquad
\mathcal{T}^{(P)},
\qquad
\mathcal{T}^{(S)}
$$

independently and verifies their respective output contracts.

No representation or state from one semantic pathway is supplied to another pathway.

The information flow remains:

$$
\left(
\mathbf{r}^{(F)},
\mathbf{h}^{(F)}
\right)
\rightarrow
\boldsymbol{\tau}^{(F)},
$$

$$
\left(
\mathbf{r}^{(P)},
\mathbf{h}^{(P)}
\right)
\rightarrow
\boldsymbol{\tau}^{(P)},
$$

$$
\left(
\mathbf{r}^{(S)},
\mathbf{h}^{(S)}
\right)
\rightarrow
\boldsymbol{\tau}^{(S)}.
$$

This preserves the semantic separation established by the representation architecture.

### Deterministic evaluation boundary

Because the transformative modules contain dropout, their behaviour differs between training and evaluation modes.

Controlled validation is therefore performed in evaluation mode:

$$
\mathcal{T}^{(k)}.\operatorname{eval}().
$$

This disables stochastic dropout behaviour during the probe forward passes and allows repeated evaluation of an identical input pair to be tested for exact reproducibility.

For an unchanged deterministic probe pair,

$$
\left(
\tilde{\mathbf{r}}^{(k)},
\tilde{\mathbf{h}}^{(k)}
\right),
$$

repeated evaluation must satisfy

$$
\mathcal{T}^{(k)}
\left(
\tilde{\mathbf{r}}^{(k)},
\tilde{\mathbf{h}}^{(k)}
\right)
=
\mathcal{T}^{(k)}
\left(
\tilde{\mathbf{r}}^{(k)},
\tilde{\mathbf{h}}^{(k)}
\right).
$$

The modules may subsequently be returned to their intended training state after validation. Evaluation mode in this block is therefore a validation condition rather than a permanent freezing policy for the newly introduced transformative parameters.

### Parameter immutability during validation

Although Block 5 executes the transformative forward path, it does not train the transformative mechanisms.

No loss function is evaluated, no optimiser is created, no backward pass is executed, and no parameter update occurs.

The transformative parameter state before validation,

$$
\theta^{(k)}_{T,\mathrm{before}},
$$

must therefore equal the parameter state after validation,

$$
\theta^{(k)}_{T,\mathrm{after}}.
$$

Accordingly,

$$
\theta^{(k)}_{T,\mathrm{before}}
=
\theta^{(k)}_{T,\mathrm{after}}.
$$

The inherited representation model must likewise remain frozen and unchanged.

This ensures that Block 5 tests behaviour without modifying either the inherited representation system or the newly constructed transformative mechanisms.

### Candidate transformation remains distinct from recurrent state

The output produced in this block is still only

$$
\boldsymbol{\tau}^{(k)}_t.
$$

It must not be interpreted as

$$
\mathbf{h}^{(k)}_t.
$$

In particular, Block 5 does not apply an update such as

$$
\mathbf{h}^{(k)}_t
=
\mathbf{h}^{(k)}_{t-1}
+
TW^{(k)}_t
\boldsymbol{\tau}^{(k)}_t.
$$

Neither the Transformative Weight

$$
TW^{(k)}_t
$$

nor the recurrent update rule belongs to Notebook 08.

Those mechanisms are reserved for Notebook 09, where candidate transformations will be integrated into the explicit Transformative Weight and recurrent-state architecture.

### Block validation contract

Block 5 is considered valid only if all three pathways satisfy the controlled forward-path contract.

The validation requires:

- correct factual, psychological, and social output dimensions;
- finite candidate-transformation values;
- deterministic repeated evaluation under identical inputs;
- numerical sensitivity to changes in the current representation;
- numerical sensitivity to changes in the preceding state;
- preservation of pathway independence;
- unchanged transformative parameters throughout validation;
- unchanged and frozen inherited representation-model parameters;
- absence of loss calculation, optimisation, backward propagation, and parameter updates.

At completion, Notebook 08 will therefore have demonstrated that

$$
\mathcal{T}^{(F)},
\qquad
\mathcal{T}^{(P)},
\qquad
\mathcal{T}^{(S)}
$$

are not merely structurally valid modules, but executable state-conditioned transformation functions satisfying the intended dimensional and causal interface.

The validated boundary remains:

$$
\boxed{
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(k)}_t
}
$$

while

$$
\boxed{
\boldsymbol{\tau}^{(k)}_t
\longrightarrow
TW^{(k)}_t
\longrightarrow
\mathbf{h}^{(k)}_t
}
$$

remains explicitly outside Notebook 08 and is deferred to Notebook 09.

In [5]:
# =============================================================================
# Media AI — Notebook 08, Block 5
# Controlled Transformative Forward-Path Validation
# =============================================================================

from copy import deepcopy

import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_5 = 5

NOTEBOOK_08_BLOCK_5_NAME = (
    "Controlled Transformative Forward-Path Validation"
)

NOTEBOOK_08_BLOCK_5_VERSION = "1.0"


# =============================================================================
# Dependency checks
# =============================================================================

required_block_5_objects = [
    # -------------------------------------------------------------------------
    # Block 1 — deterministic runtime
    # -------------------------------------------------------------------------
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Block 3 — restored representation model
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_3_COMPLETE",
    "NOTEBOOK_08_BLOCK_3_VALID",
    "NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED",

    "NOTEBOOK_08_RESTORED_BACKBONE",
    "NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT",
    "NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD",
    "NOTEBOOK_08_RESTORED_FACTUAL_HEAD",
    "NOTEBOOK_08_RESTORED_SOCIAL_HEAD",

    # -------------------------------------------------------------------------
    # Block 4 — transformative architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_4_COMPLETE",
    "NOTEBOOK_08_BLOCK_4_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY",

    "NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM",

    "NOTEBOOK_08_FACTUAL_TRANSFORM_DIM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM",
    "NOTEBOOK_08_SOCIAL_TRANSFORM_DIM",
]


missing_block_5_objects = [
    object_name
    for object_name
    in required_block_5_objects
    if object_name not in globals()
]


if missing_block_5_objects:

    raise NameError(
        "Notebook 08 Block 5 prerequisites are not initialised. "
        f"Missing: {missing_block_5_objects}"
    )


if not all(
    [
        NOTEBOOK_08_BLOCK_3_COMPLETE,
        NOTEBOOK_08_BLOCK_3_VALID,
        NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED,
        NOTEBOOK_08_BLOCK_4_COMPLETE,
        NOTEBOOK_08_BLOCK_4_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY,
    ]
):

    raise RuntimeError(
        "Notebook 08 Blocks 3 and 4 must be complete and valid "
        "before Block 5 forward-path validation."
    )


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_08_BLOCK_5_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_5_TRANSFORM_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_5_LOSS_CALCULATED = False

NOTEBOOK_08_BLOCK_5_OPTIMIZER_CREATED = False

NOTEBOOK_08_BLOCK_5_ZERO_GRAD_EXECUTED = False

NOTEBOOK_08_BLOCK_5_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_5_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_08_BLOCK_5_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_08_BLOCK_5_TRANSFORMATION_WEIGHTS_INSTANTIATED = False

NOTEBOOK_08_BLOCK_5_TRANSFORMATION_WEIGHTS_EXECUTED = False

NOTEBOOK_08_BLOCK_5_RECURRENT_STATE_INSTANTIATED = False

NOTEBOOK_08_BLOCK_5_RECURRENT_UPDATE_EXECUTED = False


# =============================================================================
# Controlled validation modules
# =============================================================================

BLOCK_5_TRANSFORMATIVE_MODULES = {
    "factual":
        NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM,

    "psychological":
        NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,

    "social":
        NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM,
}


BLOCK_5_PATHWAY_DIMS = {
    "factual":
        NOTEBOOK_08_FACTUAL_TRANSFORM_DIM,

    "psychological":
        NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM,

    "social":
        NOTEBOOK_08_SOCIAL_TRANSFORM_DIM,
}


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_5_snapshot_module(
    module,
):

    return {
        parameter_name:
            parameter.detach()
            .cpu()
            .clone()

        for (
            parameter_name,
            parameter,
        ) in module.named_parameters()
    }


def block_5_states_equal(
    before_state,
    after_state,
):

    if (
        tuple(
            before_state.keys()
        )
        !=
        tuple(
            after_state.keys()
        )
    ):

        return False


    return all(
        torch.equal(
            before_state[
                parameter_name
            ],
            after_state[
                parameter_name
            ],
        )

        for parameter_name
        in before_state
    )


# =============================================================================
# Snapshot transformative state before validation
# =============================================================================

BLOCK_5_TRANSFORMATIVE_STATE_BEFORE = {
    pathway_name:
        block_5_snapshot_module(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_5_TRANSFORMATIVE_MODULES.items()
}


# =============================================================================
# Snapshot inherited representation state before validation
# =============================================================================

BLOCK_5_REPRESENTATION_MODULES = {
    "backbone":
        NOTEBOOK_08_RESTORED_BACKBONE,

    "global_confluent":
        NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,

    "psychological_head":
        NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,

    "factual_head":
        NOTEBOOK_08_RESTORED_FACTUAL_HEAD,

    "social_head":
        NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
}


BLOCK_5_REPRESENTATION_STATE_BEFORE = {
    module_name:
        block_5_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_5_REPRESENTATION_MODULES.items()
}


# =============================================================================
# Preserve original transformative module modes
# =============================================================================

BLOCK_5_ORIGINAL_TRAINING_MODES = {
    pathway_name:
        module.training

    for (
        pathway_name,
        module,
    ) in BLOCK_5_TRANSFORMATIVE_MODULES.items()
}


# =============================================================================
# Controlled deterministic evaluation mode
# =============================================================================

for module in BLOCK_5_TRANSFORMATIVE_MODULES.values():

    module.eval()


BLOCK_5_EVAL_MODE_VALID = all(
    not module.training

    for module
    in BLOCK_5_TRANSFORMATIVE_MODULES.values()
)


if not BLOCK_5_EVAL_MODE_VALID:

    raise RuntimeError(
        "One or more transformative modules could not be placed "
        "in evaluation mode."
    )


# =============================================================================
# Deterministic probe construction
# =============================================================================
#
# Probe A:
#   representation = deterministic linear ramp
#   state          = deterministic reversed linear ramp
#
# Probe B:
#   representation perturbation with state fixed
#
# Probe C:
#   state perturbation with representation fixed
#
# Probes are architectural validation tensors only.
# =============================================================================

BLOCK_5_PROBES = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_5_PATHWAY_DIMS.items():

    representation_probe = torch.linspace(
        -1.0,
        1.0,
        steps=
            pathway_dim,
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
    ).unsqueeze(
        0
    )


    state_probe = torch.linspace(
        1.0,
        -1.0,
        steps=
            pathway_dim,
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
    ).unsqueeze(
        0
    )


    representation_perturbation = torch.full(
        (
            1,
            pathway_dim,
        ),
        0.125,
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
    )


    state_perturbation = torch.full(
        (
            1,
            pathway_dim,
        ),
        -0.175,
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
    )


    BLOCK_5_PROBES[
        pathway_name
    ] = {
        "representation":
            representation_probe,

        "state":
            state_probe,

        "representation_changed":
            (
                representation_probe
                +
                representation_perturbation
            ),

        "state_changed":
            (
                state_probe
                +
                state_perturbation
            ),
    }


# =============================================================================
# Probe dimension validation
# =============================================================================

BLOCK_5_PROBE_DIMENSIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_5_PATHWAY_DIMS.items():

    probes = (
        BLOCK_5_PROBES[
            pathway_name
        ]
    )


    BLOCK_5_PROBE_DIMENSIONS_VALID[
        pathway_name
    ] = all(
        [
            (
                probes[
                    "representation"
                ].shape
                ==
                (
                    1,
                    pathway_dim,
                )
            ),

            (
                probes[
                    "state"
                ].shape
                ==
                (
                    1,
                    pathway_dim,
                )
            ),

            (
                probes[
                    "representation_changed"
                ].shape
                ==
                (
                    1,
                    pathway_dim,
                )
            ),

            (
                probes[
                    "state_changed"
                ].shape
                ==
                (
                    1,
                    pathway_dim,
                )
            ),
        ]
    )


BLOCK_5_ALL_PROBE_DIMENSIONS_VALID = all(
    BLOCK_5_PROBE_DIMENSIONS_VALID.values()
)


if not BLOCK_5_ALL_PROBE_DIMENSIONS_VALID:

    raise RuntimeError(
        "One or more controlled validation probes violate "
        "the pathway dimensional contract."
    )


# =============================================================================
# Controlled transformative forward passes
# =============================================================================

BLOCK_5_BASELINE_OUTPUTS = {}

BLOCK_5_REPEAT_OUTPUTS = {}

BLOCK_5_REPRESENTATION_CHANGED_OUTPUTS = {}

BLOCK_5_STATE_CHANGED_OUTPUTS = {}


with torch.no_grad():

    for (
        pathway_name,
        module,
    ) in BLOCK_5_TRANSFORMATIVE_MODULES.items():

        probes = (
            BLOCK_5_PROBES[
                pathway_name
            ]
        )


        BLOCK_5_BASELINE_OUTPUTS[
            pathway_name
        ] = module(
            probes[
                "representation"
            ],

            probes[
                "state"
            ],
        )


        BLOCK_5_REPEAT_OUTPUTS[
            pathway_name
        ] = module(
            probes[
                "representation"
            ],

            probes[
                "state"
            ],
        )


        BLOCK_5_REPRESENTATION_CHANGED_OUTPUTS[
            pathway_name
        ] = module(
            probes[
                "representation_changed"
            ],

            probes[
                "state"
            ],
        )


        BLOCK_5_STATE_CHANGED_OUTPUTS[
            pathway_name
        ] = module(
            probes[
                "representation"
            ],

            probes[
                "state_changed"
            ],
        )


NOTEBOOK_08_BLOCK_5_TRANSFORM_FORWARD_PASS_EXECUTED = True


# =============================================================================
# Output shape validation
# =============================================================================

BLOCK_5_OUTPUT_SHAPES_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_5_PATHWAY_DIMS.items():

    expected_shape = (
        1,
        pathway_dim,
    )


    BLOCK_5_OUTPUT_SHAPES_VALID[
        pathway_name
    ] = all(
        [
            (
                BLOCK_5_BASELINE_OUTPUTS[
                    pathway_name
                ].shape
                ==
                expected_shape
            ),

            (
                BLOCK_5_REPEAT_OUTPUTS[
                    pathway_name
                ].shape
                ==
                expected_shape
            ),

            (
                BLOCK_5_REPRESENTATION_CHANGED_OUTPUTS[
                    pathway_name
                ].shape
                ==
                expected_shape
            ),

            (
                BLOCK_5_STATE_CHANGED_OUTPUTS[
                    pathway_name
                ].shape
                ==
                expected_shape
            ),
        ]
    )


BLOCK_5_ALL_OUTPUT_SHAPES_VALID = all(
    BLOCK_5_OUTPUT_SHAPES_VALID.values()
)


if not BLOCK_5_ALL_OUTPUT_SHAPES_VALID:

    raise RuntimeError(
        "One or more transformative outputs violate "
        "the pathway output-dimensional contract."
    )


# =============================================================================
# Numerical finiteness validation
# =============================================================================

BLOCK_5_OUTPUTS_FINITE = {}


for pathway_name in BLOCK_5_TRANSFORMATIVE_MODULES:

    BLOCK_5_OUTPUTS_FINITE[
        pathway_name
    ] = all(
        [
            torch.isfinite(
                BLOCK_5_BASELINE_OUTPUTS[
                    pathway_name
                ]
            ).all().item(),

            torch.isfinite(
                BLOCK_5_REPEAT_OUTPUTS[
                    pathway_name
                ]
            ).all().item(),

            torch.isfinite(
                BLOCK_5_REPRESENTATION_CHANGED_OUTPUTS[
                    pathway_name
                ]
            ).all().item(),

            torch.isfinite(
                BLOCK_5_STATE_CHANGED_OUTPUTS[
                    pathway_name
                ]
            ).all().item(),
        ]
    )


BLOCK_5_ALL_OUTPUTS_FINITE = all(
    BLOCK_5_OUTPUTS_FINITE.values()
)


if not BLOCK_5_ALL_OUTPUTS_FINITE:

    raise RuntimeError(
        "One or more transformative outputs contain "
        "non-finite numerical values."
    )


# =============================================================================
# Deterministic repeated-evaluation validation
# =============================================================================

BLOCK_5_REPEAT_DETERMINISTIC = {
    pathway_name:
        torch.equal(
            BLOCK_5_BASELINE_OUTPUTS[
                pathway_name
            ],

            BLOCK_5_REPEAT_OUTPUTS[
                pathway_name
            ],
        )

    for pathway_name
    in BLOCK_5_TRANSFORMATIVE_MODULES
}


BLOCK_5_ALL_REPEAT_DETERMINISTIC = all(
    BLOCK_5_REPEAT_DETERMINISTIC.values()
)


if not BLOCK_5_ALL_REPEAT_DETERMINISTIC:

    raise RuntimeError(
        "One or more transformative mechanisms are not deterministic "
        "under identical evaluation-mode inputs."
    )


# =============================================================================
# Representation-sensitivity validation
# =============================================================================
#
# The preceding state remains fixed while the current representation changes.
# =============================================================================

BLOCK_5_REPRESENTATION_SENSITIVITY_MAX_DIFF = {
    pathway_name:
        torch.max(
            torch.abs(
                BLOCK_5_REPRESENTATION_CHANGED_OUTPUTS[
                    pathway_name
                ]
                -
                BLOCK_5_BASELINE_OUTPUTS[
                    pathway_name
                ]
            )
        ).item()

    for pathway_name
    in BLOCK_5_TRANSFORMATIVE_MODULES
}


BLOCK_5_REPRESENTATION_SENSITIVE = {
    pathway_name:
        (
            max_difference
            >
            0.0
        )

    for (
        pathway_name,
        max_difference,
    ) in BLOCK_5_REPRESENTATION_SENSITIVITY_MAX_DIFF.items()
}


BLOCK_5_ALL_PATHWAYS_REPRESENTATION_SENSITIVE = all(
    BLOCK_5_REPRESENTATION_SENSITIVE.values()
)


if not BLOCK_5_ALL_PATHWAYS_REPRESENTATION_SENSITIVE:

    raise RuntimeError(
        "One or more transformative mechanisms failed to respond "
        "to a controlled change in the current representation."
    )


# =============================================================================
# State-sensitivity validation
# =============================================================================
#
# The current representation remains fixed while the preceding state changes.
# =============================================================================

BLOCK_5_STATE_SENSITIVITY_MAX_DIFF = {
    pathway_name:
        torch.max(
            torch.abs(
                BLOCK_5_STATE_CHANGED_OUTPUTS[
                    pathway_name
                ]
                -
                BLOCK_5_BASELINE_OUTPUTS[
                    pathway_name
                ]
            )
        ).item()

    for pathway_name
    in BLOCK_5_TRANSFORMATIVE_MODULES
}


BLOCK_5_STATE_SENSITIVE = {
    pathway_name:
        (
            max_difference
            >
            0.0
        )

    for (
        pathway_name,
        max_difference,
    ) in BLOCK_5_STATE_SENSITIVITY_MAX_DIFF.items()
}


BLOCK_5_ALL_PATHWAYS_STATE_SENSITIVE = all(
    BLOCK_5_STATE_SENSITIVE.values()
)


if not BLOCK_5_ALL_PATHWAYS_STATE_SENSITIVE:

    raise RuntimeError(
        "One or more transformative mechanisms failed to respond "
        "to a controlled change in the preceding state."
    )


# =============================================================================
# Pathway independence validation
# =============================================================================

BLOCK_5_PATHWAY_OBJECTS_DISTINCT = (
    len(
        {
            id(
                module
            )

            for module
            in BLOCK_5_TRANSFORMATIVE_MODULES.values()
        }
    )
    ==
    3
)


BLOCK_5_PARAMETER_OBJECT_IDS = {
    pathway_name:
        {
            id(
                parameter
            )

            for parameter
            in module.parameters()
        }

    for (
        pathway_name,
        module,
    ) in BLOCK_5_TRANSFORMATIVE_MODULES.items()
}


BLOCK_5_PATHWAY_PARAMETERS_DISTINCT = all(
    [
        BLOCK_5_PARAMETER_OBJECT_IDS[
            "factual"
        ].isdisjoint(
            BLOCK_5_PARAMETER_OBJECT_IDS[
                "psychological"
            ]
        ),

        BLOCK_5_PARAMETER_OBJECT_IDS[
            "factual"
        ].isdisjoint(
            BLOCK_5_PARAMETER_OBJECT_IDS[
                "social"
            ]
        ),

        BLOCK_5_PARAMETER_OBJECT_IDS[
            "psychological"
        ].isdisjoint(
            BLOCK_5_PARAMETER_OBJECT_IDS[
                "social"
            ]
        ),
    ]
)


BLOCK_5_PATHWAY_INDEPENDENCE_VALID = all(
    [
        BLOCK_5_PATHWAY_OBJECTS_DISTINCT,
        BLOCK_5_PATHWAY_PARAMETERS_DISTINCT,
    ]
)


if not BLOCK_5_PATHWAY_INDEPENDENCE_VALID:

    raise RuntimeError(
        "Transformative pathway independence was not preserved "
        "during controlled forward validation."
    )


# =============================================================================
# Transformative-state immutability validation
# =============================================================================

BLOCK_5_TRANSFORMATIVE_STATE_AFTER = {
    pathway_name:
        block_5_snapshot_module(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_5_TRANSFORMATIVE_MODULES.items()
}


BLOCK_5_TRANSFORMATIVE_PARAMETERS_UNCHANGED = {
    pathway_name:
        block_5_states_equal(
            BLOCK_5_TRANSFORMATIVE_STATE_BEFORE[
                pathway_name
            ],

            BLOCK_5_TRANSFORMATIVE_STATE_AFTER[
                pathway_name
            ],
        )

    for pathway_name
    in BLOCK_5_TRANSFORMATIVE_MODULES
}


BLOCK_5_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED = all(
    BLOCK_5_TRANSFORMATIVE_PARAMETERS_UNCHANGED.values()
)


if not BLOCK_5_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED:

    raise RuntimeError(
        "One or more transformative parameters changed during "
        "controlled forward-path validation."
    )


# =============================================================================
# Inherited representation-state immutability validation
# =============================================================================

BLOCK_5_REPRESENTATION_STATE_AFTER = {
    module_name:
        block_5_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_5_REPRESENTATION_MODULES.items()
}


BLOCK_5_REPRESENTATION_PARAMETERS_UNCHANGED = {
    module_name:
        block_5_states_equal(
            BLOCK_5_REPRESENTATION_STATE_BEFORE[
                module_name
            ],

            BLOCK_5_REPRESENTATION_STATE_AFTER[
                module_name
            ],
        )

    for module_name
    in BLOCK_5_REPRESENTATION_MODULES
}


BLOCK_5_ALL_REPRESENTATION_PARAMETERS_UNCHANGED = all(
    BLOCK_5_REPRESENTATION_PARAMETERS_UNCHANGED.values()
)


if not BLOCK_5_ALL_REPRESENTATION_PARAMETERS_UNCHANGED:

    raise RuntimeError(
        "One or more inherited representation-model parameters "
        "changed during Block 5."
    )


# =============================================================================
# Frozen inherited representation-model validation
# =============================================================================

BLOCK_5_REPRESENTATION_MODULES_FROZEN = {
    module_name:
        all(
            not parameter.requires_grad

            for parameter
            in module.parameters()
        )

    for (
        module_name,
        module,
    ) in BLOCK_5_REPRESENTATION_MODULES.items()
}


BLOCK_5_ALL_REPRESENTATION_MODULES_FROZEN = all(
    BLOCK_5_REPRESENTATION_MODULES_FROZEN.values()
)


if not BLOCK_5_ALL_REPRESENTATION_MODULES_FROZEN:

    raise RuntimeError(
        "The inherited representation model is no longer frozen."
    )


# =============================================================================
# Restore transformative module training modes
# =============================================================================

for (
    pathway_name,
    module,
) in BLOCK_5_TRANSFORMATIVE_MODULES.items():

    module.train(
        BLOCK_5_ORIGINAL_TRAINING_MODES[
            pathway_name
        ]
    )


BLOCK_5_TRAINING_MODES_RESTORED = all(
    (
        module.training
        ==
        BLOCK_5_ORIGINAL_TRAINING_MODES[
            pathway_name
        ]
    )

    for (
        pathway_name,
        module,
    ) in BLOCK_5_TRANSFORMATIVE_MODULES.items()
)


if not BLOCK_5_TRAINING_MODES_RESTORED:

    raise RuntimeError(
        "Transformative module training modes were not restored "
        "after controlled validation."
    )


# =============================================================================
# Controlled forward-path readiness
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED = all(
    [
        BLOCK_5_ALL_PROBE_DIMENSIONS_VALID,
        BLOCK_5_ALL_OUTPUT_SHAPES_VALID,
        BLOCK_5_ALL_OUTPUTS_FINITE,
        BLOCK_5_ALL_REPEAT_DETERMINISTIC,
        BLOCK_5_ALL_PATHWAYS_REPRESENTATION_SENSITIVE,
        BLOCK_5_ALL_PATHWAYS_STATE_SENSITIVE,
        BLOCK_5_PATHWAY_INDEPENDENCE_VALID,
        BLOCK_5_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED,
        BLOCK_5_ALL_REPRESENTATION_PARAMETERS_UNCHANGED,
        BLOCK_5_ALL_REPRESENTATION_MODULES_FROZEN,
        BLOCK_5_TRAINING_MODES_RESTORED,
    ]
)


# =============================================================================
# Final Block 5 validation
# =============================================================================

NOTEBOOK_08_BLOCK_5_VALID = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED,

        NOTEBOOK_08_BLOCK_5_TRANSFORM_FORWARD_PASS_EXECUTED,

        not NOTEBOOK_08_BLOCK_5_REPRESENTATION_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_5_LOSS_CALCULATED,
        not NOTEBOOK_08_BLOCK_5_OPTIMIZER_CREATED,
        not NOTEBOOK_08_BLOCK_5_ZERO_GRAD_EXECUTED,
        not NOTEBOOK_08_BLOCK_5_BACKWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_5_OPTIMIZER_STEP_EXECUTED,
        not NOTEBOOK_08_BLOCK_5_PARAMETER_UPDATE_EXECUTED,
        not NOTEBOOK_08_BLOCK_5_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_5_TRANSFORMATION_WEIGHTS_EXECUTED,
        not NOTEBOOK_08_BLOCK_5_RECURRENT_STATE_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_5_RECURRENT_UPDATE_EXECUTED,
    ]
)


if not NOTEBOOK_08_BLOCK_5_VALID:

    raise RuntimeError(
        "Notebook 08 Block 5 controlled transformative "
        "forward-path validation failed."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_5_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_08_BLOCK_5_SUMMARY = {
    "block":
        NOTEBOOK_08_BLOCK_5,

    "block_name":
        NOTEBOOK_08_BLOCK_5_NAME,

    "version":
        NOTEBOOK_08_BLOCK_5_VERSION,

    "forward_path_validated":
        NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED,

    "representation_sensitive":
        deepcopy(
            BLOCK_5_REPRESENTATION_SENSITIVE
        ),

    "state_sensitive":
        deepcopy(
            BLOCK_5_STATE_SENSITIVE
        ),

    "repeat_deterministic":
        deepcopy(
            BLOCK_5_REPEAT_DETERMINISTIC
        ),

    "transformative_parameters_unchanged":
        BLOCK_5_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED,

    "representation_parameters_unchanged":
        BLOCK_5_ALL_REPRESENTATION_PARAMETERS_UNCHANGED,

    "transformative_weights_instantiated":
        False,

    "recurrent_state_instantiated":
        False,

    "block_valid":
        NOTEBOOK_08_BLOCK_5_VALID,

    "block_complete":
        NOTEBOOK_08_BLOCK_5_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 5: "
    "Controlled Transformative Forward-Path Validation"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_5_VERSION}"
)

print("-" * 72)

print(
    "Controlled probe dimensions"
)

for (
    pathway_name,
    pathway_dim,
) in BLOCK_5_PATHWAY_DIMS.items():

    print(
        f"{pathway_name:<28}: "
        f"(1, {pathway_dim})"
    )

print("-" * 72)

print(
    "Forward-path validation"
)

for pathway_name in BLOCK_5_TRANSFORMATIVE_MODULES:

    print(
        f"{pathway_name:<14} output shape valid : "
        f"{BLOCK_5_OUTPUT_SHAPES_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} output finite      : "
        f"{BLOCK_5_OUTPUTS_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{BLOCK_5_REPEAT_DETERMINISTIC[pathway_name]}"
    )

print("-" * 72)

print(
    "Current-representation sensitivity"
)

for pathway_name in BLOCK_5_TRANSFORMATIVE_MODULES:

    print(
        f"{pathway_name:<14} responsive         : "
        f"{BLOCK_5_REPRESENTATION_SENSITIVE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} max abs difference : "
        f"{BLOCK_5_REPRESENTATION_SENSITIVITY_MAX_DIFF[pathway_name]:.10f}"
    )

print("-" * 72)

print(
    "Preceding-state sensitivity"
)

for pathway_name in BLOCK_5_TRANSFORMATIVE_MODULES:

    print(
        f"{pathway_name:<14} responsive         : "
        f"{BLOCK_5_STATE_SENSITIVE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} max abs difference : "
        f"{BLOCK_5_STATE_SENSITIVITY_MAX_DIFF[pathway_name]:.10f}"
    )

print("-" * 72)

print(
    f"Pathways independent        : "
    f"{BLOCK_5_PATHWAY_INDEPENDENCE_VALID}"
)

print(
    f"Transform params unchanged  : "
    f"{BLOCK_5_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED}"
)

print(
    f"Representation unchanged    : "
    f"{BLOCK_5_ALL_REPRESENTATION_PARAMETERS_UNCHANGED}"
)

print(
    f"Representation frozen       : "
    f"{BLOCK_5_ALL_REPRESENTATION_MODULES_FROZEN}"
)

print(
    f"Training modes restored     : "
    f"{BLOCK_5_TRAINING_MODES_RESTORED}"
)

print("-" * 72)

print(
    f"Transform forward executed  : "
    f"{NOTEBOOK_08_BLOCK_5_TRANSFORM_FORWARD_PASS_EXECUTED}"
)

print(
    f"Representation forward pass : "
    f"{NOTEBOOK_08_BLOCK_5_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Loss calculated             : "
    f"{NOTEBOOK_08_BLOCK_5_LOSS_CALCULATED}"
)

print(
    f"Optimizer created           : "
    f"{NOTEBOOK_08_BLOCK_5_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed      : "
    f"{NOTEBOOK_08_BLOCK_5_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed   : "
    f"{NOTEBOOK_08_BLOCK_5_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Transform weights created   : "
    f"{NOTEBOOK_08_BLOCK_5_TRANSFORMATION_WEIGHTS_INSTANTIATED}"
)

print(
    f"Recurrent state created     : "
    f"{NOTEBOOK_08_BLOCK_5_RECURRENT_STATE_INSTANTIATED}"
)

print("-" * 72)

print(
    f"Forward path validated      : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_5_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_5_COMPLETE}"
)

print("=" * 72)

print(
    "All three transformative mechanisms executed successfully "
    "under controlled deterministic probes."
)

print(
    "Each pathway produced a finite candidate transformation with "
    "the required pathway-specific dimensionality."
)

print(
    "Repeated identical probe evaluations were deterministic."
)

print(
    "Each mechanism responded independently to controlled changes "
    "in the current representation and in the preceding state."
)

print(
    "The transformative parameters and the inherited representation "
    "model remained unchanged throughout validation."
)

print(
    "No Transformative Weight or recurrent-state update mechanism "
    "was introduced; these remain reserved for Notebook 09."
)

print(
    "No loss, optimiser, backward pass or parameter update was executed."
)

print("=" * 72)

Media AI — Notebook 08, Block 5: Controlled Transformative Forward-Path Validation
Block version               : 1.0
------------------------------------------------------------------------
Controlled probe dimensions
factual                     : (1, 10)
psychological               : (1, 34)
social                      : (1, 7)
------------------------------------------------------------------------
Forward-path validation
factual        output shape valid : True
factual        output finite      : True
factual        deterministic      : True
psychological  output shape valid : True
psychological  output finite      : True
psychological  deterministic      : True
social         output shape valid : True
social         output finite      : True
social         deterministic      : True
------------------------------------------------------------------------
Current-representation sensitivity
factual        responsive         : True
factual        max abs difference : 0.0724539608
psych

## Block 6 — Transformative Objective and Training Policy

This block defines the optimisation contract for the factual, psychological, and social transformative mechanisms introduced and validated in Blocks 4 and 5.

The preceding blocks established that each pathway implements a valid state-conditioned candidate-transformation function:

$$
\boldsymbol{\tau}^{(k)}_t
=========================

\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1}
\right),
\qquad
k \in {F,P,S}.
$$

Block 6 determines how the parameters

$$
\theta_T^{(F)},
\qquad
\theta_T^{(P)},
\qquad
\theta_T^{(S)}
$$

may be optimised while preserving the architectural boundary of Notebook 08.

The block does not yet execute training. It defines the objective families, trainability policy, masking rules, optimisation constraints, and prohibited operations that must govern the subsequent controlled training block.

### Training objective of the transformative mechanism

The transformative mechanism is not trained to reproduce the current representation directly.

Its purpose is to model the change implied by the arrival of a new observation in the context of a preceding state.

Accordingly, the learning problem must distinguish between:

* the current pathway representation;
* the state available immediately before the current observation;
* the candidate transformation generated from their interaction;
* the supervised or derived target describing the intended transformation.

For pathway (k), the generic optimisation problem is

$$
\theta_T^{(k),*}
================

\arg\min_{\theta_T^{(k)}}
\mathcal{L}_T^{(k)}
\left(
\boldsymbol{\tau}^{(k)}*t,
\mathbf{y}^{(k)}*{T,t}
\right),
$$

where

$$
\mathbf{y}^{(k)}_{T,t}
$$

denotes the pathway-specific transformation target available under the current supervision contract.

The objective is therefore defined over the **candidate transformation space**, not over the final recurrent state.

### Separation from Transformative Weights

The optimisation introduced in Notebook 08 must not depend on

$$
TW_F,
\qquad
TW_P,
\qquad
TW_S.
$$

Those quantities belong to Notebook 09.

Therefore, Block 6 does not define an objective of the form

$$
\mathcal{L}
\left(
TW_t^{(k)}
\boldsymbol{\tau}_t^{(k)},
\mathbf{y}_t
\right),
$$

because that would couple candidate-transformation learning with a weighting mechanism that has not yet been introduced.

The Notebook 08 optimisation boundary remains

$$
\left(
\mathbf{r}^{(k)}*t,
\mathbf{h}^{(k)}*{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(k)}_t
\longrightarrow
\mathcal{L}_T^{(k)}.
$$

### Separation from recurrent-state optimisation

The recurrent state

$$
\mathbf{h}^{(k)}_t
$$

is likewise outside the optimisation scope of Notebook 08.

No objective in this notebook is permitted to depend on a recurrent update such as

$$
\mathbf{h}^{(k)}_t
==================

g
\left(
\mathbf{h}^{(k)}_{t-1},
\boldsymbol{\tau}^{(k)}_t,
TW_t^{(k)}
\right).
$$

The preceding state entering the transformative mechanism is therefore treated as an input condition, not as a trainable recurrent state produced by the current notebook.

This separation ensures that the transformation function can be trained and validated independently before the architecture is extended with weighting and recurrent propagation.

### Pathway-specific objective families

The factual, psychological, and social pathways retain distinct semantic spaces and may therefore require different objective families.

The factual pathway operates over the factual representation dimensions:

$$
\boldsymbol{\tau}^{(F)}_t
\in
\mathbb{R}^{10}.
$$

The psychological pathway operates over:

$$
\boldsymbol{\tau}^{(P)}_t
\in
\mathbb{R}^{34}.
$$

The social pathway operates over:

$$
\boldsymbol{\tau}^{(S)}_t
\in
\mathbb{R}^{7}.
$$

The objective applied to each dimension must remain consistent with the semantic type of the corresponding supervision signal.

Where a transformation target is continuous, the default regression objective is a masked squared-error family:

$$
\mathcal{L}_{\mathrm{MSE}}
==========================

\frac{
\sum_i
m_i
\left(
\tau_i-y_i
\right)^2
}{
\sum_i m_i
}.
$$

Where the target is binary and the candidate output is treated as a logit, the corresponding masked binary objective is

$$
\mathcal{L}_{\mathrm{BCE}}
==========================

*

\frac{
\sum_i
m_i
\left[
y_i\log\sigma(\tau_i)
+
(1-y_i)\log(1-\sigma(\tau_i))
\right]
}{
\sum_i m_i
}.
$$

The precise assignment of objective families must follow the supervision contract inherited from the earlier representation stages and must not be inferred from dimensionality alone.

### Mask-aware supervision

The Media AI supervision framework distinguishes between a genuinely observed zero and a target that is unavailable.

That distinction remains mandatory for transformative training.

For every supervised dimension, a mask

$$
m_i \in {0,1}
$$

determines whether the target contributes to the objective.

If

$$
m_i = 0,
$$

the corresponding target is excluded from the loss.

Therefore:

$$
\text{missing}
\neq
0,
$$

and

$$
\text{missing}
\neq
\text{negative}.
$$

Unavailable transformation targets must not silently enter the objective as numerical zeros.

This preserves the methodological boundary established in the earlier supervision and representation notebooks.

### Trainability boundary

The inherited representation model remains frozen throughout transformative optimisation.

For all inherited representation parameters,

$$
\mathrm{requires_grad}
\left(
\theta_{\mathrm{repr}}
\right)
=======

\mathrm{False}.
$$

Only the transformative parameters are eligible for optimisation:

$$
\mathrm{requires_grad}
\left(
\theta_T^{(F)}
\right)
=======

\mathrm{True},
$$

$$
\mathrm{requires_grad}
\left(
\theta_T^{(P)}
\right)
=======

\mathrm{True},
$$

$$
\mathrm{requires_grad}
\left(
\theta_T^{(S)}
\right)
=======

\mathrm{True}.
$$

The intended trainability boundary is therefore

$$
\boxed{
\theta_{\mathrm{repr}}
\text{ frozen}
}
$$

and

$$
\boxed{
\theta_T^{(F)},
\theta_T^{(P)},
\theta_T^{(S)}
\text{ trainable}
}.
$$

No inherited representation parameter may be updated implicitly through the transformative objective.

### Optimisation isolation

The three transformative pathways remain independently parameterised.

A combined training objective may be expressed as

$$
\mathcal{L}_{T,\mathrm{total}}
==============================

\lambda_F
\mathcal{L}_T^{(F)}
+
\lambda_P
\mathcal{L}_T^{(P)}
+
\lambda_S
\mathcal{L}_T^{(S)},
$$

subject to

$$
\lambda_F,\lambda_P,\lambda_S \geq 0
$$

and

$$
\lambda_F+\lambda_P+\lambda_S=1.
$$

The weighting coefficients control contribution to the optimisation objective only.

They must not be confused with the later Transformative Weights

$$
TW_F,
\qquad
TW_P,
\qquad
TW_S.
$$

The symbols serve entirely different architectural roles:

* (\lambda_F,\lambda_P,\lambda_S) are optimisation-loss weights;
* (TW_F,TW_P,TW_S) are future model outputs governing recurrent state transformation.

### Controlled optimisation policy

The subsequent training block must use an explicitly defined optimisation policy.

At minimum, the policy must specify:

* which transformative parameters are trainable;
* the objective associated with each pathway;
* any objective-family weights;
* the optimiser family;
* learning rate;
* weight decay;
* gradient clipping policy;
* number of epochs or optimisation steps;
* deterministic data ordering;
* mask-aware target inclusion;
* checkpointing and reproducibility requirements.

The optimiser must receive only parameters belonging to the transformative mechanisms.

The inherited representation-model parameters must not appear in the optimiser parameter groups.

### Sequence-order boundary

Although the transformative mechanism is state-conditioned, Notebook 08 still does not implement recurrent state propagation.

Training inputs may therefore include a controlled preceding-state condition

$$
\mathbf{h}^{(k)}_{t-1},
$$

but Block 6 does not define a recursive procedure that updates that state across the article sequence.

If sequence-derived state proxies are used during transformative training, they must preserve causal ordering:

$$
t' < t
$$

for all information contributing to the state supplied at position (t).

Future observations remain prohibited.

Thus:

$$
\mathbf{r}*{t+1},
\mathbf{r}*{t+2},
\ldots
$$

must not contribute to the transformation target or preceding-state condition used for position (t).

### Zero-information and unavailable-target policy

A pathway or individual dimension must not be optimised where the current supervision provides no informative target variation.

If a target dimension is:

* entirely unavailable;
* completely masked;
* constant under the current sample;
* or otherwise unable to define a meaningful optimisation signal,

that dimension must be explicitly deferred rather than forced into the loss.

This is the same methodological principle used in earlier factual and social training stages.

Structural presence in the architecture does not imply that every dimension is empirically trainable under every available sample.

### Training does not establish empirical validity

Successful reduction of the transformative training objective would demonstrate that the candidate-transformation parameters can be optimised under the available supervision.

It would not establish:

* generalisation;
* causal validity;
* real-world media-effect validity;
* longitudinal validity;
* population-level psychological validity;
* social-effect validity;
* or validated recurrent behaviour.

Accordingly,

$$
\text{successful optimisation}
\neq
\text{empirical validation}.
$$

This distinction must remain explicit in all subsequent checkpoint metadata and notebook handover artefacts.

### Block 6 execution boundary

Block 6 defines the training contract only.

It does not:

* execute a transformative training forward pass;
* calculate a training loss;
* create an optimiser;
* execute `zero_grad`;
* execute a backward pass;
* clip gradients;
* perform an optimiser step;
* modify transformative parameters;
* modify inherited representation parameters;
* instantiate Transformative Weights;
* instantiate recurrent states;
* perform recurrent propagation.

The block may inspect target availability, masks, dimensions, and parameter trainability in order to establish whether the subsequent optimisation stage is well-defined.

At completion, Notebook 08 should therefore possess a complete and auditable policy describing **what may be trained, against which targets, under which constraints**, while leaving the actual parameter state unchanged.

The resulting transition is:

$$
\boxed{
\text{validated transformative architecture}
}
$$

$$
\downarrow
$$

$$
\boxed{
\text{explicit transformative optimisation contract}
}
$$

while actual optimisation remains reserved for the following block.


In [8]:
# =============================================================================
# Media AI — Notebook 08, Block 6
# Transformative Objective and Training Policy
# =============================================================================

from copy import deepcopy

import math

import torch
import torch.nn as nn


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_6 = 6

NOTEBOOK_08_BLOCK_6_NAME = (
    "Transformative Objective and Training Policy"
)

NOTEBOOK_08_BLOCK_6_VERSION = "1.1"


# =============================================================================
# Dependency checks
# =============================================================================

required_block_6_objects = [
    # -------------------------------------------------------------------------
    # Block 1 — deterministic runtime
    # -------------------------------------------------------------------------
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Block 3 — inherited representation model
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_3_COMPLETE",
    "NOTEBOOK_08_BLOCK_3_VALID",
    "NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED",

    "NOTEBOOK_08_RESTORED_BACKBONE",
    "NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT",
    "NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD",
    "NOTEBOOK_08_RESTORED_FACTUAL_HEAD",
    "NOTEBOOK_08_RESTORED_SOCIAL_HEAD",

    # -------------------------------------------------------------------------
    # Block 4 — transformative architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_4_COMPLETE",
    "NOTEBOOK_08_BLOCK_4_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY",

    "NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM",

    # -------------------------------------------------------------------------
    # Block 5 — controlled forward-path validation
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_5_COMPLETE",
    "NOTEBOOK_08_BLOCK_5_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED",

    # -------------------------------------------------------------------------
    # Dimensional contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_FACTUAL_TRANSFORM_DIM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM",
    "NOTEBOOK_08_SOCIAL_TRANSFORM_DIM",
]


missing_block_6_objects = [
    object_name
    for object_name
    in required_block_6_objects
    if object_name not in globals()
]


if missing_block_6_objects:

    raise NameError(
        "Notebook 08 Block 6 prerequisites are not initialised. "
        f"Missing: {missing_block_6_objects}"
    )


if not all(
    [
        NOTEBOOK_08_BLOCK_3_COMPLETE,
        NOTEBOOK_08_BLOCK_3_VALID,
        NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED,
        NOTEBOOK_08_BLOCK_4_COMPLETE,
        NOTEBOOK_08_BLOCK_4_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY,
        NOTEBOOK_08_BLOCK_5_COMPLETE,
        NOTEBOOK_08_BLOCK_5_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED,
    ]
):

    raise RuntimeError(
        "Notebook 08 Blocks 3–5 must be complete and valid "
        "before Block 6 objective-policy construction."
    )


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_08_BLOCK_6_TRAINING_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_6_LOSS_CALCULATED = False

NOTEBOOK_08_BLOCK_6_OPTIMIZER_CREATED = False

NOTEBOOK_08_BLOCK_6_ZERO_GRAD_EXECUTED = False

NOTEBOOK_08_BLOCK_6_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_6_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_08_BLOCK_6_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_08_BLOCK_6_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_08_BLOCK_6_TRANSFORMATION_WEIGHTS_INSTANTIATED = False

NOTEBOOK_08_BLOCK_6_TRANSFORMATION_WEIGHTS_EXECUTED = False

NOTEBOOK_08_BLOCK_6_RECURRENT_STATE_INSTANTIATED = False

NOTEBOOK_08_BLOCK_6_RECURRENT_UPDATE_EXECUTED = False


# =============================================================================
# Model-group contracts
# =============================================================================

BLOCK_6_REPRESENTATION_MODULES = {
    "backbone":
        NOTEBOOK_08_RESTORED_BACKBONE,

    "global_confluent":
        NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,

    "psychological_head":
        NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,

    "factual_head":
        NOTEBOOK_08_RESTORED_FACTUAL_HEAD,

    "social_head":
        NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
}


BLOCK_6_TRANSFORMATIVE_MODULES = {
    "factual":
        NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM,

    "psychological":
        NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,

    "social":
        NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM,
}


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_6_snapshot_module(
    module,
):

    return {
        parameter_name:
            parameter.detach()
            .cpu()
            .clone()

        for (
            parameter_name,
            parameter,
        ) in module.named_parameters()
    }


def block_6_states_equal(
    before_state,
    after_state,
):

    if (
        tuple(
            before_state.keys()
        )
        !=
        tuple(
            after_state.keys()
        )
    ):

        return False


    return all(
        torch.equal(
            before_state[
                parameter_name
            ],
            after_state[
                parameter_name
            ],
        )

        for parameter_name
        in before_state
    )


# =============================================================================
# Pre-policy parameter snapshots
# =============================================================================

BLOCK_6_REPRESENTATION_STATE_BEFORE = {
    module_name:
        block_6_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_6_REPRESENTATION_MODULES.items()
}


BLOCK_6_TRANSFORMATIVE_STATE_BEFORE = {
    pathway_name:
        block_6_snapshot_module(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_6_TRANSFORMATIVE_MODULES.items()
}


# =============================================================================
# Trainability policy
# =============================================================================
#
# Inherited representation model:
#     frozen
#
# Transformative mechanisms:
#     trainable
# =============================================================================

BLOCK_6_REPRESENTATION_MODULES_FROZEN = {
    module_name:
        all(
            not parameter.requires_grad

            for parameter
            in module.parameters()
        )

    for (
        module_name,
        module,
    ) in BLOCK_6_REPRESENTATION_MODULES.items()
}


BLOCK_6_ALL_REPRESENTATION_MODULES_FROZEN = all(
    BLOCK_6_REPRESENTATION_MODULES_FROZEN.values()
)


BLOCK_6_TRANSFORMATIVE_MODULES_TRAINABLE = {
    pathway_name:
        all(
            parameter.requires_grad

            for parameter
            in module.parameters()
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_6_TRANSFORMATIVE_MODULES.items()
}


BLOCK_6_ALL_TRANSFORMATIVE_MODULES_TRAINABLE = all(
    BLOCK_6_TRANSFORMATIVE_MODULES_TRAINABLE.values()
)


if not BLOCK_6_ALL_REPRESENTATION_MODULES_FROZEN:

    raise RuntimeError(
        "The inherited representation model must remain frozen "
        "under the transformative optimisation policy."
    )


if not BLOCK_6_ALL_TRANSFORMATIVE_MODULES_TRAINABLE:

    raise RuntimeError(
        "All transformative mechanisms must remain trainable "
        "under the Notebook 08 optimisation policy."
    )


# =============================================================================
# Transformative objective-family contract
# =============================================================================
#
# Block 6 defines the admissible objective family for each architectural
# pathway, but it does not yet activate training. Objective activation remains
# contingent on real target and mask validation in a later block.
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_PATHWAYS = tuple(
    BLOCK_6_TRANSFORMATIVE_MODULES.keys()
)


NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_FAMILIES = {
    pathway_name:
        "masked_mean_squared_error"

    for pathway_name
    in NOTEBOOK_08_TRANSFORMATIVE_PATHWAYS
}


NOTEBOOK_08_TRANSFORMATIVE_TARGET_TYPES = {
    pathway_name:
        "continuous_signed_transformation"

    for pathway_name
    in NOTEBOOK_08_TRANSFORMATIVE_PATHWAYS
}


BLOCK_6_OBJECTIVE_FAMILIES_VALID = all(
    objective_family
    ==
    "masked_mean_squared_error"

    for objective_family
    in NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_FAMILIES.values()
)


if not BLOCK_6_OBJECTIVE_FAMILIES_VALID:

    raise RuntimeError(
        "Transformative objective-family assignment is invalid."
    )


# =============================================================================
# Mask-aware MSE objective
# =============================================================================

def notebook_08_masked_mse(
    prediction,
    target,
    mask,
):

    if prediction.shape != target.shape:

        raise ValueError(
            "Prediction and target shapes must match "
            "for masked transformative MSE."
        )


    if mask.shape != target.shape:

        raise ValueError(
            "Mask and target shapes must match "
            "for masked transformative MSE."
        )


    if not torch.is_floating_point(
        prediction
    ):

        raise TypeError(
            "Transformative prediction tensor must be floating point."
        )


    if not torch.is_floating_point(
        target
    ):

        raise TypeError(
            "Transformative target tensor must be floating point."
        )


    mask_float = mask.to(
        dtype=
            prediction.dtype,

        device=
            prediction.device,
    )


    valid_count = (
        mask_float.sum()
    )


    if valid_count.item() <= 0:

        raise ValueError(
            "Masked transformative MSE requires at least "
            "one available target element."
        )


    squared_error = (
        prediction
        -
        target
    ).pow(
        2
    )


    masked_squared_error = (
        squared_error
        *
        mask_float
    )


    return (
        masked_squared_error.sum()
        /
        valid_count
    )


# =============================================================================
# Masking policy
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY = {
    "missing_equals_zero":
        False,

    "missing_equals_negative":
        False,

    "unavailable_enters_loss":
        False,

    "mask_required":
        True,

    "minimum_available_elements":
        1,
}


BLOCK_6_MASK_POLICY_VALID = all(
    [
        (
            NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY[
                "missing_equals_zero"
            ]
            is False
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY[
                "missing_equals_negative"
            ]
            is False
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY[
                "unavailable_enters_loss"
            ]
            is False
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY[
                "mask_required"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY[
                "minimum_available_elements"
            ]
            ==
            1
        ),
    ]
)


if not BLOCK_6_MASK_POLICY_VALID:

    raise RuntimeError(
        "Transformative mask-aware supervision policy is invalid."
    )


# =============================================================================
# Pathway optimisation-loss weighting policy
# =============================================================================
#
# Objective weights are not activated in Block 6 because target availability
# has not yet been validated. Once eligible pathways are known, the default
# controlled policy is equal weighting across those eligible pathways only.
#
# These optimisation coefficients remain distinct from Transformative Weights
# TW_F, TW_P and TW_S.
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY = {
    "policy":
        "equal_across_eligible_pathways",

    "weights_activated":
        False,

    "eligible_pathways":
        [],

    "weights":
        {},
}


BLOCK_6_OBJECTIVE_WEIGHT_POLICY_VALID = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY[
            "policy"
        ]
        ==
        "equal_across_eligible_pathways",

        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY[
            "weights_activated"
        ]
        is False,

        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY[
            "eligible_pathways"
        ]
        ==
        [],

        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY[
            "weights"
        ]
        ==
        {},
    ]
)


if not BLOCK_6_OBJECTIVE_WEIGHT_POLICY_VALID:

    raise RuntimeError(
        "Transformative optimisation-loss weighting policy is invalid."
    )


# =============================================================================
# Optimisation policy
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY = {
    "optimizer":
        "AdamW",

    "learning_rate":
        0.001,

    "weight_decay":
        0.0001,

    "maximum_gradient_norm":
        1.0,

    "epochs":
        10,

    "shuffle_within_sequence":
        False,

    "sequence_order_preserved":
        True,

    "future_context_allowed":
        False,

    "representation_parameters_trainable":
        False,

    "transformative_parameters_trainable":
        True,
}


BLOCK_6_OPTIMIZER_NAME_VALID = (
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "optimizer"
    ]
    ==
    "AdamW"
)


BLOCK_6_LEARNING_RATE_VALID = (
    float(
        NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
            "learning_rate"
        ]
    )
    >
    0.0
)


BLOCK_6_WEIGHT_DECAY_VALID = (
    float(
        NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
            "weight_decay"
        ]
    )
    >=
    0.0
)


BLOCK_6_GRADIENT_NORM_VALID = (
    float(
        NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
            "maximum_gradient_norm"
        ]
    )
    >
    0.0
)


BLOCK_6_EPOCHS_VALID = (
    int(
        NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
            "epochs"
        ]
    )
    >
    0
)


BLOCK_6_SEQUENCE_POLICY_VALID = all(
    [
        (
            NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
                "shuffle_within_sequence"
            ]
            is False
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
                "sequence_order_preserved"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
                "future_context_allowed"
            ]
            is False
        ),
    ]
)


BLOCK_6_TRAINABILITY_POLICY_VALID = all(
    [
        (
            NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
                "representation_parameters_trainable"
            ]
            is False
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
                "transformative_parameters_trainable"
            ]
            is True
        ),
    ]
)


BLOCK_6_OPTIMIZER_POLICY_VALID = all(
    [
        BLOCK_6_OPTIMIZER_NAME_VALID,
        BLOCK_6_LEARNING_RATE_VALID,
        BLOCK_6_WEIGHT_DECAY_VALID,
        BLOCK_6_GRADIENT_NORM_VALID,
        BLOCK_6_EPOCHS_VALID,
        BLOCK_6_SEQUENCE_POLICY_VALID,
        BLOCK_6_TRAINABILITY_POLICY_VALID,
    ]
)


if not BLOCK_6_OPTIMIZER_POLICY_VALID:

    raise RuntimeError(
        "Notebook 08 transformative optimisation policy is invalid."
    )


# =============================================================================
# Eligible optimiser parameter contract
# =============================================================================
#
# Parameters are identified here, but no optimiser is instantiated.
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TRAINABLE_PARAMETERS = [
    parameter

    for module
    in BLOCK_6_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()

    if parameter.requires_grad
]


BLOCK_6_TRANSFORMATIVE_TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel()

    for parameter
    in NOTEBOOK_08_TRANSFORMATIVE_TRAINABLE_PARAMETERS
)


BLOCK_6_EXPECTED_TRANSFORMATIVE_PARAMETER_COUNT = sum(
    sum(
        parameter.numel()
        for parameter
        in module.parameters()
    )

    for module
    in BLOCK_6_TRANSFORMATIVE_MODULES.values()
)


BLOCK_6_TRANSFORMATIVE_PARAMETER_COUNT_VALID = (
    BLOCK_6_TRANSFORMATIVE_TRAINABLE_PARAMETER_COUNT
    ==
    BLOCK_6_EXPECTED_TRANSFORMATIVE_PARAMETER_COUNT
)


if not BLOCK_6_TRANSFORMATIVE_PARAMETER_COUNT_VALID:

    raise RuntimeError(
        "Transformative trainable parameter count does not match "
        "the validated Block 4 architecture."
    )


# =============================================================================
# Optimiser isolation validation
# =============================================================================

BLOCK_6_REPRESENTATION_PARAMETER_IDS = {
    id(
        parameter
    )

    for module
    in BLOCK_6_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
}


BLOCK_6_TRANSFORMATIVE_PARAMETER_IDS = {
    id(
        parameter
    )

    for parameter
    in NOTEBOOK_08_TRANSFORMATIVE_TRAINABLE_PARAMETERS
}


BLOCK_6_PARAMETER_GROUPS_DISJOINT = (
    BLOCK_6_REPRESENTATION_PARAMETER_IDS
    .isdisjoint(
        BLOCK_6_TRANSFORMATIVE_PARAMETER_IDS
    )
)


if not BLOCK_6_PARAMETER_GROUPS_DISJOINT:

    raise RuntimeError(
        "Representation and transformative parameter groups "
        "must be disjoint."
    )


# =============================================================================
# Transformative target-availability policy
# =============================================================================
#
# This block establishes the rules only.
#
# Actual transformation targets and masks are not materialised here.
# Their availability must be validated before training begins.
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY = {
    "target_space":
        "candidate_transformation",

    "signed_targets":
        True,

    "mask_aware":
        True,

    "zero_information_dimensions_deferred":
        True,

    "all_masked_dimensions_deferred":
        True,

    "future_information_prohibited":
        True,

    "training_loss_is_generalisation_evidence":
        False,

    "training_loss_is_empirical_validity_evidence":
        False,
}


BLOCK_6_TARGET_POLICY_VALID = all(
    [
        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "target_space"
            ]
            ==
            "candidate_transformation"
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "signed_targets"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "mask_aware"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "zero_information_dimensions_deferred"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "all_masked_dimensions_deferred"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "future_information_prohibited"
            ]
            is True
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "training_loss_is_generalisation_evidence"
            ]
            is False
        ),

        (
            NOTEBOOK_08_TRANSFORMATIVE_TARGET_POLICY[
                "training_loss_is_empirical_validity_evidence"
            ]
            is False
        ),
    ]
)


if not BLOCK_6_TARGET_POLICY_VALID:

    raise RuntimeError(
        "Notebook 08 transformative target policy is invalid."
    )


# =============================================================================
# Target-validation and optimisation-readiness boundary
# =============================================================================
#
# Block 6 defines policy only. No factual, psychological or social
# transformation target tensor or mask has yet been validated here.
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED = False

NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS = tuple()

NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVES_ACTIVATED = False

NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY = False

NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY = False


BLOCK_6_TARGET_READINESS_BOUNDARY_VALID = all(
    [
        not NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED,
        len(
            NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS
        )
        ==
        0,
        not NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVES_ACTIVATED,
        not NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY,
        not NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY,
    ]
)


if not BLOCK_6_TARGET_READINESS_BOUNDARY_VALID:

    raise RuntimeError(
        "Notebook 08 Block 6 target-readiness boundary is invalid."
    )


# =============================================================================
# Notebook 09 architectural exclusion
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_WEIGHTS_RESERVED_FOR_NOTEBOOK = 9

NOTEBOOK_08_RECURRENT_MECHANISM_RESERVED_FOR_NOTEBOOK = 9


BLOCK_6_NOTEBOOK_09_BOUNDARY_VALID = all(
    [
        (
            NOTEBOOK_08_TRANSFORMATIVE_WEIGHTS_RESERVED_FOR_NOTEBOOK
            ==
            9
        ),

        (
            NOTEBOOK_08_RECURRENT_MECHANISM_RESERVED_FOR_NOTEBOOK
            ==
            9
        ),

        not NOTEBOOK_08_BLOCK_6_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_6_RECURRENT_STATE_INSTANTIATED,
    ]
)


if not BLOCK_6_NOTEBOOK_09_BOUNDARY_VALID:

    raise RuntimeError(
        "Notebook 08 incorrectly crossed the Notebook 09 "
        "Transformative Weight or recurrent-mechanism boundary."
    )


# =============================================================================
# Post-policy parameter-state validation
# =============================================================================

BLOCK_6_REPRESENTATION_STATE_AFTER = {
    module_name:
        block_6_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_6_REPRESENTATION_MODULES.items()
}


BLOCK_6_TRANSFORMATIVE_STATE_AFTER = {
    pathway_name:
        block_6_snapshot_module(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_6_TRANSFORMATIVE_MODULES.items()
}


BLOCK_6_REPRESENTATION_PARAMETERS_UNCHANGED = {
    module_name:
        block_6_states_equal(
            BLOCK_6_REPRESENTATION_STATE_BEFORE[
                module_name
            ],

            BLOCK_6_REPRESENTATION_STATE_AFTER[
                module_name
            ],
        )

    for module_name
    in BLOCK_6_REPRESENTATION_MODULES
}


BLOCK_6_TRANSFORMATIVE_PARAMETERS_UNCHANGED = {
    pathway_name:
        block_6_states_equal(
            BLOCK_6_TRANSFORMATIVE_STATE_BEFORE[
                pathway_name
            ],

            BLOCK_6_TRANSFORMATIVE_STATE_AFTER[
                pathway_name
            ],
        )

    for pathway_name
    in BLOCK_6_TRANSFORMATIVE_MODULES
}


BLOCK_6_ALL_REPRESENTATION_PARAMETERS_UNCHANGED = all(
    BLOCK_6_REPRESENTATION_PARAMETERS_UNCHANGED.values()
)


BLOCK_6_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED = all(
    BLOCK_6_TRANSFORMATIVE_PARAMETERS_UNCHANGED.values()
)


if not all(
    [
        BLOCK_6_ALL_REPRESENTATION_PARAMETERS_UNCHANGED,
        BLOCK_6_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Model parameters changed during objective-policy construction."
    )


# =============================================================================
# Transformative optimisation-policy readiness
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED = all(
    [
        BLOCK_6_OBJECTIVE_FAMILIES_VALID,
        BLOCK_6_MASK_POLICY_VALID,
        BLOCK_6_OBJECTIVE_WEIGHT_POLICY_VALID,
        BLOCK_6_OPTIMIZER_POLICY_VALID,
        BLOCK_6_TRANSFORMATIVE_PARAMETER_COUNT_VALID,
        BLOCK_6_PARAMETER_GROUPS_DISJOINT,
        BLOCK_6_TARGET_POLICY_VALID,
        BLOCK_6_TARGET_READINESS_BOUNDARY_VALID,
        BLOCK_6_NOTEBOOK_09_BOUNDARY_VALID,
        BLOCK_6_ALL_REPRESENTATION_MODULES_FROZEN,
        BLOCK_6_ALL_TRANSFORMATIVE_MODULES_TRAINABLE,
        BLOCK_6_ALL_REPRESENTATION_PARAMETERS_UNCHANGED,
        BLOCK_6_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED,
    ]
)


# =============================================================================
# Final Block 6 validation
# =============================================================================

NOTEBOOK_08_BLOCK_6_VALID = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED,

        not NOTEBOOK_08_BLOCK_6_TRAINING_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_LOSS_CALCULATED,
        not NOTEBOOK_08_BLOCK_6_OPTIMIZER_CREATED,
        not NOTEBOOK_08_BLOCK_6_ZERO_GRAD_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_BACKWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_GRADIENT_CLIPPING_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_OPTIMIZER_STEP_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_PARAMETER_UPDATE_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_6_TRANSFORMATION_WEIGHTS_EXECUTED,
        not NOTEBOOK_08_BLOCK_6_RECURRENT_STATE_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_6_RECURRENT_UPDATE_EXECUTED,
    ]
)


if not NOTEBOOK_08_BLOCK_6_VALID:

    raise RuntimeError(
        "Notebook 08 Block 6 transformative objective "
        "and training-policy validation failed."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_6_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_08_BLOCK_6_SUMMARY = {
    "block":
        NOTEBOOK_08_BLOCK_6,

    "block_name":
        NOTEBOOK_08_BLOCK_6_NAME,

    "version":
        NOTEBOOK_08_BLOCK_6_VERSION,

    "objective_families":
        deepcopy(
            NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_FAMILIES
        ),

    "objective_weight_policy":
        deepcopy(
            NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY
        ),

    "optimizer_policy":
        deepcopy(
            NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY
        ),

    "mask_policy":
        deepcopy(
            NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY
        ),

    "trainable_parameters":
        BLOCK_6_TRANSFORMATIVE_TRAINABLE_PARAMETER_COUNT,

    "target_contract_validated":
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED,

    "eligible_pathways":
        list(
            NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS
        ),

    "objectives_activated":
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVES_ACTIVATED,

    "optimisation_ready":
        NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY,

    "training_ready":
        NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY,

    "representation_frozen":
        BLOCK_6_ALL_REPRESENTATION_MODULES_FROZEN,

    "transformative_trainable":
        BLOCK_6_ALL_TRANSFORMATIVE_MODULES_TRAINABLE,

    "transformative_weights_reserved_for_notebook":
        NOTEBOOK_08_TRANSFORMATIVE_WEIGHTS_RESERVED_FOR_NOTEBOOK,

    "recurrent_mechanism_reserved_for_notebook":
        NOTEBOOK_08_RECURRENT_MECHANISM_RESERVED_FOR_NOTEBOOK,

    "block_valid":
        NOTEBOOK_08_BLOCK_6_VALID,

    "block_complete":
        NOTEBOOK_08_BLOCK_6_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 6: "
    "Transformative Objective and Training Policy"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_6_VERSION}"
)

print("-" * 72)

print(
    "Transformative objective families"
)

for (
    pathway_name,
    objective_family,
) in NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_FAMILIES.items():

    print(
        f"{pathway_name:<28}: "
        f"{objective_family}"
    )

print("-" * 72)

print(
    "Optimisation-loss weighting policy"
)

print(
    f"Policy                      : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY['policy']}"
)

print(
    f"Weights activated           : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY['weights_activated']}"
)

print(
    f"Eligible pathways           : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY['eligible_pathways']}"
)

print(
    f"Active weights              : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY['weights']}"
)

print("-" * 72)

print("-" * 72)

print(
    "Optimisation policy"
)

print(
    f"Optimizer                   : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['optimizer']}"
)

print(
    f"Learning rate               : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['learning_rate']}"
)

print(
    f"Weight decay                : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['weight_decay']}"
)

print(
    f"Maximum gradient norm       : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['maximum_gradient_norm']}"
)

print(
    f"Epochs                      : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['epochs']}"
)

print(
    f"Sequence order preserved    : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['sequence_order_preserved']}"
)

print(
    f"Future context allowed      : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY['future_context_allowed']}"
)

print("-" * 72)

print(
    f"Mask-aware supervision      : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY['mask_required']}"
)

print(
    f"Missing equals zero         : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY['missing_equals_zero']}"
)

print(
    f"Unavailable enters loss     : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_MASK_POLICY['unavailable_enters_loss']}"
)

print("-" * 72)

print(
    f"Representation model frozen : "
    f"{BLOCK_6_ALL_REPRESENTATION_MODULES_FROZEN}"
)

print(
    f"Transform modules trainable : "
    f"{BLOCK_6_ALL_TRANSFORMATIVE_MODULES_TRAINABLE}"
)

print(
    f"Eligible trainable params   : "
    f"{BLOCK_6_TRANSFORMATIVE_TRAINABLE_PARAMETER_COUNT:,}"
)

print(
    f"Parameter groups disjoint   : "
    f"{BLOCK_6_PARAMETER_GROUPS_DISJOINT}"
)

print("-" * 72)

print(
    f"Representation unchanged    : "
    f"{BLOCK_6_ALL_REPRESENTATION_PARAMETERS_UNCHANGED}"
)

print(
    f"Transform params unchanged  : "
    f"{BLOCK_6_ALL_TRANSFORMATIVE_PARAMETERS_UNCHANGED}"
)

print("-" * 72)


print(
    f"Target contract validated   : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED}"
)

print(
    f"Eligible pathways           : "
    f"{list(NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS)}"
)

print(
    f"Objectives activated        : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVES_ACTIVATED}"
)

print(
    f"Optimisation ready          : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY}"
)

print(
    f"Training ready              : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY}"
)

print("-" * 72)

print(
    f"Training forward executed   : "
    f"{NOTEBOOK_08_BLOCK_6_TRAINING_FORWARD_PASS_EXECUTED}"
)

print(
    f"Loss calculated             : "
    f"{NOTEBOOK_08_BLOCK_6_LOSS_CALCULATED}"
)

print(
    f"Optimizer created           : "
    f"{NOTEBOOK_08_BLOCK_6_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed      : "
    f"{NOTEBOOK_08_BLOCK_6_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed   : "
    f"{NOTEBOOK_08_BLOCK_6_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Transform weights created   : "
    f"{NOTEBOOK_08_BLOCK_6_TRANSFORMATION_WEIGHTS_INSTANTIATED}"
)

print(
    f"Recurrent state created     : "
    f"{NOTEBOOK_08_BLOCK_6_RECURRENT_STATE_INSTANTIATED}"
)

print("-" * 72)

print(
    f"Objective policy defined    : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_6_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_6_COMPLETE}"
)

print("=" * 72)

print(
    "The transformative objective and optimisation policy was defined "
    "successfully without executing training."
)

print(
    "Candidate transformations are defined as signed continuous "
    "quantities with mask-aware mean squared error as the admissible "
    "objective family."
)

print(
    f"The inherited representation model remains frozen; "
    f"{BLOCK_6_TRANSFORMATIVE_TRAINABLE_PARAMETER_COUNT:,} transformative "
    "parameters are structurally eligible for later optimisation."
)

print(
    "No pathway-specific optimisation coefficient has been activated "
    "because real transformative target and mask availability has not yet "
    "been validated."
)

print(
    "Any later optimisation-loss coefficients remain distinct from "
    "Transformative Weights."
)

print(
    "No loss, optimiser, backward pass, gradient clipping, "
    "optimiser step or parameter update was executed."
)


print(
    "Target tensors and masks remain to be validated before objective "
    "activation, optimiser construction or controlled training."
)

print(
    "Transformative Weights and the recurrent mechanism remain "
    "reserved for Notebook 09."
)

print("=" * 72)

Media AI — Notebook 08, Block 6: Transformative Objective and Training Policy
Block version               : 1.1
------------------------------------------------------------------------
Transformative objective families
factual                     : masked_mean_squared_error
psychological               : masked_mean_squared_error
social                      : masked_mean_squared_error
------------------------------------------------------------------------
Optimisation-loss weighting policy
Policy                      : equal_across_eligible_pathways
Weights activated           : False
Eligible pathways           : []
Active weights              : {}
------------------------------------------------------------------------
------------------------------------------------------------------------
Optimisation policy
Optimizer                   : AdamW
Learning rate               : 0.001
Weight decay                : 0.0001
Maximum gradient norm       : 1.0
Epochs                      : 10


## Block 7 — Transformative Target Construction and Mask Contract

This block defines and validates the supervised targets required to train the factual, psychological, and social transformative mechanisms.

Blocks 4 and 5 established the pathway-specific candidate-transformation functions

$$
\boldsymbol{\tau}^{(k)}_t
=
\mathcal{T}^{(k)}
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1}
\right),
\qquad
k \in \{F,P,S\},
$$

and Block 6 established the optimisation policy under which those mechanisms may subsequently be trained.

The remaining requirement is an explicit target describing **what transformation should be learned**.

Block 7 therefore constructs the pathway-specific supervised transformation targets

$$
\mathbf{y}^{(F)}_{T,t},
\qquad
\mathbf{y}^{(P)}_{T,t},
\qquad
\mathbf{y}^{(S)}_{T,t},
$$

together with the masks that determine which target dimensions contain valid supervision.

No optimisation is performed in this block.

### Transformation as supervised change

The transformative mechanism is intended to model the change associated with the arrival of the current observation relative to the condition that preceded it.

For pathway \(k\), let

$$
\mathbf{y}^{(k)}_t
$$

denote the supervised pathway state associated with the current observation, and let

$$
\mathbf{h}^{(k)}_{t-1}
$$

denote the causally available preceding-state condition.

The transformation target is defined as the signed difference

$$
\boxed{
\mathbf{y}^{(k)}_{T,t}
=
\mathbf{y}^{(k)}_t
-
\mathbf{h}^{(k)}_{t-1}
}
$$

for every dimension for which the required supervision is available.

This definition gives the target a direct interpretation.

If

$$
y^{(k)}_{T,t,i} > 0,
$$

the current observation implies an increase in dimension \(i\) relative to the preceding condition.

If

$$
y^{(k)}_{T,t,i} < 0,
$$

the current observation implies a decrease.

If

$$
y^{(k)}_{T,t,i} = 0,
$$

the supervised current condition and preceding condition are equal for that dimension.

The candidate transformation is therefore explicitly signed and directional.

### Pathway-specific target spaces

Transformation targets remain within their corresponding semantic spaces.

For the factual pathway,

$$
\mathbf{y}^{(F)}_{T,t}
\in
\mathbb{R}^{10}.
$$

For the psychological pathway,

$$
\mathbf{y}^{(P)}_{T,t}
\in
\mathbb{R}^{34}.
$$

For the social pathway,

$$
\mathbf{y}^{(S)}_{T,t}
\in
\mathbb{R}^{7}.
$$

Consequently,

$$
\operatorname{dim}
\left(
\mathbf{y}^{(k)}_{T,t}
\right)
=
\operatorname{dim}
\left(
\boldsymbol{\tau}^{(k)}_t
\right).
$$

No projection between factual, psychological, and social spaces is introduced during target construction.

### Current supervision is distinct from current representation

The current supervised state

$$
\mathbf{y}^{(k)}_t
$$

must not be confused with the current learned representation

$$
\mathbf{r}^{(k)}_t.
$$

The representation is an input to the transformative mechanism:

$$
\mathbf{r}^{(k)}_t
\longrightarrow
\mathcal{T}^{(k)}.
$$

The supervised state contributes to the training target:

$$
\mathbf{y}^{(k)}_t
\longrightarrow
\mathbf{y}^{(k)}_{T,t}.
$$

The intended learning relation is therefore

$$
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(k)}_t
\approx
\mathbf{y}^{(k)}_{T,t}.
$$

This separation is essential.

Defining the target directly from the learned representation would risk training the transformative mechanism to reproduce properties of the representation model rather than learning a transformation grounded in the available supervision.

### Preceding-state training condition

Notebook 08 has not yet introduced the recurrent state-update mechanism.

Accordingly, the symbol

$$
\mathbf{h}^{(k)}_{t-1}
$$

in the target-construction contract denotes a **causally available preceding-state training condition**.

It must not yet be interpreted as a recurrent state produced by

$$
TW^{(k)}
$$

or by the recurrent update mechanism reserved for Notebook 09.

Where a preceding condition is constructed from supervision available before position \(t\), that construction must use only information satisfying

$$
t' < t.
$$

The training condition may therefore summarise valid prior information, but it must not depend on the current target itself or on any future observation.

This allows Notebook 08 to learn the transformation function without prematurely defining how recurrent state is propagated.

### Causal target construction

Target construction must preserve chronological causality.

For observation \(t\), the preceding condition may depend only on information available before that observation:

$$
\mathbf{h}^{(k)}_{t-1}
=
f
\left(
\mathbf{y}^{(k)}_1,
\ldots,
\mathbf{y}^{(k)}_{t-1}
\right),
$$

where the precise preceding-condition construction is explicitly defined and reproducible.

It must not depend on

$$
\mathbf{y}^{(k)}_t,
$$

except through the subsequent target subtraction, and it must never depend on

$$
\mathbf{y}^{(k)}_{t+1},
\mathbf{y}^{(k)}_{t+2},
\ldots.
$$

Thus,

$$
\boxed{
\text{future information permitted}
=
\mathrm{False}
}
$$

throughout target construction.

### Initial-position policy

The first valid observation in a sequence requires special treatment because no earlier recurrent condition exists.

Notebook 08 must not silently invent historical information for this position.

The initial preceding condition must therefore follow an explicit deterministic policy.

Where no valid earlier condition exists, the neutral initial condition is

$$
\mathbf{h}^{(k)}_0
=
\mathbf{0}^{d_k},
$$

provided that this convention is recorded explicitly as an architectural initialisation condition rather than interpreted as observed supervision.

The corresponding first-position target is then

$$
\mathbf{y}^{(k)}_{T,1}
=
\mathbf{y}^{(k)}_1
-
\mathbf{0}
=
\mathbf{y}^{(k)}_1.
$$

This zero initialisation provides a reproducible origin for transformation learning without importing future or fabricated historical information.

It does not imply that the real-world factual, psychological, or social state before the sequence was actually zero.

### Mask propagation

A transformation target is valid only where the information required to construct it is valid.

Let

$$
\mathbf{m}^{(k)}_t
$$

denote the availability mask for the current supervised state, and let

$$
\mathbf{m}^{(k)}_{h,t-1}
$$

denote the validity mask for the preceding-state condition.

For non-initial positions, the transformation mask is defined element-wise as

$$
\boxed{
\mathbf{m}^{(k)}_{T,t}
=
\mathbf{m}^{(k)}_t
\odot
\mathbf{m}^{(k)}_{h,t-1}
}
$$

where

$$
\odot
$$

denotes element-wise multiplication.

Therefore, a transformation dimension contributes to training only when both the current supervised value and the preceding condition required for the difference are valid.

For dimensions excluded by the mask,

$$
m^{(k)}_{T,t,i}=0,
$$

the corresponding numerical target value must not contribute to the optimisation objective.

### Missing values are not zero transformations

A missing target must remain semantically distinct from a genuine zero transformation.

If

$$
m^{(k)}_{T,t,i}=0,
$$

the model has no valid supervised transformation target for that element.

By contrast, if

$$
m^{(k)}_{T,t,i}=1
$$

and

$$
y^{(k)}_{T,t,i}=0,
$$

the supervision explicitly indicates no change between the preceding and current conditions.

Accordingly,

$$
\boxed{
\text{missing transformation}
\neq
\text{zero transformation}
}
$$

and missing values must never be converted into valid zero-change observations through target construction.

### Signed target preservation

Transformation targets are not probabilities.

They may legitimately contain positive, negative, or zero values:

$$
\mathbf{y}^{(k)}_{T,t}
\in
\mathbb{R}^{d_k}.
$$

No sigmoid, softmax, absolute-value operation, clipping to \([0,1]\), or sign removal is applied merely to make the targets resemble the original annotation space.

The signed difference carries the transformation information itself.

For example,

$$
y^{(k)}_{T,t,i}=-0.4
$$

and

$$
y^{(k)}_{T,t,i}=+0.4
$$

represent transformations of equal magnitude but opposite direction and must remain distinguishable.

### Target availability

The existence of an architectural output dimension does not guarantee that a valid transformation target exists for every observation and dimension.

For each pathway, Block 7 therefore records:

- total target elements;
- available target elements;
- masked target elements;
- availability proportion;
- number of positive transformations;
- number of negative transformations;
- number of genuine zero transformations.

These diagnostics distinguish structural dimensionality from empirical supervision coverage.

A pathway may proceed to optimisation only if it contains sufficient valid target information to define a meaningful loss.

### Target variation

A valid transformation target should also contain meaningful variation.

For each supervised dimension, the block must determine whether the available target values are:

- variable;
- constant;
- entirely zero;
- or entirely unavailable.

A dimension that is entirely unavailable cannot contribute to training.

A dimension that is valid but constant contains structurally weaker information and must be identified explicitly.

A dimension for which all valid transformation targets satisfy

$$
y^{(k)}_{T,t,i}=0
$$

must likewise be reported as a zero-information transformation dimension rather than silently treated as informative variation.

Such dimensions may be deferred from optimisation according to the policy established in Block 6.

### Numerical validity

Every available transformation target must be finite.

For all elements satisfying

$$
m^{(k)}_{T,t,i}=1,
$$

the block requires

$$
y^{(k)}_{T,t,i}
\in
\mathbb{R}
$$

with

$$
\operatorname{isfinite}
\left(
y^{(k)}_{T,t,i}
\right)
=
\mathrm{True}.
$$

`NaN`, positive infinity, and negative infinity are not valid supervised transformations.

Invalid numerical values must be excluded explicitly rather than allowed to propagate into the optimisation objective.

### Separation from Transformative Weights

The target

$$
\mathbf{y}^{(k)}_{T,t}
$$

describes the transformation that the candidate mechanism should learn to propose.

It does not determine how strongly that transformation will later affect recurrent state.

Therefore, Block 7 does not define

$$
TW^{(k)}_t
$$

and does not construct a target for a Transformative Weight.

The conceptual separation remains

$$
\boxed{
\text{Notebook 08: learn what change is proposed}
}
$$

and

$$
\boxed{
\text{Notebook 09: determine how that change enters state}
}.
$$

### Separation from recurrent-state updates

Target construction also does not execute

$$
\mathbf{h}^{(k)}_t
=
\mathbf{h}^{(k)}_{t-1}
+
TW^{(k)}_t
\boldsymbol{\tau}^{(k)}_t
$$

or any alternative recurrent update rule.

The preceding-state training condition is used only as an input to the transformative mechanism and as a reference point for constructing the supervised change.

No candidate transformation produced in Notebook 08 becomes a new recurrent state.

This distinction prevents target generation from becoming an implicit recurrent simulation.

### Parameter immutability

Target construction is a data and supervision operation.

It must not modify either the inherited representation model or the transformative mechanisms.

Therefore,

$$
\theta_{\mathrm{repr,before}}
=
\theta_{\mathrm{repr,after}}
$$

and

$$
\theta_{T,\mathrm{before}}
=
\theta_{T,\mathrm{after}}.
$$

No optimiser, backward pass, gradient calculation, or parameter update is required to construct transformation targets.

### Block 7 validation contract

Block 7 is considered valid only if the target-construction process establishes that:

- factual targets have dimension \(10\);
- psychological targets have dimension \(34\);
- social targets have dimension \(7\);
- transformation targets are signed continuous quantities;
- the preceding-state condition is constructed causally;
- the initial-position policy is explicit and deterministic;
- future information is excluded;
- masks distinguish missing targets from genuine zero transformations;
- available target values are finite;
- target availability is quantified;
- target variation is assessed dimension by dimension;
- zero-information and unavailable dimensions are identified explicitly;
- current supervision remains distinct from current learned representations;
- no Transformative Weight is instantiated;
- no recurrent update is performed;
- inherited representation parameters remain frozen and unchanged;
- transformative parameters remain unchanged;
- no optimiser or training operation is executed.

At completion, the supervised learning relation required for transformative training will be fully specified:

$$
\boxed{
\left(
\mathbf{r}^{(k)}_t,
\mathbf{h}^{(k)}_{t-1}
\right)
\longrightarrow
\boldsymbol{\tau}^{(k)}_t
\approx
\mathbf{y}^{(k)}_{T,t}
}
$$

with

$$
\boxed{
\mathbf{y}^{(k)}_{T,t}
=
\mathbf{y}^{(k)}_t
-
\mathbf{h}^{(k)}_{t-1}
}
$$

under explicit causal and mask-aware supervision.

Block 7 therefore establishes the final supervised-data contract required before transformative optimisation can begin, while preserving the Notebook 08 boundary against Transformative Weight construction and recurrent-state propagation.

In [11]:
# =============================================================================
# Media AI — Notebook 08, Block 7
# Transformative Target Construction and Mask Contract
# =============================================================================

from copy import deepcopy
from datetime import datetime, timezone

import io
import json

import numpy as np
import torch

from googleapiclient.http import MediaIoBaseDownload


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_7 = 7

NOTEBOOK_08_BLOCK_7_NAME = (
    "Transformative Target Construction and Mask Contract"
)

NOTEBOOK_08_BLOCK_7_VERSION = "1.7"

BLOCK_7_EXECUTED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# =============================================================================
# Dependency checks
# =============================================================================

required_block_7_objects = [
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Block 2
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_DRIVE_SERVICE",
    "NOTEBOOK_08_BLOCK_2_COMPLETE",
    "NOTEBOOK_08_BLOCK_2_VALID",
    "NOTEBOOK_08_REPRESENTATION_HANDOVER_READY",

    # -------------------------------------------------------------------------
    # Block 3
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_3_COMPLETE",
    "NOTEBOOK_08_BLOCK_3_VALID",
    "NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED",

    "NOTEBOOK_08_RESTORED_BACKBONE",
    "NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT",
    "NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD",
    "NOTEBOOK_08_RESTORED_FACTUAL_HEAD",
    "NOTEBOOK_08_RESTORED_SOCIAL_HEAD",

    # -------------------------------------------------------------------------
    # Block 4
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_4_COMPLETE",
    "NOTEBOOK_08_BLOCK_4_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY",

    "NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM",

    # -------------------------------------------------------------------------
    # Block 5
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_5_COMPLETE",
    "NOTEBOOK_08_BLOCK_5_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED",

    # -------------------------------------------------------------------------
    # Block 6
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_6_COMPLETE",
    "NOTEBOOK_08_BLOCK_6_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED",

    # -------------------------------------------------------------------------
    # Dimensions
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_FACTUAL_TRANSFORM_DIM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM",
    "NOTEBOOK_08_SOCIAL_TRANSFORM_DIM",
]


missing_block_7_objects = [
    object_name
    for object_name
    in required_block_7_objects
    if object_name not in globals()
]


if missing_block_7_objects:

    raise NameError(
        "Notebook 08 Block 7 prerequisites are not initialised. "
        f"Missing: {missing_block_7_objects}"
    )


if not all(
    [
        NOTEBOOK_08_BLOCK_2_COMPLETE,
        NOTEBOOK_08_BLOCK_2_VALID,
        NOTEBOOK_08_REPRESENTATION_HANDOVER_READY,

        NOTEBOOK_08_BLOCK_3_COMPLETE,
        NOTEBOOK_08_BLOCK_3_VALID,
        NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED,

        NOTEBOOK_08_BLOCK_4_COMPLETE,
        NOTEBOOK_08_BLOCK_4_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_ARCHITECTURE_READY,

        NOTEBOOK_08_BLOCK_5_COMPLETE,
        NOTEBOOK_08_BLOCK_5_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_FORWARD_PATH_VALIDATED,

        NOTEBOOK_08_BLOCK_6_COMPLETE,
        NOTEBOOK_08_BLOCK_6_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED,
    ]
):

    raise RuntimeError(
        "Notebook 08 Blocks 2–6 must be complete and valid "
        "before Block 7 target construction."
    )


# =============================================================================
# Pathway dimensional contract
# =============================================================================

BLOCK_7_PATHWAY_DIMS = {
    "factual":
        int(
            NOTEBOOK_08_FACTUAL_TRANSFORM_DIM
        ),

    "psychological":
        int(
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM
        ),

    "social":
        int(
            NOTEBOOK_08_SOCIAL_TRANSFORM_DIM
        ),
}


BLOCK_7_PATHWAY_DIMENSIONS_VALID = all(
    [
        set(
            BLOCK_7_PATHWAY_DIMS.keys()
        )
        ==
        {
            "factual",
            "psychological",
            "social",
        },

        all(
            pathway_dim > 0

            for pathway_dim
            in BLOCK_7_PATHWAY_DIMS.values()
        ),
    ]
)


if not BLOCK_7_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 08 Block 7 pathway dimensions do not match "
        "the validated representation contract."
    )


# =============================================================================
# Methodological target contract
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TARGET_SOURCE_TYPE = (
    "derived_from_validated_representation_supervision"
)

NOTEBOOK_08_TRANSFORMATIVE_DIRECT_SUPERVISION_CLAIMED = False

NOTEBOOK_08_TRANSFORMATIVE_TARGETS_SIGNED = True

NOTEBOOK_08_TRANSFORMATIVE_TARGETS_MASK_AWARE = True

NOTEBOOK_08_TRANSFORMATIVE_FUTURE_CONTEXT_ALLOWED = False

NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITION_POLICY = (
    "previous_supervised_state_with_zero_initial_condition"
)

NOTEBOOK_08_TRANSFORMATIVE_INITIAL_STATE_POLICY = (
    "deterministic_zero_architectural_initial_condition"
)


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_08_BLOCK_7_MODEL_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_7_TRANSFORM_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_7_LOSS_CALCULATED = False

NOTEBOOK_08_BLOCK_7_OPTIMIZER_CREATED = False

NOTEBOOK_08_BLOCK_7_ZERO_GRAD_EXECUTED = False

NOTEBOOK_08_BLOCK_7_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_7_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_08_BLOCK_7_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_08_BLOCK_7_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_08_BLOCK_7_TRANSFORMATION_WEIGHTS_INSTANTIATED = False

NOTEBOOK_08_BLOCK_7_RECURRENT_STATE_INSTANTIATED = False

NOTEBOOK_08_BLOCK_7_RECURRENT_UPDATE_EXECUTED = False


# =============================================================================
# Google Drive helpers
# =============================================================================

def block_7_download_drive_bytes(
    drive_service,
    file_id,
):

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    complete = False


    while not complete:

        _, complete = (
            downloader.next_chunk()
        )


    buffer.seek(
        0
    )


    return buffer.read()


def block_7_find_drive_file_by_name(
    drive_service,
    filename,
):

    escaped_filename = filename.replace(
        "'",
        "\\'",
    )


    response = (
        drive_service.files()
        .list(
            q=(
                f"name = '{escaped_filename}' "
                "and trashed = false"
            ),
            spaces=
                "drive",
            fields=
                "files(id,name,mimeType,modifiedTime,parents)",
            orderBy=
                "modifiedTime desc",
            pageSize=
                20,
        )
        .execute()
    )


    files = response.get(
        "files",
        [],
    )


    if not files:

        return None


    return files[
        0
    ]


def block_7_load_json_from_drive(
    drive_service,
    file_id,
):

    raw_bytes = (
        block_7_download_drive_bytes(
            drive_service,
            file_id,
        )
    )


    if not raw_bytes:

        raise RuntimeError(
            "Required persisted JSON artefact is empty."
        )


    try:

        return json.loads(
            raw_bytes.decode(
                "utf-8"
            )
        )


    except (
        UnicodeDecodeError,
        json.JSONDecodeError,
    ) as error:

        raise RuntimeError(
            "Required persisted artefact is not valid UTF-8 JSON."
        ) from error


# =============================================================================
# Resolve Notebook 06 → Notebook 07 handover
# =============================================================================

BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER_FILENAME = (
    "notebook_06_to_07_handover.json"
)


BLOCK_7_NOTEBOOK_06_TO_07_DRIVE_FILE = (
    block_7_find_drive_file_by_name(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER_FILENAME,
    )
)


if BLOCK_7_NOTEBOOK_06_TO_07_DRIVE_FILE is None:

    raise RuntimeError(
        "Persisted Notebook 06 → Notebook 07 handover "
        "could not be located in Drive."
    )


BLOCK_7_NOTEBOOK_06_TO_07_DRIVE_FILE_ID = (
    BLOCK_7_NOTEBOOK_06_TO_07_DRIVE_FILE[
        "id"
    ]
)


BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER = (
    block_7_load_json_from_drive(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_7_NOTEBOOK_06_TO_07_DRIVE_FILE_ID,
    )
)


BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER_VALID = all(
    [
        isinstance(
            BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER,
            dict,
        ),

        (
            BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER.get(
                "source_notebook"
            )
            ==
            "06_factual_path"
        ),

        (
            BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER.get(
                "target_notebook"
            )
            ==
            "07_social_path"
        ),
    ]
)


if not BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER_VALID:

    raise RuntimeError(
        "Persisted Notebook 06 → Notebook 07 handover "
        "failed schema validation."
    )


# =============================================================================
# Canonical identity
# =============================================================================

BLOCK_7_CANONICAL_IDENTITY = (
    BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER.get(
        "canonical_identity",
        {},
    )
)


BLOCK_7_SENTENCE_IDS = list(
    BLOCK_7_CANONICAL_IDENTITY.get(
        "sentence_ids",
        [],
    )
)


BLOCK_7_SENTENCE_COUNT = int(
    BLOCK_7_CANONICAL_IDENTITY.get(
        "sentences",
        BLOCK_7_CANONICAL_IDENTITY.get(
            "sentence_count",
            0,
        ),
    )
)


BLOCK_7_ARTICLE_COUNT = int(
    BLOCK_7_CANONICAL_IDENTITY.get(
        "articles",
        BLOCK_7_CANONICAL_IDENTITY.get(
            "article_count",
            0,
        ),
    )
)


BLOCK_7_IDENTITY_VALID = all(
    [
        BLOCK_7_ARTICLE_COUNT > 0,
        BLOCK_7_SENTENCE_COUNT > 0,

        len(
            BLOCK_7_SENTENCE_IDS
        )
        ==
        BLOCK_7_SENTENCE_COUNT,

        len(
            set(
                BLOCK_7_SENTENCE_IDS
            )
        )
        ==
        BLOCK_7_SENTENCE_COUNT,

        all(
            bool(
                sentence_id
            )
            for sentence_id
            in BLOCK_7_SENTENCE_IDS
        ),
    ]
)


if not BLOCK_7_IDENTITY_VALID:

    raise RuntimeError(
        "Notebook 08 Block 7 canonical sentence identity is invalid."
    )


# =============================================================================
# Canonical article-boundary identity
# =============================================================================

def block_7_article_id_from_sentence_id(
    sentence_id,
):

    if not isinstance(
        sentence_id,
        str,
    ) or not sentence_id:

        raise ValueError(
            "Canonical sentence IDs must be non-empty strings."
        )


    if "::" in sentence_id:

        return sentence_id.split(
            "::",
            1,
        )[0]


    return sentence_id


BLOCK_7_ARTICLE_IDS_BY_SENTENCE = tuple(
    block_7_article_id_from_sentence_id(
        sentence_id
    )

    for sentence_id
    in BLOCK_7_SENTENCE_IDS
)


BLOCK_7_CANONICAL_ARTICLE_IDS = tuple(
    dict.fromkeys(
        BLOCK_7_ARTICLE_IDS_BY_SENTENCE
    )
)


BLOCK_7_ARTICLE_BOUNDARY_IDENTITY_VALID = all(
    [
        len(
            BLOCK_7_ARTICLE_IDS_BY_SENTENCE
        )
        ==
        BLOCK_7_SENTENCE_COUNT,

        len(
            BLOCK_7_CANONICAL_ARTICLE_IDS
        )
        ==
        BLOCK_7_ARTICLE_COUNT,

        all(
            bool(
                article_id
            )
            for article_id
            in BLOCK_7_ARTICLE_IDS_BY_SENTENCE
        ),
    ]
)


if not BLOCK_7_ARTICLE_BOUNDARY_IDENTITY_VALID:

    raise RuntimeError(
        "Notebook 08 Block 7 canonical article-boundary identity "
        "is invalid."
    )


# =============================================================================
# Canonical record-order helper
# =============================================================================

def block_7_reorder_records_to_canonical(
    records,
    source_name,
):

    if not isinstance(
        records,
        list,
    ):

        raise TypeError(
            f"{source_name} records must be a list."
        )


    record_lookup = {}


    persisted_sentence_ids = []


    for (
        record_index,
        record,
    ) in enumerate(
        records
    ):

        if not isinstance(
            record,
            dict,
        ):

            raise TypeError(
                f"{source_name} record {record_index} "
                "is not dictionary-like."
            )


        identity = record.get(
            "identity",
            {},
        )


        sentence_id = (
            identity.get(
                "sentence_id"
            )
            if isinstance(
                identity,
                dict,
            )
            else None
        )


        if not sentence_id:

            raise RuntimeError(
                f"{source_name} record {record_index} "
                "has no sentence_id."
            )


        if sentence_id in record_lookup:

            raise RuntimeError(
                f"{source_name} contains duplicate sentence_id "
                f"{sentence_id!r}."
            )


        persisted_sentence_ids.append(
            sentence_id
        )


        record_lookup[
            sentence_id
        ] = record


    canonical_set = set(
        BLOCK_7_SENTENCE_IDS
    )


    persisted_set = set(
        persisted_sentence_ids
    )


    set_valid = (
        persisted_set
        ==
        canonical_set
    )


    if not set_valid:

        missing_ids = sorted(
            canonical_set
            -
            persisted_set
        )


        unexpected_ids = sorted(
            persisted_set
            -
            canonical_set
        )


        raise RuntimeError(
            f"{source_name} sentence-ID set does not match the "
            "canonical Notebook 08 identity. "
            f"Missing IDs: {missing_ids[:5]}; "
            f"unexpected IDs: {unexpected_ids[:5]}."
        )


    persisted_order_canonical = (
        tuple(
            persisted_sentence_ids
        )
        ==
        tuple(
            BLOCK_7_SENTENCE_IDS
        )
    )


    canonical_records = [
        record_lookup[
            sentence_id
        ]

        for sentence_id
        in BLOCK_7_SENTENCE_IDS
    ]


    canonical_order_valid = all(
        canonical_records[
            record_index
        ][
            "identity"
        ][
            "sentence_id"
        ]
        ==
        BLOCK_7_SENTENCE_IDS[
            record_index
        ]

        for record_index
        in range(
            BLOCK_7_SENTENCE_COUNT
        )
    )


    return (
        canonical_records,
        tuple(
            persisted_sentence_ids
        ),
        set_valid,
        persisted_order_canonical,
        canonical_order_valid,
    )


# =============================================================================
# Restore inherited Notebook 05 → Notebook 06 handover
# =============================================================================

BLOCK_7_INHERITED_NOTEBOOK_05_TO_06_HANDOVER = (
    BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER.get(
        "inherited_notebook_05_to_06_handover",
        {},
    )
)


if not isinstance(
    BLOCK_7_INHERITED_NOTEBOOK_05_TO_06_HANDOVER,
    dict,
):

    raise RuntimeError(
        "Inherited Notebook 05 → Notebook 06 handover "
        "is unavailable."
    )


# =============================================================================
# FACTUAL SUPERVISION
# =============================================================================

BLOCK_7_FACTUAL_PERSISTED_ARTEFACTS = (
    BLOCK_7_NOTEBOOK_06_TO_07_HANDOVER.get(
        "factual_persisted_artefacts",
        {},
    )
)


BLOCK_7_FACTUAL_TARGET_REFERENCE = (
    BLOCK_7_FACTUAL_PERSISTED_ARTEFACTS.get(
        "target_contract",
        {},
    )
)


BLOCK_7_FACTUAL_TARGET_DRIVE_FILE_ID = (
    BLOCK_7_FACTUAL_TARGET_REFERENCE.get(
        "drive_file_id"
    )
)


BLOCK_7_FACTUAL_TARGET_FILENAME = (
    BLOCK_7_FACTUAL_TARGET_REFERENCE.get(
        "filename"
    )
)


if not BLOCK_7_FACTUAL_TARGET_DRIVE_FILE_ID:

    raise RuntimeError(
        "Notebook 06 → Notebook 07 handover does not contain "
        "the factual target-contract Drive file ID."
    )


BLOCK_7_FACTUAL_ARTEFACT = (
    block_7_load_json_from_drive(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_7_FACTUAL_TARGET_DRIVE_FILE_ID,
    )
)


# =============================================================================
# Factual target contract
# =============================================================================

BLOCK_7_FACTUAL_ARCHITECTURE = (
    BLOCK_7_FACTUAL_ARTEFACT.get(
        "architecture",
        {},
    )
)


BLOCK_7_FACTUAL_RECORDS = deepcopy(
    BLOCK_7_FACTUAL_ARTEFACT.get(
        "records",
        [],
    )
)


(
    BLOCK_7_FACTUAL_RECORDS,
    BLOCK_7_FACTUAL_PERSISTED_SENTENCE_IDS,
    BLOCK_7_FACTUAL_CANONICAL_SET_VALID,
    BLOCK_7_FACTUAL_PERSISTED_ORDER_CANONICAL,
    BLOCK_7_FACTUAL_CANONICAL_ORDER_VALID,
) = block_7_reorder_records_to_canonical(
    BLOCK_7_FACTUAL_RECORDS,
    "Factual supervision",
)


BLOCK_7_FACTUAL_DIMENSION_ORDER = tuple(
    BLOCK_7_FACTUAL_ARCHITECTURE.get(
        "dimension_order",
        [],
    )
)


BLOCK_7_FACTUAL_CONTRACT_VALID = all(
    [
        isinstance(
            BLOCK_7_FACTUAL_ARTEFACT,
            dict,
        ),

        (
            BLOCK_7_FACTUAL_ARTEFACT.get(
                "artefact_type"
            )
            ==
            "media_ai_factual_target_contract"
        ),

        (
            BLOCK_7_FACTUAL_ARCHITECTURE.get(
                "factual_dimension"
            )
            ==
            BLOCK_7_PATHWAY_DIMS[
                "factual"
            ]
        ),

        (
            len(
                BLOCK_7_FACTUAL_DIMENSION_ORDER
            )
            ==
            BLOCK_7_PATHWAY_DIMS[
                "factual"
            ]
        ),

        (
            len(
                BLOCK_7_FACTUAL_RECORDS
            )
            ==
            BLOCK_7_SENTENCE_COUNT
        ),
    ]
)


if not BLOCK_7_FACTUAL_CONTRACT_VALID:

    raise RuntimeError(
        "Persisted Notebook 06 factual target contract "
        "failed validation."
    )


# =============================================================================
# Reconstruct factual supervision
# =============================================================================

BLOCK_7_FACTUAL_SENTENCE_IDS = []

factual_target_rows = []

factual_mask_rows = []


for (
    record_index,
    record,
) in enumerate(
    BLOCK_7_FACTUAL_RECORDS
):

    if not isinstance(
        record,
        dict,
    ):

        raise TypeError(
            f"Factual record {record_index} is not dictionary-like."
        )


    identity = record.get(
        "identity",
        {},
    )


    sentence_id = (
        identity.get(
            "sentence_id"
        )
        if isinstance(
            identity,
            dict,
        )
        else None
    )


    if not sentence_id:

        raise RuntimeError(
            f"Factual record {record_index} has no sentence_id."
        )


    target = np.asarray(
        record.get(
            "factual_target"
        ),
        dtype=np.float32,
    )


    mask = np.asarray(
        record.get(
            "factual_target_mask"
        ),
        dtype=bool,
    )


    if target.shape != (
        BLOCK_7_PATHWAY_DIMS[
            "factual"
        ],
    ):

        raise RuntimeError(
            f"Factual record {record_index} target shape is invalid."
        )


    if mask.shape != (
        BLOCK_7_PATHWAY_DIMS[
            "factual"
        ],
    ):

        raise RuntimeError(
            f"Factual record {record_index} mask shape is invalid."
        )


    if not np.isfinite(
        target[
            mask
        ]
    ).all():

        raise RuntimeError(
            f"Factual record {record_index} contains "
            "non-finite observed supervision."
        )


    BLOCK_7_FACTUAL_SENTENCE_IDS.append(
        sentence_id
    )


    factual_target_rows.append(
        target
    )


    factual_mask_rows.append(
        mask
    )


BLOCK_7_FACTUAL_CANONICAL_ALIGNMENT_VALID = all(
    [
        BLOCK_7_FACTUAL_CANONICAL_SET_VALID,
        BLOCK_7_FACTUAL_CANONICAL_ORDER_VALID,

        tuple(
            BLOCK_7_FACTUAL_SENTENCE_IDS
        )
        ==
        tuple(
            BLOCK_7_SENTENCE_IDS
        ),
    ]
)


if not BLOCK_7_FACTUAL_CANONICAL_ALIGNMENT_VALID:

    raise RuntimeError(
        "Factual supervision could not be aligned safely to the "
        "canonical sentence identity."
    )


BLOCK_7_FACTUAL_CURRENT_SUPERVISION_RAW = np.asarray(
    factual_target_rows,
    dtype=np.float32,
)


BLOCK_7_FACTUAL_CURRENT_MASK = np.asarray(
    factual_mask_rows,
    dtype=bool,
)


BLOCK_7_FACTUAL_CURRENT_SUPERVISION = np.where(
    BLOCK_7_FACTUAL_CURRENT_MASK,
    BLOCK_7_FACTUAL_CURRENT_SUPERVISION_RAW,
    0.0,
).astype(
    np.float32
)


BLOCK_7_FACTUAL_SHAPES_VALID = all(
    [
        (
            BLOCK_7_FACTUAL_CURRENT_SUPERVISION.shape
            ==
            (
                BLOCK_7_SENTENCE_COUNT,
                BLOCK_7_PATHWAY_DIMS[
                    "factual"
                ],
            )
        ),

        (
            BLOCK_7_FACTUAL_CURRENT_MASK.shape
            ==
            (
                BLOCK_7_SENTENCE_COUNT,
                BLOCK_7_PATHWAY_DIMS[
                    "factual"
                ],
            )
        ),
    ]
)


BLOCK_7_FACTUAL_OBSERVED_VALUES_FINITE = bool(
    np.isfinite(
        BLOCK_7_FACTUAL_CURRENT_SUPERVISION[
            BLOCK_7_FACTUAL_CURRENT_MASK
        ]
    ).all()
)


if not all(
    [
        BLOCK_7_FACTUAL_SHAPES_VALID,
        BLOCK_7_FACTUAL_OBSERVED_VALUES_FINITE,
    ]
):

    raise RuntimeError(
        "Reconstructed factual supervision failed validation."
    )


# =============================================================================
# PSYCHOLOGICAL SUPERVISION
# Restore original accepted Notebook 03 supervision
# =============================================================================

BLOCK_7_NOTEBOOK_03_SUPERVISION_REFERENCE = (
    BLOCK_7_INHERITED_NOTEBOOK_05_TO_06_HANDOVER.get(
        "notebook_03_supervision",
        {},
    )
)


BLOCK_7_NOTEBOOK_03_ACCEPTED_FILE_ID = (
    BLOCK_7_NOTEBOOK_03_SUPERVISION_REFERENCE.get(
        "accepted_file_id"
    )
)


BLOCK_7_NOTEBOOK_03_ACCEPTED_RECORD_COUNT = (
    BLOCK_7_NOTEBOOK_03_SUPERVISION_REFERENCE.get(
        "accepted_record_count"
    )
)


if not BLOCK_7_NOTEBOOK_03_ACCEPTED_FILE_ID:

    raise RuntimeError(
        "Inherited Notebook 05 → Notebook 06 handover does not "
        "contain the accepted Notebook 03 supervision file ID."
    )


BLOCK_7_PSYCHOLOGICAL_ACCEPTED_ARTEFACT = (
    block_7_load_json_from_drive(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_7_NOTEBOOK_03_ACCEPTED_FILE_ID,
    )
)


BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS = deepcopy(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_ARTEFACT.get(
        "records",
        [],
    )
)


(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS,
    BLOCK_7_PSYCHOLOGICAL_PERSISTED_SENTENCE_IDS,
    BLOCK_7_PSYCHOLOGICAL_CANONICAL_SET_VALID,
    BLOCK_7_PSYCHOLOGICAL_PERSISTED_ORDER_CANONICAL,
    BLOCK_7_PSYCHOLOGICAL_CANONICAL_ORDER_VALID,
) = block_7_reorder_records_to_canonical(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS,
    "Psychological supervision",
)


BLOCK_7_PSYCHOLOGICAL_RECORD_COUNT_VALID = all(
    [
        (
            len(
                BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS
            )
            ==
            BLOCK_7_SENTENCE_COUNT
        ),

        (
            int(
                BLOCK_7_NOTEBOOK_03_ACCEPTED_RECORD_COUNT
            )
            ==
            BLOCK_7_SENTENCE_COUNT
        ),
    ]
)


if not BLOCK_7_PSYCHOLOGICAL_RECORD_COUNT_VALID:

    raise RuntimeError(
        "Notebook 03 accepted psychological supervision "
        "record count is invalid."
    )


# =============================================================================
# Psychological canonical dimensions
# =============================================================================

BLOCK_7_EMOTION_DIMENSION_ORDER = (
    "fear",
    "anger",
    "sadness",
    "disgust",
    "tension",
    "uncertainty",
    "hope",
    "relief",
    "trust",
    "compassion",
    "solidarity",
)


BLOCK_7_PSYCHOLOGICAL_FAMILY_ORDER = (
    "REV",
    "TEV",
    "IEV",
    "EW",
)


BLOCK_7_PSYCHOLOGICAL_FAMILY_KEYS = {
    "REV":
        "reported_emotion",

    "TEV":
        "transformative_emotion",

    "IEV":
        "immediate_emotional_state",

    "EW":
        "emotional_weight",
}


# =============================================================================
# Psychological canonical identity
# =============================================================================

BLOCK_7_PSYCHOLOGICAL_SENTENCE_IDS = []


for (
    record_index,
    record,
) in enumerate(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS
):

    if not isinstance(
        record,
        dict,
    ):

        raise TypeError(
            f"Psychological record {record_index} "
            "is not dictionary-like."
        )


    identity = record.get(
        "identity",
        {},
    )


    sentence_id = (
        identity.get(
            "sentence_id"
        )
        if isinstance(
            identity,
            dict,
        )
        else None
    )


    if not sentence_id:

        raise RuntimeError(
            f"Psychological record {record_index} has no sentence_id."
        )


    BLOCK_7_PSYCHOLOGICAL_SENTENCE_IDS.append(
        sentence_id
    )


BLOCK_7_PSYCHOLOGICAL_CANONICAL_ALIGNMENT_VALID = all(
    [
        BLOCK_7_PSYCHOLOGICAL_CANONICAL_SET_VALID,
        BLOCK_7_PSYCHOLOGICAL_CANONICAL_ORDER_VALID,

        tuple(
            BLOCK_7_PSYCHOLOGICAL_SENTENCE_IDS
        )
        ==
        tuple(
            BLOCK_7_SENTENCE_IDS
        ),
    ]
)


if not BLOCK_7_PSYCHOLOGICAL_CANONICAL_ALIGNMENT_VALID:

    raise RuntimeError(
        "Psychological supervision could not be aligned safely to the "
        "canonical sentence identity."
    )


# =============================================================================
# Psychological family extraction
# =============================================================================

def block_7_extract_emotion_family(
    records,
    annotation_key,
):

    targets = np.zeros(
        (
            BLOCK_7_SENTENCE_COUNT,
            len(
                BLOCK_7_EMOTION_DIMENSION_ORDER
            ),
        ),
        dtype=np.float32,
    )


    masks = np.zeros(
        (
            BLOCK_7_SENTENCE_COUNT,
            len(
                BLOCK_7_EMOTION_DIMENSION_ORDER
            ),
        ),
        dtype=bool,
    )


    for (
        record_index,
        record,
    ) in enumerate(
        records
    ):

        annotations = record.get(
            "annotations",
            {},
        )


        family = annotations.get(
            annotation_key,
            {},
        )


        if not isinstance(
            family,
            dict,
        ):

            raise RuntimeError(
                f"Psychological record {record_index} does not "
                f"contain {annotation_key}."
            )


        for (
            dimension_index,
            emotion_name,
        ) in enumerate(
            BLOCK_7_EMOTION_DIMENSION_ORDER
        ):

            annotation = family.get(
                emotion_name
            )


            if not isinstance(
                annotation,
                dict,
            ):

                raise RuntimeError(
                    f"Psychological record {record_index} is missing "
                    f"{annotation_key}.{emotion_name}."
                )


            available = (
                annotation.get(
                    "value_state"
                )
                ==
                "observed"
            )


            value = annotation.get(
                "value"
            )


            if available:

                if value is None:

                    raise RuntimeError(
                        f"Observed psychological target "
                        f"{annotation_key}.{emotion_name} "
                        f"in record {record_index} has no value."
                    )


                numeric_value = float(
                    value
                )


                if not np.isfinite(
                    numeric_value
                ):

                    raise RuntimeError(
                        f"Observed psychological target "
                        f"{annotation_key}.{emotion_name} "
                        f"in record {record_index} is non-finite."
                    )


                targets[
                    record_index,
                    dimension_index,
                ] = numeric_value


                masks[
                    record_index,
                    dimension_index,
                ] = True


    return (
        targets,
        masks,
    )


# =============================================================================
# REV
# =============================================================================

(
    BLOCK_7_REV_TARGETS,
    BLOCK_7_REV_MASK,
) = block_7_extract_emotion_family(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS,
    "reported_emotion",
)


# =============================================================================
# TEV
# =============================================================================

(
    BLOCK_7_TEV_TARGETS,
    BLOCK_7_TEV_MASK,
) = block_7_extract_emotion_family(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS,
    "transformative_emotion",
)


# =============================================================================
# IEV
# =============================================================================

(
    BLOCK_7_IEV_TARGETS,
    BLOCK_7_IEV_MASK,
) = block_7_extract_emotion_family(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS,
    "immediate_emotional_state",
)


# =============================================================================
# EW
# =============================================================================

BLOCK_7_EW_TARGETS = np.zeros(
    (
        BLOCK_7_SENTENCE_COUNT,
        1,
    ),
    dtype=np.float32,
)


BLOCK_7_EW_MASK = np.zeros(
    (
        BLOCK_7_SENTENCE_COUNT,
        1,
    ),
    dtype=bool,
)


for (
    record_index,
    record,
) in enumerate(
    BLOCK_7_PSYCHOLOGICAL_ACCEPTED_RECORDS
):

    annotations = record.get(
        "annotations",
        {},
    )


    emotional_weight = annotations.get(
        "emotional_weight",
        {},
    )


    if not isinstance(
        emotional_weight,
        dict,
    ):

        raise RuntimeError(
            f"Psychological record {record_index} "
            "does not contain emotional_weight."
        )


    available = (
        emotional_weight.get(
            "value_state"
        )
        ==
        "observed"
    )


    value = emotional_weight.get(
        "value"
    )


    if available:

        if value is None:

            raise RuntimeError(
                f"Observed emotional_weight in record "
                f"{record_index} has no value."
            )


        numeric_value = float(
            value
        )


        if not np.isfinite(
            numeric_value
        ):

            raise RuntimeError(
                f"Observed emotional_weight in record "
                f"{record_index} is non-finite."
            )


        BLOCK_7_EW_TARGETS[
            record_index,
            0,
        ] = numeric_value


        BLOCK_7_EW_MASK[
            record_index,
            0,
        ] = True


# =============================================================================
# Psychological supervision-count validation
# =============================================================================

BLOCK_7_PSYCHOLOGICAL_REPRODUCED_SUPERVISION_COUNTS = {
    "REV":
        (
            int(
                BLOCK_7_REV_MASK.sum()
            ),
            int(
                BLOCK_7_REV_MASK.size
            ),
        ),

    "TEV":
        (
            int(
                BLOCK_7_TEV_MASK.sum()
            ),
            int(
                BLOCK_7_TEV_MASK.size
            ),
        ),

    "IEV":
        (
            int(
                BLOCK_7_IEV_MASK.sum()
            ),
            int(
                BLOCK_7_IEV_MASK.size
            ),
        ),

    "EW":
        (
            int(
                BLOCK_7_EW_MASK.sum()
            ),
            int(
                BLOCK_7_EW_MASK.size
            ),
        ),
}


BLOCK_7_PSYCHOLOGICAL_SUPERVISION_COUNTS_VALID = all(
    [
        (
            BLOCK_7_PSYCHOLOGICAL_REPRODUCED_SUPERVISION_COUNTS[
                family_name
            ][
                1
            ]
            >
            0
        )

        for family_name
        in BLOCK_7_PSYCHOLOGICAL_FAMILY_ORDER
    ]
)


if not BLOCK_7_PSYCHOLOGICAL_SUPERVISION_COUNTS_VALID:

    raise RuntimeError(
        "Recovered Notebook 03 psychological supervision counts "
        "are invalid."
    )


# =============================================================================
# Canonical 34-dimensional psychological supervision
# =============================================================================

BLOCK_7_PSYCHOLOGICAL_CURRENT_SUPERVISION = np.concatenate(
    [
        BLOCK_7_REV_TARGETS,
        BLOCK_7_TEV_TARGETS,
        BLOCK_7_IEV_TARGETS,
        BLOCK_7_EW_TARGETS,
    ],
    axis=
        1,
).astype(
    np.float32
)


BLOCK_7_PSYCHOLOGICAL_CURRENT_MASK = np.concatenate(
    [
        BLOCK_7_REV_MASK,
        BLOCK_7_TEV_MASK,
        BLOCK_7_IEV_MASK,
        BLOCK_7_EW_MASK,
    ],
    axis=
        1,
).astype(
    bool
)


BLOCK_7_PSYCHOLOGICAL_SHAPES_VALID = all(
    [
        (
            BLOCK_7_PSYCHOLOGICAL_CURRENT_SUPERVISION.shape
            ==
            (
                BLOCK_7_SENTENCE_COUNT,
                BLOCK_7_PATHWAY_DIMS[
                    "psychological"
                ],
            )
        ),

        (
            BLOCK_7_PSYCHOLOGICAL_CURRENT_MASK.shape
            ==
            (
                BLOCK_7_SENTENCE_COUNT,
                BLOCK_7_PATHWAY_DIMS[
                    "psychological"
                ],
            )
        ),
    ]
)


BLOCK_7_PSYCHOLOGICAL_OBSERVED_VALUES_FINITE = bool(
    np.isfinite(
        BLOCK_7_PSYCHOLOGICAL_CURRENT_SUPERVISION[
            BLOCK_7_PSYCHOLOGICAL_CURRENT_MASK
        ]
    ).all()
)


if not all(
    [
        BLOCK_7_PSYCHOLOGICAL_SHAPES_VALID,
        BLOCK_7_PSYCHOLOGICAL_OBSERVED_VALUES_FINITE,
    ]
):

    raise RuntimeError(
        "Canonical psychological supervision failed validation."
    )


BLOCK_7_PSYCHOLOGICAL_FAMILIES = {
    "REV":
        {
            "targets":
                BLOCK_7_REV_TARGETS,

            "mask":
                BLOCK_7_REV_MASK,

            "observed_elements":
                int(
                    BLOCK_7_REV_MASK.sum()
                ),

            "possible_elements":
                int(
                    BLOCK_7_REV_MASK.size
                ),
        },

    "TEV":
        {
            "targets":
                BLOCK_7_TEV_TARGETS,

            "mask":
                BLOCK_7_TEV_MASK,

            "observed_elements":
                int(
                    BLOCK_7_TEV_MASK.sum()
                ),

            "possible_elements":
                int(
                    BLOCK_7_TEV_MASK.size
                ),
        },

    "IEV":
        {
            "targets":
                BLOCK_7_IEV_TARGETS,

            "mask":
                BLOCK_7_IEV_MASK,

            "observed_elements":
                int(
                    BLOCK_7_IEV_MASK.sum()
                ),

            "possible_elements":
                int(
                    BLOCK_7_IEV_MASK.size
                ),
        },

    "EW":
        {
            "targets":
                BLOCK_7_EW_TARGETS,

            "mask":
                BLOCK_7_EW_MASK,

            "observed_elements":
                int(
                    BLOCK_7_EW_MASK.sum()
                ),

            "possible_elements":
                int(
                    BLOCK_7_EW_MASK.size
                ),
        },
}


# =============================================================================
# SOCIAL SUPERVISION
# =============================================================================

BLOCK_7_SOCIAL_ACCEPTED_FILENAME = (
    "notebook_07_accepted_social_supervision.json"
)


BLOCK_7_SOCIAL_ACCEPTED_DRIVE_FILE = (
    block_7_find_drive_file_by_name(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_7_SOCIAL_ACCEPTED_FILENAME,
    )
)


if BLOCK_7_SOCIAL_ACCEPTED_DRIVE_FILE is None:

    raise RuntimeError(
        "Persisted Notebook 07 accepted social supervision "
        "could not be located."
    )


BLOCK_7_SOCIAL_ACCEPTED_DRIVE_FILE_ID = (
    BLOCK_7_SOCIAL_ACCEPTED_DRIVE_FILE[
        "id"
    ]
)


BLOCK_7_SOCIAL_ACCEPTED_ARTEFACT = (
    block_7_load_json_from_drive(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_7_SOCIAL_ACCEPTED_DRIVE_FILE_ID,
    )
)


BLOCK_7_SOCIAL_ACCEPTED_ARCHITECTURE = (
    BLOCK_7_SOCIAL_ACCEPTED_ARTEFACT.get(
        "architecture",
        {},
    )
)


BLOCK_7_SOCIAL_ACCEPTED_RECORDS = deepcopy(
    BLOCK_7_SOCIAL_ACCEPTED_ARTEFACT.get(
        "records",
        [],
    )
)


(
    BLOCK_7_SOCIAL_ACCEPTED_RECORDS,
    BLOCK_7_SOCIAL_PERSISTED_SENTENCE_IDS,
    BLOCK_7_SOCIAL_CANONICAL_SET_VALID,
    BLOCK_7_SOCIAL_PERSISTED_ORDER_CANONICAL,
    BLOCK_7_SOCIAL_CANONICAL_ORDER_VALID,
) = block_7_reorder_records_to_canonical(
    BLOCK_7_SOCIAL_ACCEPTED_RECORDS,
    "Social supervision",
)


BLOCK_7_SOCIAL_DIMENSION_ORDER = tuple(
    BLOCK_7_SOCIAL_ACCEPTED_ARCHITECTURE.get(
        "dimension_order",
        [],
    )
)


BLOCK_7_SOCIAL_ACCEPTED_CONTRACT_VALID = all(
    [
        (
            BLOCK_7_SOCIAL_ACCEPTED_ARTEFACT.get(
                "artefact_type"
            )
            ==
            "media_ai_accepted_social_supervision"
        ),

        (
            BLOCK_7_SOCIAL_ACCEPTED_ARCHITECTURE.get(
                "representation_dimension"
            )
            ==
            BLOCK_7_PATHWAY_DIMS[
                "social"
            ]
        ),

        (
            len(
                BLOCK_7_SOCIAL_DIMENSION_ORDER
            )
            ==
            BLOCK_7_PATHWAY_DIMS[
                "social"
            ]
        ),

        (
            len(
                BLOCK_7_SOCIAL_ACCEPTED_RECORDS
            )
            ==
            BLOCK_7_SENTENCE_COUNT
        ),
    ]
)


if not BLOCK_7_SOCIAL_ACCEPTED_CONTRACT_VALID:

    raise RuntimeError(
        "Accepted Notebook 07 social supervision contract is invalid."
    )


# =============================================================================
# Reconstruct social supervision
# =============================================================================

BLOCK_7_SOCIAL_SENTENCE_IDS = []


BLOCK_7_SOCIAL_CURRENT_SUPERVISION_RAW = np.full(
    (
        BLOCK_7_SENTENCE_COUNT,
        BLOCK_7_PATHWAY_DIMS[
            "social"
        ],
    ),
    np.nan,
    dtype=np.float32,
)


BLOCK_7_SOCIAL_CURRENT_MASK = np.zeros(
    (
        BLOCK_7_SENTENCE_COUNT,
        BLOCK_7_PATHWAY_DIMS[
            "social"
        ],
    ),
    dtype=bool,
)


for (
    record_index,
    record,
) in enumerate(
    BLOCK_7_SOCIAL_ACCEPTED_RECORDS
):

    identity = record.get(
        "identity",
        {},
    )


    sentence_id = (
        identity.get(
            "sentence_id"
        )
        if isinstance(
            identity,
            dict,
        )
        else None
    )


    if not sentence_id:

        raise RuntimeError(
            f"Social record {record_index} has no sentence_id."
        )


    BLOCK_7_SOCIAL_SENTENCE_IDS.append(
        sentence_id
    )


    social_annotations = record.get(
        "social_annotations",
        {},
    )


    if not isinstance(
        social_annotations,
        dict,
    ):

        raise RuntimeError(
            f"Social record {record_index} has no "
            "social_annotations dictionary."
        )


    for (
        dimension_index,
        dimension_name,
    ) in enumerate(
        BLOCK_7_SOCIAL_DIMENSION_ORDER
    ):

        annotation = social_annotations.get(
            dimension_name
        )


        if not isinstance(
            annotation,
            dict,
        ):

            raise RuntimeError(
                f"Social record {record_index} is missing "
                f"{dimension_name}."
            )


        available = bool(
            annotation.get(
                "available",
                False,
            )
        )


        value = annotation.get(
            "value"
        )


        if available:

            if value is None:

                raise RuntimeError(
                    f"Available social target "
                    f"{dimension_name} in record "
                    f"{record_index} has no value."
                )


            numeric_value = float(
                value
            )


            if not np.isfinite(
                numeric_value
            ):

                raise RuntimeError(
                    f"Available social target "
                    f"{dimension_name} is non-finite."
                )


            BLOCK_7_SOCIAL_CURRENT_SUPERVISION_RAW[
                record_index,
                dimension_index,
            ] = numeric_value


            BLOCK_7_SOCIAL_CURRENT_MASK[
                record_index,
                dimension_index,
            ] = True


BLOCK_7_SOCIAL_CANONICAL_ALIGNMENT_VALID = all(
    [
        BLOCK_7_SOCIAL_CANONICAL_SET_VALID,
        BLOCK_7_SOCIAL_CANONICAL_ORDER_VALID,

        tuple(
            BLOCK_7_SOCIAL_SENTENCE_IDS
        )
        ==
        tuple(
            BLOCK_7_SENTENCE_IDS
        ),
    ]
)


if not BLOCK_7_SOCIAL_CANONICAL_ALIGNMENT_VALID:

    raise RuntimeError(
        "Social supervision could not be aligned safely to the "
        "canonical sentence identity."
    )


BLOCK_7_SOCIAL_MASKED_VALUES_NAN = bool(
    np.isnan(
        BLOCK_7_SOCIAL_CURRENT_SUPERVISION_RAW[
            ~BLOCK_7_SOCIAL_CURRENT_MASK
        ]
    ).all()
)


BLOCK_7_SOCIAL_OBSERVED_VALUES_FINITE = bool(
    np.isfinite(
        BLOCK_7_SOCIAL_CURRENT_SUPERVISION_RAW[
            BLOCK_7_SOCIAL_CURRENT_MASK
        ]
    ).all()
)


if not all(
    [
        BLOCK_7_SOCIAL_MASKED_VALUES_NAN,
        BLOCK_7_SOCIAL_OBSERVED_VALUES_FINITE,
    ]
):

    raise RuntimeError(
        "Social supervision violates the masking contract."
    )


BLOCK_7_SOCIAL_CURRENT_SUPERVISION = np.where(
    BLOCK_7_SOCIAL_CURRENT_MASK,
    BLOCK_7_SOCIAL_CURRENT_SUPERVISION_RAW,
    0.0,
).astype(
    np.float32
)


# =============================================================================
# Canonical current-supervision matrices
# =============================================================================

BLOCK_7_CURRENT_SUPERVISION = {
    "factual":
        BLOCK_7_FACTUAL_CURRENT_SUPERVISION,

    "psychological":
        BLOCK_7_PSYCHOLOGICAL_CURRENT_SUPERVISION,

    "social":
        BLOCK_7_SOCIAL_CURRENT_SUPERVISION,
}


BLOCK_7_CURRENT_SUPERVISION_MASKS = {
    "factual":
        BLOCK_7_FACTUAL_CURRENT_MASK,

    "psychological":
        BLOCK_7_PSYCHOLOGICAL_CURRENT_MASK,

    "social":
        BLOCK_7_SOCIAL_CURRENT_MASK,
}


# =============================================================================
# Common supervision-shape validation
# =============================================================================

BLOCK_7_CURRENT_SUPERVISION_SHAPES_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_7_PATHWAY_DIMS.items():

    expected_shape = (
        BLOCK_7_SENTENCE_COUNT,
        pathway_dim,
    )


    BLOCK_7_CURRENT_SUPERVISION_SHAPES_VALID[
        pathway_name
    ] = all(
        [
            (
                BLOCK_7_CURRENT_SUPERVISION[
                    pathway_name
                ].shape
                ==
                expected_shape
            ),

            (
                BLOCK_7_CURRENT_SUPERVISION_MASKS[
                    pathway_name
                ].shape
                ==
                expected_shape
            ),
        ]
    )


BLOCK_7_ALL_CURRENT_SUPERVISION_SHAPES_VALID = all(
    BLOCK_7_CURRENT_SUPERVISION_SHAPES_VALID.values()
)


if not BLOCK_7_ALL_CURRENT_SUPERVISION_SHAPES_VALID:

    raise RuntimeError(
        "One or more representation-supervision matrices "
        "have invalid dimensions."
    )


# =============================================================================
# Construct causally available preceding conditions
# =============================================================================
#
# At the first sentence of every article:
#
#     h_(t-1) = 0
#
# Otherwise:
#
#     h_(t-1) = y_(t-1)
#
# These are controlled training conditions only.
# They are NOT Notebook 09 recurrent states.
# Cross-article state carry is explicitly prohibited.
# =============================================================================

BLOCK_7_ARTICLE_START_MASK = np.zeros(
    BLOCK_7_SENTENCE_COUNT,
    dtype=bool,
)


for sentence_index in range(
    BLOCK_7_SENTENCE_COUNT
):

    BLOCK_7_ARTICLE_START_MASK[
        sentence_index
    ] = (
        sentence_index == 0
        or
        BLOCK_7_ARTICLE_IDS_BY_SENTENCE[
            sentence_index
        ]
        !=
        BLOCK_7_ARTICLE_IDS_BY_SENTENCE[
            sentence_index - 1
        ]
    )


BLOCK_7_ARTICLE_START_INDICES = tuple(
    np.flatnonzero(
        BLOCK_7_ARTICLE_START_MASK
    ).tolist()
)


BLOCK_7_ARTICLE_BOUNDARY_COUNT_VALID = (
    len(
        BLOCK_7_ARTICLE_START_INDICES
    )
    ==
    BLOCK_7_ARTICLE_COUNT
)


if not BLOCK_7_ARTICLE_BOUNDARY_COUNT_VALID:

    raise RuntimeError(
        "Derived article-start count does not match the canonical "
        "article count."
    )


BLOCK_7_PRECEDING_CONDITIONS = {}

BLOCK_7_PRECEDING_CONDITION_MASKS = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_7_PATHWAY_DIMS.items():

    current_values = (
        BLOCK_7_CURRENT_SUPERVISION[
            pathway_name
        ]
    )


    current_mask = (
        BLOCK_7_CURRENT_SUPERVISION_MASKS[
            pathway_name
        ]
    )


    preceding_values = np.zeros(
        (
            BLOCK_7_SENTENCE_COUNT,
            pathway_dim,
        ),
        dtype=np.float32,
    )


    preceding_mask = np.zeros(
        (
            BLOCK_7_SENTENCE_COUNT,
            pathway_dim,
        ),
        dtype=bool,
    )


    for sentence_index in range(
        BLOCK_7_SENTENCE_COUNT
    ):

        if BLOCK_7_ARTICLE_START_MASK[
            sentence_index
        ]:

            preceding_values[
                sentence_index,
                :,
            ] = 0.0


            preceding_mask[
                sentence_index,
                :,
            ] = True


        else:

            preceding_values[
                sentence_index,
                :,
            ] = current_values[
                sentence_index - 1,
                :,
            ]


            preceding_mask[
                sentence_index,
                :,
            ] = current_mask[
                sentence_index - 1,
                :,
            ]


    BLOCK_7_PRECEDING_CONDITIONS[
        pathway_name
    ] = preceding_values


    BLOCK_7_PRECEDING_CONDITION_MASKS[
        pathway_name
    ] = preceding_mask


# =============================================================================
# Construct signed transformative targets
# =============================================================================
#
# Delta_t = y_t - h_(t-1)
#
# m_Delta,t = m_y,t AND m_h,t-1
# =============================================================================

BLOCK_7_TRANSFORMATIVE_TARGETS = {}

BLOCK_7_TRANSFORMATIVE_MASKS = {}


for pathway_name in BLOCK_7_PATHWAY_DIMS:

    current_values = (
        BLOCK_7_CURRENT_SUPERVISION[
            pathway_name
        ]
    )


    current_mask = (
        BLOCK_7_CURRENT_SUPERVISION_MASKS[
            pathway_name
        ]
    )


    preceding_values = (
        BLOCK_7_PRECEDING_CONDITIONS[
            pathway_name
        ]
    )


    preceding_mask = (
        BLOCK_7_PRECEDING_CONDITION_MASKS[
            pathway_name
        ]
    )


    transformation_mask = (
        current_mask
        &
        preceding_mask
    )


    raw_transformation = (
        current_values
        -
        preceding_values
    )


    transformation_target = np.where(
        transformation_mask,
        raw_transformation,
        0.0,
    ).astype(
        np.float32
    )


    BLOCK_7_TRANSFORMATIVE_TARGETS[
        pathway_name
    ] = transformation_target


    BLOCK_7_TRANSFORMATIVE_MASKS[
        pathway_name
    ] = transformation_mask


# =============================================================================
# Causal target validation
# =============================================================================

BLOCK_7_INITIAL_CONDITIONS_ZERO = all(
    np.array_equal(
        BLOCK_7_PRECEDING_CONDITIONS[
            pathway_name
        ][
            list(
                BLOCK_7_ARTICLE_START_INDICES
            )
        ],
        np.zeros(
            (
                len(
                    BLOCK_7_ARTICLE_START_INDICES
                ),
                pathway_dim,
            ),
            dtype=np.float32,
        ),
    )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_7_PATHWAY_DIMS.items()
)


BLOCK_7_ARTICLE_START_MASKS_AVAILABLE = all(
    bool(
        BLOCK_7_PRECEDING_CONDITION_MASKS[
            pathway_name
        ][
            list(
                BLOCK_7_ARTICLE_START_INDICES
            )
        ].all()
    )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
)


BLOCK_7_PREVIOUS_ONLY_VALUES_VALID = True

BLOCK_7_PREVIOUS_ONLY_MASKS_VALID = True


for pathway_name in BLOCK_7_PATHWAY_DIMS:

    for sentence_index in range(
        BLOCK_7_SENTENCE_COUNT
    ):

        if BLOCK_7_ARTICLE_START_MASK[
            sentence_index
        ]:

            continue


        if not np.array_equal(
            BLOCK_7_PRECEDING_CONDITIONS[
                pathway_name
            ][
                sentence_index
            ],
            BLOCK_7_CURRENT_SUPERVISION[
                pathway_name
            ][
                sentence_index - 1
            ],
        ):

            BLOCK_7_PREVIOUS_ONLY_VALUES_VALID = False


        if not np.array_equal(
            BLOCK_7_PRECEDING_CONDITION_MASKS[
                pathway_name
            ][
                sentence_index
            ],
            BLOCK_7_CURRENT_SUPERVISION_MASKS[
                pathway_name
            ][
                sentence_index - 1
            ],
        ):

            BLOCK_7_PREVIOUS_ONLY_MASKS_VALID = False


BLOCK_7_CROSS_ARTICLE_STATE_CARRY = any(
    (
        sentence_index > 0
        and
        BLOCK_7_ARTICLE_START_MASK[
            sentence_index
        ]
        and
        np.any(
            BLOCK_7_PRECEDING_CONDITIONS[
                pathway_name
            ][
                sentence_index
            ]
            !=
            0.0
        )
    )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS

    for sentence_index
    in range(
        BLOCK_7_SENTENCE_COUNT
    )
)


BLOCK_7_FUTURE_INFORMATION_USED = False


BLOCK_7_CAUSAL_CONSTRUCTION_VALID = all(
    [
        BLOCK_7_ARTICLE_BOUNDARY_COUNT_VALID,
        BLOCK_7_INITIAL_CONDITIONS_ZERO,
        BLOCK_7_ARTICLE_START_MASKS_AVAILABLE,
        BLOCK_7_PREVIOUS_ONLY_VALUES_VALID,
        BLOCK_7_PREVIOUS_ONLY_MASKS_VALID,
        not BLOCK_7_CROSS_ARTICLE_STATE_CARRY,
        not BLOCK_7_FUTURE_INFORMATION_USED,
    ]
)


if not BLOCK_7_CAUSAL_CONSTRUCTION_VALID:

    raise RuntimeError(
        "Transformative target construction violated the causal "
        "or article-boundary sequence contract."
    )


# =============================================================================
# Target shape and numerical validation
# =============================================================================

BLOCK_7_TARGET_SHAPES_VALID = {}

BLOCK_7_AVAILABLE_TARGETS_FINITE = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_7_PATHWAY_DIMS.items():

    expected_shape = (
        BLOCK_7_SENTENCE_COUNT,
        pathway_dim,
    )


    target_matrix = (
        BLOCK_7_TRANSFORMATIVE_TARGETS[
            pathway_name
        ]
    )


    target_mask = (
        BLOCK_7_TRANSFORMATIVE_MASKS[
            pathway_name
        ]
    )


    BLOCK_7_TARGET_SHAPES_VALID[
        pathway_name
    ] = all(
        [
            target_matrix.shape
            ==
            expected_shape,

            target_mask.shape
            ==
            expected_shape,
        ]
    )


    available_values = (
        target_matrix[
            target_mask
        ]
    )


    BLOCK_7_AVAILABLE_TARGETS_FINITE[
        pathway_name
    ] = (
        available_values.size
        >
        0
        and
        bool(
            np.isfinite(
                available_values
            ).all()
        )
    )


BLOCK_7_ALL_TARGET_SHAPES_VALID = all(
    BLOCK_7_TARGET_SHAPES_VALID.values()
)


BLOCK_7_ALL_AVAILABLE_TARGETS_FINITE = all(
    BLOCK_7_AVAILABLE_TARGETS_FINITE.values()
)


if not all(
    [
        BLOCK_7_ALL_TARGET_SHAPES_VALID,
        BLOCK_7_ALL_AVAILABLE_TARGETS_FINITE,
    ]
):

    raise RuntimeError(
        "Transformative target dimensional or finite-value "
        "validation failed."
    )


# =============================================================================
# Dimension-level transformation diagnostics
# =============================================================================

BLOCK_7_TARGET_DIAGNOSTICS = {}

BLOCK_7_DERIVED_ACTIVE_DIMENSION_INDICES = {}

BLOCK_7_DERIVED_DEFERRED_DIMENSION_INDICES = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_7_PATHWAY_DIMS.items():

    target_matrix = (
        BLOCK_7_TRANSFORMATIVE_TARGETS[
            pathway_name
        ]
    )


    target_mask = (
        BLOCK_7_TRANSFORMATIVE_MASKS[
            pathway_name
        ]
    )


    dimension_diagnostics = []

    active_indices = []

    deferred_indices = []


    for dimension_index in range(
        pathway_dim
    ):

        dimension_mask = (
            target_mask[
                :,
                dimension_index,
            ]
        )


        dimension_values = (
            target_matrix[
                dimension_mask,
                dimension_index,
            ]
        )


        available_count = int(
            dimension_values.size
        )


        if available_count > 0:

            positive_count = int(
                np.sum(
                    dimension_values
                    >
                    0.0
                )
            )


            negative_count = int(
                np.sum(
                    dimension_values
                    <
                    0.0
                )
            )


            zero_count = int(
                np.sum(
                    dimension_values
                    ==
                    0.0
                )
            )


            variance = float(
                np.var(
                    dimension_values
                )
            )


            all_zero = bool(
                np.all(
                    dimension_values
                    ==
                    0.0
                )
            )


            variable = bool(
                available_count
                >=
                2
                and
                variance
                >
                1e-12
            )


        else:

            positive_count = 0

            negative_count = 0

            zero_count = 0

            variance = None

            all_zero = False

            variable = False


        training_active = bool(
            available_count
            >=
            2
            and
            variable
            and
            not all_zero
        )


        if training_active:

            active_indices.append(
                dimension_index
            )


            deferral_reason = None


        else:

            deferred_indices.append(
                dimension_index
            )


            if available_count == 0:

                deferral_reason = (
                    "unavailable"
                )


            elif available_count < 2:

                deferral_reason = (
                    "insufficient_available_observations"
                )


            elif all_zero:

                deferral_reason = (
                    "all_zero_transformation"
                )


            else:

                deferral_reason = (
                    "zero_or_insufficient_variance"
                )


        dimension_diagnostics.append(
            {
                "dimension_index":
                    dimension_index,

                "available":
                    available_count,

                "masked":
                    int(
                        BLOCK_7_SENTENCE_COUNT
                        -
                        available_count
                    ),

                "positive":
                    positive_count,

                "negative":
                    negative_count,

                "zero":
                    zero_count,

                "variance":
                    variance,

                "variable":
                    variable,

                "all_zero":
                    all_zero,

                "training_active":
                    training_active,

                "deferral_reason":
                    deferral_reason,
            }
        )


    BLOCK_7_TARGET_DIAGNOSTICS[
        pathway_name
    ] = {
        "total_elements":
            int(
                target_matrix.size
            ),

        "available_elements":
            int(
                target_mask.sum()
            ),

        "masked_elements":
            int(
                target_matrix.size
                -
                target_mask.sum()
            ),

        "availability_fraction":
            float(
                target_mask.mean()
            ),

        "dimension_diagnostics":
            dimension_diagnostics,
    }


    BLOCK_7_DERIVED_ACTIVE_DIMENSION_INDICES[
        pathway_name
    ] = tuple(
        active_indices
    )


    BLOCK_7_DERIVED_DEFERRED_DIMENSION_INDICES[
        pathway_name
    ] = tuple(
        deferred_indices
    )


# =============================================================================
# Preserve validated upstream representation-training boundaries
# =============================================================================
#
# A transformative dimension can only be eligible where the recovered
# upstream supervision supplies at least one observed element for that
# dimension. Block 7 therefore derives allowed indices directly from the
# persisted target masks rather than recreating earlier pilot active sets.
# =============================================================================

BLOCK_7_UPSTREAM_ALLOWED_INDICES = {
    pathway_name:
        tuple(
            dimension_index

            for dimension_index
            in range(
                BLOCK_7_PATHWAY_DIMS[
                    pathway_name
                ]
            )

            if bool(
                BLOCK_7_CURRENT_SUPERVISION_MASKS[
                    pathway_name
                ][
                    :,
                    dimension_index
                ].any()
            )
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


BLOCK_7_ACTIVE_DIMENSION_INDICES = {}

BLOCK_7_DEFERRED_DIMENSION_INDICES = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_7_PATHWAY_DIMS.items():

    derived_active = set(
        BLOCK_7_DERIVED_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    upstream_allowed = set(
        BLOCK_7_UPSTREAM_ALLOWED_INDICES[
            pathway_name
        ]
    )


    final_active = tuple(
        dimension_index

        for dimension_index
        in range(
            pathway_dim
        )

        if (
            dimension_index
            in derived_active
            and
            dimension_index
            in upstream_allowed
        )
    )


    final_deferred = tuple(
        dimension_index

        for dimension_index
        in range(
            pathway_dim
        )

        if dimension_index
        not in final_active
    )


    BLOCK_7_ACTIVE_DIMENSION_INDICES[
        pathway_name
    ] = final_active


    BLOCK_7_DEFERRED_DIMENSION_INDICES[
        pathway_name
    ] = final_deferred


# =============================================================================
# Training-readiness contract
# =============================================================================

BLOCK_7_PATHWAY_TRAINING_READY = {
    pathway_name:
        all(
            [
                (
                    BLOCK_7_TARGET_DIAGNOSTICS[
                        pathway_name
                    ][
                        "available_elements"
                    ]
                    >
                    0
                ),

                (
                    len(
                        BLOCK_7_ACTIVE_DIMENSION_INDICES[
                            pathway_name
                        ]
                    )
                    >
                    0
                ),
            ]
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS = tuple(
    pathway_name

    for (
        pathway_name,
        ready,
    ) in BLOCK_7_PATHWAY_TRAINING_READY.items()

    if ready
)


BLOCK_7_AT_LEAST_ONE_PATHWAY_TRAINABLE = (
    len(
        NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS
    )
    >
    0
)


if not BLOCK_7_AT_LEAST_ONE_PATHWAY_TRAINABLE:

    raise RuntimeError(
        "No transformative pathway has a validated training signal."
    )


# =============================================================================
# Convert validated contract to runtime tensors
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_CURRENT_SUPERVISION = {
    pathway_name:
        torch.tensor(
            BLOCK_7_CURRENT_SUPERVISION[
                pathway_name
            ],
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_CURRENT_SUPERVISION_MASKS = {
    pathway_name:
        torch.tensor(
            BLOCK_7_CURRENT_SUPERVISION_MASKS[
                pathway_name
            ],
            dtype=
                torch.bool,
            device=
                DEVICE,
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITIONS = {
    pathway_name:
        torch.tensor(
            BLOCK_7_PRECEDING_CONDITIONS[
                pathway_name
            ],
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITION_MASKS = {
    pathway_name:
        torch.tensor(
            BLOCK_7_PRECEDING_CONDITION_MASKS[
                pathway_name
            ],
            dtype=
                torch.bool,
            device=
                DEVICE,
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_TARGETS = {
    pathway_name:
        torch.tensor(
            BLOCK_7_TRANSFORMATIVE_TARGETS[
                pathway_name
            ],
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_TARGET_MASKS = {
    pathway_name:
        torch.tensor(
            BLOCK_7_TRANSFORMATIVE_MASKS[
                pathway_name
            ],
            dtype=
                torch.bool,
            device=
                DEVICE,
        )

    for pathway_name
    in BLOCK_7_PATHWAY_DIMS
}


NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES = (
    deepcopy(
        BLOCK_7_ACTIVE_DIMENSION_INDICES
    )
)


NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES = (
    deepcopy(
        BLOCK_7_DEFERRED_DIMENSION_INDICES
    )
)


# =============================================================================
# Model-state boundary
# =============================================================================

BLOCK_7_REPRESENTATION_MODULES = {
    "backbone":
        NOTEBOOK_08_RESTORED_BACKBONE,

    "global_confluent":
        NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,

    "psychological_head":
        NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,

    "factual_head":
        NOTEBOOK_08_RESTORED_FACTUAL_HEAD,

    "social_head":
        NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
}


BLOCK_7_TRANSFORMATIVE_MODULES = {
    "factual":
        NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM,

    "psychological":
        NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,

    "social":
        NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM,
}


BLOCK_7_REPRESENTATION_MODULES_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_7_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_TRANSFORMATIVE_MODULES_TRAINABLE = all(
    parameter.requires_grad

    for module
    in BLOCK_7_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


if not all(
    [
        BLOCK_7_REPRESENTATION_MODULES_FROZEN,
        BLOCK_7_TRANSFORMATIVE_MODULES_TRAINABLE,
    ]
):

    raise RuntimeError(
        "Notebook 08 Block 7 parameter-freezing boundary is invalid."
    )


# =============================================================================
# Transformative target contract
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT = {
    "contract_version":
        NOTEBOOK_08_BLOCK_7_VERSION,

    "created_at_utc":
        BLOCK_7_EXECUTED_AT_UTC,

    "source_type":
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_SOURCE_TYPE,

    "direct_transformation_supervision_claimed":
        False,

    "article_count":
        BLOCK_7_ARTICLE_COUNT,

    "sentence_count":
        BLOCK_7_SENTENCE_COUNT,

    "sentence_ids":
        list(
            BLOCK_7_SENTENCE_IDS
        ),

    "article_ids":
        list(
            BLOCK_7_CANONICAL_ARTICLE_IDS
        ),

    "article_ids_by_sentence":
        list(
            BLOCK_7_ARTICLE_IDS_BY_SENTENCE
        ),

    "article_start_indices":
        list(
            BLOCK_7_ARTICLE_START_INDICES
        ),

    "article_boundary_reset":
        True,

    "cross_article_state_carry":
        False,

    "target_definition":
        (
            "current_supervised_state_minus_"
            "previous_supervised_state"
        ),

    "initial_condition":
        NOTEBOOK_08_TRANSFORMATIVE_INITIAL_STATE_POLICY,

    "preceding_condition":
        NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITION_POLICY,

    "mask_rule":
        "current_mask_and_preceding_condition_mask",

    "signed_targets":
        True,

    "missing_equals_zero":
        False,

    "missing_equals_negative":
        False,

    "future_context_allowed":
        False,

    "recurrent_state_interpretation":
        False,

    "pathway_dimensions":
        deepcopy(
            BLOCK_7_PATHWAY_DIMS
        ),

    "derived_active_dimension_indices":
        deepcopy(
            BLOCK_7_DERIVED_ACTIVE_DIMENSION_INDICES
        ),

    "upstream_allowed_indices":
        deepcopy(
            BLOCK_7_UPSTREAM_ALLOWED_INDICES
        ),

    "active_dimension_indices":
        deepcopy(
            BLOCK_7_ACTIVE_DIMENSION_INDICES
        ),

    "deferred_dimension_indices":
        deepcopy(
            BLOCK_7_DEFERRED_DIMENSION_INDICES
        ),

    "diagnostics":
        deepcopy(
            BLOCK_7_TARGET_DIAGNOSTICS
        ),

    "source_references":
        {
            "factual":
                {
                    "filename":
                        BLOCK_7_FACTUAL_TARGET_FILENAME,

                    "drive_file_id":
                        BLOCK_7_FACTUAL_TARGET_DRIVE_FILE_ID,

                    "target_path":
                        "records[*].factual_target",

                    "mask_path":
                        "records[*].factual_target_mask",
                },

            "psychological":
                {
                    "drive_file_id":
                        BLOCK_7_NOTEBOOK_03_ACCEPTED_FILE_ID,

                    "family_order":
                        list(
                            BLOCK_7_PSYCHOLOGICAL_FAMILY_ORDER
                        ),

                    "emotion_dimension_order":
                        list(
                            BLOCK_7_EMOTION_DIMENSION_ORDER
                        ),

                    "REV":
                        (
                            "records[*].annotations."
                            "reported_emotion"
                        ),

                    "TEV":
                        (
                            "records[*].annotations."
                            "transformative_emotion"
                        ),

                    "IEV":
                        (
                            "records[*].annotations."
                            "immediate_emotional_state"
                        ),

                    "EW":
                        (
                            "records[*].annotations."
                            "emotional_weight"
                        ),
                },

            "social":
                {
                    "filename":
                        BLOCK_7_SOCIAL_ACCEPTED_FILENAME,

                    "drive_file_id":
                        BLOCK_7_SOCIAL_ACCEPTED_DRIVE_FILE_ID,

                    "target_path":
                        (
                            "records[*].social_annotations"
                            "[*].value"
                        ),

                    "mask_path":
                        (
                            "records[*].social_annotations"
                            "[*].available"
                        ),
                },
        },
}


# =============================================================================
# Final target-contract validation
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID = all(
    [
        BLOCK_7_IDENTITY_VALID,
        BLOCK_7_ARTICLE_BOUNDARY_IDENTITY_VALID,
        BLOCK_7_ARTICLE_BOUNDARY_COUNT_VALID,
        BLOCK_7_PATHWAY_DIMENSIONS_VALID,

        # ---------------------------------------------------------------------
        # Factual
        # ---------------------------------------------------------------------
        BLOCK_7_FACTUAL_CONTRACT_VALID,
        BLOCK_7_FACTUAL_CANONICAL_ALIGNMENT_VALID,
        BLOCK_7_FACTUAL_SHAPES_VALID,
        BLOCK_7_FACTUAL_OBSERVED_VALUES_FINITE,

        # ---------------------------------------------------------------------
        # Psychological
        # ---------------------------------------------------------------------
        BLOCK_7_PSYCHOLOGICAL_RECORD_COUNT_VALID,
        BLOCK_7_PSYCHOLOGICAL_CANONICAL_ALIGNMENT_VALID,
        BLOCK_7_PSYCHOLOGICAL_SUPERVISION_COUNTS_VALID,
        BLOCK_7_PSYCHOLOGICAL_SHAPES_VALID,
        BLOCK_7_PSYCHOLOGICAL_OBSERVED_VALUES_FINITE,

        # ---------------------------------------------------------------------
        # Social
        # ---------------------------------------------------------------------
        BLOCK_7_SOCIAL_ACCEPTED_CONTRACT_VALID,
        BLOCK_7_SOCIAL_CANONICAL_ALIGNMENT_VALID,
        BLOCK_7_SOCIAL_MASKED_VALUES_NAN,
        BLOCK_7_SOCIAL_OBSERVED_VALUES_FINITE,

        # ---------------------------------------------------------------------
        # Common target construction
        # ---------------------------------------------------------------------
        BLOCK_7_ALL_CURRENT_SUPERVISION_SHAPES_VALID,
        BLOCK_7_CAUSAL_CONSTRUCTION_VALID,
        BLOCK_7_ALL_TARGET_SHAPES_VALID,
        BLOCK_7_ALL_AVAILABLE_TARGETS_FINITE,
        BLOCK_7_AT_LEAST_ONE_PATHWAY_TRAINABLE,

        # ---------------------------------------------------------------------
        # Model boundary
        # ---------------------------------------------------------------------
        BLOCK_7_REPRESENTATION_MODULES_FROZEN,
        BLOCK_7_TRANSFORMATIVE_MODULES_TRAINABLE,

        # ---------------------------------------------------------------------
        # Methodological boundary
        # ---------------------------------------------------------------------
        not NOTEBOOK_08_TRANSFORMATIVE_DIRECT_SUPERVISION_CLAIMED,
        not NOTEBOOK_08_TRANSFORMATIVE_FUTURE_CONTEXT_ALLOWED,
    ]
)


NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED = (
    NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID
)


NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVES_ACTIVATED = False


NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED,
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED,
        BLOCK_7_AT_LEAST_ONE_PATHWAY_TRAINABLE,
    ]
)


NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY = (
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY
)


# =============================================================================
# Final Block 7 validation
# =============================================================================

NOTEBOOK_08_BLOCK_7_VALID = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY,

        not NOTEBOOK_08_BLOCK_7_MODEL_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_TRANSFORM_FORWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_LOSS_CALCULATED,
        not NOTEBOOK_08_BLOCK_7_OPTIMIZER_CREATED,
        not NOTEBOOK_08_BLOCK_7_ZERO_GRAD_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_BACKWARD_PASS_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_GRADIENT_CLIPPING_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_OPTIMIZER_STEP_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_PARAMETER_UPDATE_EXECUTED,
        not NOTEBOOK_08_BLOCK_7_TRANSFORMATION_WEIGHTS_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_7_RECURRENT_STATE_INSTANTIATED,
        not NOTEBOOK_08_BLOCK_7_RECURRENT_UPDATE_EXECUTED,
    ]
)


if not NOTEBOOK_08_BLOCK_7_VALID:

    raise RuntimeError(
        "Notebook 08 Block 7 transformative target "
        "and mask-contract validation failed."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_7_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_08_BLOCK_7_SUMMARY = {
    "block":
        NOTEBOOK_08_BLOCK_7,

    "block_name":
        NOTEBOOK_08_BLOCK_7_NAME,

    "version":
        NOTEBOOK_08_BLOCK_7_VERSION,

    "article_count":
        BLOCK_7_ARTICLE_COUNT,

    "sentence_count":
        BLOCK_7_SENTENCE_COUNT,

    "article_ids":
        list(
            BLOCK_7_CANONICAL_ARTICLE_IDS
        ),

    "article_start_indices":
        list(
            BLOCK_7_ARTICLE_START_INDICES
        ),

    "article_boundary_reset":
        True,

    "cross_article_state_carry":
        BLOCK_7_CROSS_ARTICLE_STATE_CARRY,

    "target_source_type":
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_SOURCE_TYPE,

    "direct_transformation_supervision":
        False,

    "causal_construction":
        BLOCK_7_CAUSAL_CONSTRUCTION_VALID,

    "future_context_allowed":
        False,

    "pathway_training_ready":
        deepcopy(
            BLOCK_7_PATHWAY_TRAINING_READY
        ),

    "active_dimension_indices":
        deepcopy(
            BLOCK_7_ACTIVE_DIMENSION_INDICES
        ),

    "deferred_dimension_indices":
        deepcopy(
            BLOCK_7_DEFERRED_DIMENSION_INDICES
        ),

    "target_contract_valid":
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID,

    "training_ready":
        NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY,

    "block_valid":
        NOTEBOOK_08_BLOCK_7_VALID,

    "block_complete":
        NOTEBOOK_08_BLOCK_7_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 7: "
    "Transformative Target Construction and Mask Contract"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_7_VERSION}"
)

print("-" * 72)

print(
    "Canonical sequence"
)

print(
    f"Articles                    : "
    f"{BLOCK_7_ARTICLE_COUNT}"
)

print(
    f"Sentences                   : "
    f"{BLOCK_7_SENTENCE_COUNT}"
)

print(
    f"Sentence IDs valid          : "
    f"{BLOCK_7_IDENTITY_VALID}"
)


print(
    f"Article IDs                 : "
    f"{list(BLOCK_7_CANONICAL_ARTICLE_IDS)}"
)

print(
    f"Article start indices       : "
    f"{list(BLOCK_7_ARTICLE_START_INDICES)}"
)

print(
    f"Article-boundary reset      : "
    f"{BLOCK_7_ARTICLE_BOUNDARY_COUNT_VALID}"
)

print(
    f"Initial-state policy        : "
    f"zero architectural condition"
)

print(
    f"Preceding-state policy      : "
    f"previous supervised state"
)

print(
    f"Future context used         : "
    f"{BLOCK_7_FUTURE_INFORMATION_USED}"
)

print(
    f"Cross-article state carry   : "
    f"{BLOCK_7_CROSS_ARTICLE_STATE_CARRY}"
)

print("-" * 72)

print(
    "Recovered supervision"
)


print(
    f"Factual persisted canonical : "
    f"{BLOCK_7_FACTUAL_PERSISTED_ORDER_CANONICAL}"
)

print(
    f"Psych. persisted canonical  : "
    f"{BLOCK_7_PSYCHOLOGICAL_PERSISTED_ORDER_CANONICAL}"
)

print(
    f"Social persisted canonical  : "
    f"{BLOCK_7_SOCIAL_PERSISTED_ORDER_CANONICAL}"
)

print(
    "Local canonical reorder     : "
    f"{all([BLOCK_7_FACTUAL_CANONICAL_ORDER_VALID, BLOCK_7_PSYCHOLOGICAL_CANONICAL_ORDER_VALID, BLOCK_7_SOCIAL_CANONICAL_ORDER_VALID])}"
)

print(
    f"Factual source              : "
    f"{BLOCK_7_FACTUAL_TARGET_FILENAME}"
)

print(
    f"Factual shape               : "
    f"{BLOCK_7_FACTUAL_CURRENT_SUPERVISION.shape}"
)

print(
    f"Psychological source        : "
    f"Notebook 03 accepted supervision"
)

print(
    f"Psychological shape         : "
    f"{BLOCK_7_PSYCHOLOGICAL_CURRENT_SUPERVISION.shape}"
)

print(
    f"Social source               : "
    f"{BLOCK_7_SOCIAL_ACCEPTED_FILENAME}"
)

print(
    f"Social shape                : "
    f"{BLOCK_7_SOCIAL_CURRENT_SUPERVISION.shape}"
)

print("-" * 72)

print(
    "Psychological supervision families"
)

for family_name in (
    BLOCK_7_PSYCHOLOGICAL_FAMILY_ORDER
):

    family = (
        BLOCK_7_PSYCHOLOGICAL_FAMILIES[
            family_name
        ]
    )


    print(
        f"{family_name:<4} observed / possible      : "
        f"{family['observed_elements']} / "
        f"{family['possible_elements']}"
    )

print(
    f"Psychological counts valid  : "
    f"{BLOCK_7_PSYCHOLOGICAL_SUPERVISION_COUNTS_VALID}"
)

print("-" * 72)

print(
    "Derived transformative targets"
)

for pathway_name in (
    BLOCK_7_PATHWAY_DIMS
):

    diagnostics = (
        BLOCK_7_TARGET_DIAGNOSTICS[
            pathway_name
        ]
    )


    print(
        f"{pathway_name:<14} dimension          : "
        f"{BLOCK_7_PATHWAY_DIMS[pathway_name]}"
    )


    print(
        f"{pathway_name:<14} available elements : "
        f"{diagnostics['available_elements']}"
    )


    print(
        f"{pathway_name:<14} masked elements    : "
        f"{diagnostics['masked_elements']}"
    )


    print(
        f"{pathway_name:<14} availability       : "
        f"{diagnostics['availability_fraction']:.4f}"
    )


    print(
        f"{pathway_name:<14} derived active     : "
        f"{list(BLOCK_7_DERIVED_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )


    print(
        f"{pathway_name:<14} final active       : "
        f"{list(BLOCK_7_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )


    print(
        f"{pathway_name:<14} final deferred     : "
        f"{list(BLOCK_7_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

print("-" * 72)

print(
    f"Target shapes valid         : "
    f"{BLOCK_7_ALL_TARGET_SHAPES_VALID}"
)

print(
    f"Available targets finite    : "
    f"{BLOCK_7_ALL_AVAILABLE_TARGETS_FINITE}"
)

print(
    f"Causal construction valid   : "
    f"{BLOCK_7_CAUSAL_CONSTRUCTION_VALID}"
)

print(
    f"Eligible pathways           : "
    f"{list(NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS)}"
)

print(
    f"At least one trainable      : "
    f"{BLOCK_7_AT_LEAST_ONE_PATHWAY_TRAINABLE}"
)

print("-" * 72)

print(
    f"Representation frozen       : "
    f"{BLOCK_7_REPRESENTATION_MODULES_FROZEN}"
)

print(
    f"Transform modules trainable : "
    f"{BLOCK_7_TRANSFORMATIVE_MODULES_TRAINABLE}"
)

print("-" * 72)

print(
    f"Forward pass executed       : "
    f"{NOTEBOOK_08_BLOCK_7_TRANSFORM_FORWARD_PASS_EXECUTED}"
)

print(
    f"Loss calculated             : "
    f"{NOTEBOOK_08_BLOCK_7_LOSS_CALCULATED}"
)

print(
    f"Optimizer created           : "
    f"{NOTEBOOK_08_BLOCK_7_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed      : "
    f"{NOTEBOOK_08_BLOCK_7_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed   : "
    f"{NOTEBOOK_08_BLOCK_7_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Transform weights created   : "
    f"{NOTEBOOK_08_BLOCK_7_TRANSFORMATION_WEIGHTS_INSTANTIATED}"
)

print(
    f"Recurrent state created     : "
    f"{NOTEBOOK_08_BLOCK_7_RECURRENT_STATE_INSTANTIATED}"
)

print("-" * 72)

print(
    f"Target contract valid       : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID}"
)

print(
    f"Transform training ready    : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_7_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_7_COMPLETE}"
)

print("=" * 72)

print(
    "Validated factual, psychological and social representation "
    "supervision was recovered from the persisted upstream artefacts."
)

print(
    "Factual supervision was reconstructed from the Notebook 06 "
    "factual target and mask record contract."
)

print(
    "Psychological REV, TEV, IEV and EW supervision was reconstructed "
    "directly from the canonical accepted Notebook 03 sentence records."
)

print(
    "Observed psychological zeros remain genuine supervision; "
    "insufficient-evidence values remain masked."
)

print(
    "TEV remains part of the psychological representation space and "
    "is not interpreted as the architectural psychological transformation."
)

print(
    "EW remains psychological supervision and is not interpreted "
    "as Transformative Weight TW_P."
)

print(
    "Social supervision was reconstructed directly from the accepted "
    "Notebook 07 social annotation records."
)

print(
    "Architectural transformation targets were derived as signed "
    "current-state minus preceding-state differences."
)

print(
    "The first sentence of every article uses a deterministic zero "
    "architectural initial condition; no state is carried across article "
    "boundaries and no future information is used."
)

print(
    "Missing supervision remains distinct from a genuine zero "
    "transformation."
)

print(
    "No Transformative Weight, recurrent state, loss, optimiser, "
    "backward pass or parameter update was introduced."
)

print(
    "Notebook 08 may proceed to objective activation and controlled "
    "transformative training for pathways that passed target eligibility."
)

print("=" * 72)

Media AI — Notebook 08, Block 7: Transformative Target Construction and Mask Contract
Block version               : 1.7
------------------------------------------------------------------------
Canonical sequence
Articles                    : 3
Sentences                   : 54
Sentence IDs valid          : True
Article IDs                 : ['1ef9745c-2c21-47c2-a222-e7c160448f82', '33270083-262f-44e9-9a90-ac857f026e72', 'bca49d4d-ec96-4fee-a2e0-cc1e98dc9fcb']
Article start indices       : [0, 16, 31]
Article-boundary reset      : True
Initial-state policy        : zero architectural condition
Preceding-state policy      : previous supervised state
Future context used         : False
Cross-article state carry   : False
------------------------------------------------------------------------
Recovered supervision
Factual persisted canonical : False
Psych. persisted canonical  : False
Social persisted canonical  : False
Local canonical reorder     : True
Factual source              : noteb

## Block 8 — Controlled Transformative-Mechanism Training and Parameter-Update Validation

This block performs the first controlled optimisation of the **transformative mechanisms** defined and validated in the preceding blocks.

The purpose of the block is deliberately narrow. The inherited representation model remains fixed, while the factual, psychological and social transformative mechanisms are trained to approximate the **derived signed transformation targets** constructed in Block 7.

No Transformative Weight and no recurrent-state update mechanism are introduced here. Those concepts remain outside the scope of Notebook 08 and are reserved for Notebook 09.

---

## Training objective

For each representation pathway, the transformative mechanism receives two same-dimensional inputs:

1. the current representation state; and
2. the causally available preceding state.

For pathway \(p\) and sequence position \(t\), the transformative mechanism therefore computes

$$
\hat{\Delta}^{(p)}_t
=
T^{(p)}_{\theta_p}
\left(
r^{(p)}_t,
h^{(p)}_{t-1}
\right),
$$

where:

- \(r^{(p)}_t\) is the current pathway representation;
- \(h^{(p)}_{t-1}\) is the preceding-state condition available at position \(t\);
- \(T^{(p)}_{\theta_p}\) is the pathway-specific transformative mechanism;
- \(\theta_p\) denotes the trainable parameters of that mechanism; and
- \(\hat{\Delta}^{(p)}_t\) is the predicted candidate transformation.

The three pathway dimensions remain:

- factual: \(d_F = 10\);
- psychological: \(d_P = 34\);
- social: \(d_S = 7\).

The mechanisms remain structurally independent and therefore learn separate factual, psychological and social transformation functions.

---

## Derived transformation supervision

Notebook 08 does not claim access to independently annotated architectural transformation labels.

Instead, Block 7 constructed a controlled training target from the validated representation supervision already inherited from the factual, psychological and social pathways.

For an available dimension \(j\), the signed target is

$$
\Delta^{(p)}_{t,j}
=
r^{(p)}_{t,j}
-
h^{(p)}_{t-1,j}.
$$

The target therefore represents the supervised change between the preceding condition and the current supervised representation state.

A positive value represents an increase along the corresponding representation dimension, a negative value represents a decrease, and zero represents no observed change where both states are available.

These values are **derived transformation supervision**. They are not direct annotations of an underlying causal transformation process.

For the first sequence position, the deterministic zero vector introduced in Block 7 remains the architectural initial condition. It is not interpreted as an observed pre-article representation state.

---

## Mask-aware optimisation

Transformation loss is calculated only where both the current and preceding supervision required to construct the target are available.

For pathway \(p\), dimension \(j\), and sequence position \(t\), the transformation mask is

$$
m^{(p)}_{t,j}
=
m^{(p)}_{\mathrm{current},t,j}
\land
m^{(p)}_{\mathrm{preceding},t,j}.
$$

Unavailable supervision therefore does not enter the optimisation objective.

In particular:

- missing supervision is not converted into zero;
- missing supervision is not interpreted as negative evidence;
- an unavailable transformation is not treated as a genuine zero transformation.

For the mask-valid elements of pathway \(p\), the pathway objective follows the mask-aware mean-squared-error contract established in Block 6:

$$
\mathcal{L}_p
=
\frac{
\sum_{t,j}
m^{(p)}_{t,j}
\left(
\hat{\Delta}^{(p)}_{t,j}
-
\Delta^{(p)}_{t,j}
\right)^2
}{
\sum_{t,j}m^{(p)}_{t,j}
}.
$$

The complete optimisation objective combines the three pathway losses using the equal optimisation coefficients established previously:

$$
\mathcal{L}_{\mathrm{transform}}
=
\frac{1}{3}\mathcal{L}_F
+
\frac{1}{3}\mathcal{L}_P
+
\frac{1}{3}\mathcal{L}_S.
$$

These coefficients are **optimisation-loss weights only**. They are not the Media AI Transformative Weights \(TW_F\), \(TW_P\), or \(TW_S\).

---

## Dimension-level training eligibility

Block 7 also established which derived target dimensions contain sufficient validated supervision and variation to participate in optimisation.

Training in this block must therefore respect the inherited and derived activation contract.

A dimension may contribute to the loss only when it is:

- permitted by the validated upstream representation-training boundary;
- supported by mask-valid derived transformation targets;
- sufficiently observed for controlled optimisation; and
- non-degenerate under the Block 7 target-variability checks.

Deferred dimensions remain part of their representation spaces but do not contribute artificial supervision to the training objective.

This distinction preserves the semantic architecture without converting unavailable or structurally inactive dimensions into false training signals.

---

## Parameter-freezing boundary

The complete inherited representation model remains frozen throughout this block:

$$
\theta_{\mathrm{representation}}
=
\text{constant}.
$$

This includes the inherited:

- backbone;
- global confluent layer;
- psychological representation head;
- factual representation head; and
- social representation head.

Together these components contain **894,003 parameters**, none of which is eligible for optimisation in Notebook 08.

Only the three transformative mechanisms are trainable:

$$
\theta_{\mathrm{trainable}}
=
\theta_F
\cup
\theta_P
\cup
\theta_S.
$$

Their parameter counts remain:

- factual transformative mechanism: 670 parameters;
- psychological transformative mechanism: 7,174 parameters;
- social transformative mechanism: 343 parameters.

The total optimisation scope is therefore

$$
|\theta_{\mathrm{trainable}}|
=
8{,}187.
$$

The block must verify after training that every inherited representation parameter remains exactly unchanged.

---

## Controlled optimisation procedure

The optimisation policy established in Block 6 is now activated for the first time.

Training uses:

- AdamW optimisation;
- learning rate \(0.001\);
- weight decay \(0.0001\);
- maximum gradient norm \(1.0\);
- 10 controlled epochs;
- mask-aware pathway objectives; and
- preserved sequential ordering.

No random sequence shuffling is introduced.

For every optimisation step, the block follows the controlled sequence

$$
\text{forward}
\rightarrow
\text{masked loss}
\rightarrow
\text{backward}
\rightarrow
\text{gradient clipping}
\rightarrow
\text{optimizer step}.
$$

Only the 8,187 transformative parameters may receive parameter updates.

The block records the initial objective, epoch-level objective history, final objective, gradient behaviour and maximum parameter displacement so that the optimisation can be audited rather than inferred from successful execution alone.

---

## Training validation

Successful execution alone is not sufficient to establish a valid transformative training stage.

The block must verify that:

- the expected transformative parameters were the only trainable parameters;
- all three pathway mechanisms participated according to their active target contracts;
- calculated losses were finite;
- gradients were finite;
- the configured gradient-norm boundary was applied;
- at least one eligible transformative parameter changed;
- no inherited representation parameter changed;
- the optimisation history remained finite;
- the trained modules remain dimensionally compatible with their respective pathways; and
- the resulting transformative state can be reproduced from the retained trained parameters.

The block therefore distinguishes **successful optimisation** from merely executing an optimiser.

---

## Architectural boundary

The output of each trained mechanism remains a **candidate transformation**:

$$
\hat{\Delta}^{(p)}_t.
$$

It is not yet a recurrent state update.

In particular, this block does **not** calculate an expression of the form

$$
h^{(p)}_t
=
h^{(p)}_{t-1}
+
TW_p \cdot \hat{\Delta}^{(p)}_t.
$$

Such an operation would require a defined Transformative Weight and a recurrent-state transition contract. Neither belongs to the training stage implemented in Notebook 08.

Accordingly, this block does not create:

- \(TW_F\);
- \(TW_P\);
- \(TW_S\);
- a persistent recurrent factual state;
- a persistent recurrent psychological state;
- a persistent recurrent social state; or
- a recurrent transformation-update rule.

These remain reserved for Notebook 09.

The psychological TEV supervision inherited from Notebook 05 likewise remains part of the psychological representation target space; it is not reinterpreted as the architectural psychological transformation itself. Similarly, EW remains psychological supervision and is not reinterpreted as \(TW_P\).

---

## Expected outcome

At completion, Notebook 08 should contain three trained and validated transformative mechanisms capable of mapping a current pathway representation and a causally available preceding condition to a same-dimensional **candidate transformation**.

The block should establish that:

1. the derived transformation-target contract can support controlled optimisation;
2. only the intended transformative parameters were updated;
3. the inherited representation architecture remained exactly frozen;
4. factual, psychological and social transformative pathways remained independent;
5. missing supervision did not enter the loss;
6. no future information entered the training construction; and
7. no Transformative Weight or recurrent-state mechanism was introduced.

This establishes the trained transformative mechanisms required for the final Notebook 08 validation and persistence stage while preserving a strict architectural boundary between **representation**, **candidate transformation**, **Transformative Weight**, and **recurrent state**.

In [13]:
# =============================================================================
# Media AI — Notebook 08, Block 8
# Controlled Transformative-Mechanism Training and Parameter-Update Validation
# =============================================================================

from collections import OrderedDict
from copy import deepcopy

import numpy as np
import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_8 = 8

NOTEBOOK_08_BLOCK_8_NAME = (
    "Controlled Transformative-Mechanism Training "
    "and Parameter-Update Validation"
)

NOTEBOOK_08_BLOCK_8_VERSION = "1.1"


# =============================================================================
# Dependency checks
# =============================================================================

required_block_8_objects = [
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Block 3 — restored representation model
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_RESTORED_BACKBONE",
    "NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT",
    "NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD",
    "NOTEBOOK_08_RESTORED_FACTUAL_HEAD",
    "NOTEBOOK_08_RESTORED_SOCIAL_HEAD",

    # -------------------------------------------------------------------------
    # Block 4 — transformative architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM",

    # -------------------------------------------------------------------------
    # Block 6 — optimisation policy
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_6_COMPLETE",
    "NOTEBOOK_08_BLOCK_6_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED",
    "NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY",
    "NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY",

    # -------------------------------------------------------------------------
    # Block 7 — target contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_7_COMPLETE",
    "NOTEBOOK_08_BLOCK_7_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED",
    "NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS",
    "NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY",
    "NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY",

    "NOTEBOOK_08_TRANSFORMATIVE_CURRENT_SUPERVISION",
    "NOTEBOOK_08_TRANSFORMATIVE_CURRENT_SUPERVISION_MASKS",

    "NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITIONS",
    "NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITION_MASKS",

    "NOTEBOOK_08_TRANSFORMATIVE_TARGETS",
    "NOTEBOOK_08_TRANSFORMATIVE_TARGET_MASKS",

    "NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Dimensional contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_FACTUAL_TRANSFORM_DIM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM",
    "NOTEBOOK_08_SOCIAL_TRANSFORM_DIM",
]


missing_block_8_objects = [
    object_name
    for object_name
    in required_block_8_objects
    if object_name not in globals()
]


if missing_block_8_objects:

    raise NameError(
        "Notebook 08 Block 8 prerequisites are not initialised. "
        f"Missing: {missing_block_8_objects}"
    )


if not all(
    [
        NOTEBOOK_08_BLOCK_6_COMPLETE,
        NOTEBOOK_08_BLOCK_6_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_POLICY_DEFINED,

        NOTEBOOK_08_BLOCK_7_COMPLETE,
        NOTEBOOK_08_BLOCK_7_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALIDATED,
        NOTEBOOK_08_TRANSFORMATIVE_OPTIMISATION_READY,
        NOTEBOOK_08_TRANSFORMATIVE_TRAINING_READY,
    ]
):

    raise RuntimeError(
        "Notebook 08 Blocks 6 and 7 must be complete and valid "
        "before controlled transformative training."
    )


# =============================================================================
# Deterministic training contract
# =============================================================================

torch.manual_seed(
    SEED
)

np.random.seed(
    SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# =============================================================================
# Controlled optimisation policy
# =============================================================================
#
# Block 8 consumes the validated Block 6 optimisation policy rather than
# redefining learning-rate, weight-decay, gradient, epoch or sequence values.
# =============================================================================

BLOCK_8_OPTIMIZER_NAME = str(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "optimizer"
    ]
)


BLOCK_8_LEARNING_RATE = float(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "learning_rate"
    ]
)


BLOCK_8_WEIGHT_DECAY = float(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "weight_decay"
    ]
)


BLOCK_8_MAXIMUM_GRADIENT_NORM = float(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "maximum_gradient_norm"
    ]
)


BLOCK_8_EPOCHS = int(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "epochs"
    ]
)


BLOCK_8_SEQUENCE_ORDER_PRESERVED = bool(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "sequence_order_preserved"
    ]
)


BLOCK_8_FUTURE_CONTEXT_ALLOWED = bool(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "future_context_allowed"
    ]
)


BLOCK_8_BATCH_SHUFFLING_USED = bool(
    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER_POLICY[
        "shuffle_within_sequence"
    ]
)


BLOCK_8_ELIGIBLE_PATHWAYS = tuple(
    NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS
)


if not BLOCK_8_ELIGIBLE_PATHWAYS:

    raise RuntimeError(
        "No transformative pathways are eligible for controlled training."
    )


BLOCK_8_PATHWAY_LOSS_WEIGHTS = OrderedDict(
    (
        pathway_name,
        (
            1.0
            /
            len(
                BLOCK_8_ELIGIBLE_PATHWAYS
            )
        ),
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


BLOCK_8_LOSS_WEIGHT_SUM = float(
    sum(
        BLOCK_8_PATHWAY_LOSS_WEIGHTS.values()
    )
)


BLOCK_8_WEIGHT_POLICY_REPRODUCED = all(
    [
        NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY[
            "policy"
        ]
        ==
        "equal_across_eligible_pathways",

        not NOTEBOOK_08_TRANSFORMATIVE_OBJECTIVE_WEIGHT_POLICY[
            "weights_activated"
        ],

        np.isclose(
            BLOCK_8_LOSS_WEIGHT_SUM,
            1.0,
            rtol=0.0,
            atol=1e-12,
        ),
    ]
)


BLOCK_8_OPTIMISATION_POLICY_VALID = all(
    [
        BLOCK_8_OPTIMIZER_NAME
        ==
        "AdamW",

        BLOCK_8_LEARNING_RATE > 0.0,
        BLOCK_8_WEIGHT_DECAY >= 0.0,
        BLOCK_8_MAXIMUM_GRADIENT_NORM > 0.0,
        BLOCK_8_EPOCHS > 0,

        BLOCK_8_SEQUENCE_ORDER_PRESERVED,
        not BLOCK_8_BATCH_SHUFFLING_USED,
        not BLOCK_8_FUTURE_CONTEXT_ALLOWED,

        BLOCK_8_WEIGHT_POLICY_REPRODUCED,
    ]
)


if not BLOCK_8_OPTIMISATION_POLICY_VALID:

    raise RuntimeError(
        "Block 8 optimisation policy does not reproduce "
        "the validated Block 6 contract."
    )


# =============================================================================
# Canonical pathway dimensions
# =============================================================================

BLOCK_8_PATHWAY_DIMS = OrderedDict(
    [
        (
            "factual",
            int(
                NOTEBOOK_08_FACTUAL_TRANSFORM_DIM
            ),
        ),

        (
            "psychological",
            int(
                NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM
            ),
        ),

        (
            "social",
            int(
                NOTEBOOK_08_SOCIAL_TRANSFORM_DIM
            ),
        ),
    ]
)


BLOCK_8_PATHWAY_DIMENSIONS_VALID = all(
    [
        all(
            pathway_dim > 0

            for pathway_dim
            in BLOCK_8_PATHWAY_DIMS.values()
        ),

        set(
            BLOCK_8_ELIGIBLE_PATHWAYS
        ).issubset(
            set(
                BLOCK_8_PATHWAY_DIMS.keys()
            )
        ),
    ]
)


if not BLOCK_8_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Block 8 pathway dimensional contract is invalid."
    )


# =============================================================================
# Representation-model modules
# =============================================================================

BLOCK_8_REPRESENTATION_MODULES = OrderedDict(
    [
        (
            "backbone",
            NOTEBOOK_08_RESTORED_BACKBONE,
        ),

        (
            "global_confluent",
            NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,
        ),

        (
            "psychological_head",
            NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,
        ),

        (
            "factual_head",
            NOTEBOOK_08_RESTORED_FACTUAL_HEAD,
        ),

        (
            "social_head",
            NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
        ),
    ]
)


# =============================================================================
# Transformative modules
# =============================================================================

BLOCK_8_TRANSFORMATIVE_MODULES = OrderedDict(
    [
        (
            "factual",
            NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM,
        ),

        (
            "psychological",
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,
        ),

        (
            "social",
            NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM,
        ),
    ]
)


BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES = OrderedDict(
    (
        pathway_name,
        BLOCK_8_TRANSFORMATIVE_MODULES[
            pathway_name
        ],
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


# =============================================================================
# Parameter accounting
# =============================================================================

BLOCK_8_REPRESENTATION_PARAMETER_COUNT = sum(
    parameter.numel()

    for module
    in BLOCK_8_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNTS = OrderedDict(
    (
        pathway_name,
        sum(
            parameter.numel()

            for parameter
            in module.parameters()
        ),
    )

    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.items()
)


BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNT = sum(
    BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNTS.values()
)


BLOCK_8_ELIGIBLE_TRANSFORMATIVE_PARAMETER_COUNT = sum(
    BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNTS[
        pathway_name
    ]

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


BLOCK_8_PARAMETER_ACCOUNTING_VALID = all(
    [
        BLOCK_8_REPRESENTATION_PARAMETER_COUNT > 0,
        BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNT > 0,
        BLOCK_8_ELIGIBLE_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNT
        ==
        sum(
            BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNTS.values()
        ),
    ]
)


if not BLOCK_8_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Block 8 parameter accounting is invalid."
    )


# =============================================================================
# Parameter-freezing validation
# =============================================================================

BLOCK_8_REPRESENTATION_FULLY_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_8_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_TRANSFORMATIVE_FULLY_TRAINABLE = all(
    parameter.requires_grad

    for module
    in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


if not BLOCK_8_REPRESENTATION_FULLY_FROZEN:

    raise RuntimeError(
        "Inherited representation model contains trainable parameters."
    )


if not BLOCK_8_TRANSFORMATIVE_FULLY_TRAINABLE:

    raise RuntimeError(
        "One or more transformative parameters are unexpectedly frozen."
    )


# =============================================================================
# Parameter-group independence
# =============================================================================

BLOCK_8_REPRESENTATION_PARAMETER_IDS = {
    id(
        parameter
    )

    for module
    in BLOCK_8_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
}


BLOCK_8_TRANSFORMATIVE_PARAMETER_IDS = {
    id(
        parameter
    )

    for module
    in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
}


BLOCK_8_ELIGIBLE_TRANSFORMATIVE_PARAMETER_IDS = {
    id(
        parameter
    )

    for module
    in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
}


BLOCK_8_PARAMETER_GROUPS_DISJOINT = (
    BLOCK_8_REPRESENTATION_PARAMETER_IDS.isdisjoint(
        BLOCK_8_TRANSFORMATIVE_PARAMETER_IDS
    )
)


if not BLOCK_8_PARAMETER_GROUPS_DISJOINT:

    raise RuntimeError(
        "Representation and transformative parameter groups overlap."
    )


# =============================================================================
# Parameter-state snapshot helpers
# =============================================================================

def block_8_snapshot_module(
    module,
):

    return OrderedDict(
        (
            parameter_name,
            parameter.detach()
            .cpu()
            .clone(),
        )

        for (
            parameter_name,
            parameter,
        ) in module.named_parameters()
    )


def block_8_module_matches_snapshot(
    module,
    snapshot,
):

    current_parameters = OrderedDict(
        (
            parameter_name,
            parameter.detach()
            .cpu(),
        )

        for (
            parameter_name,
            parameter,
        ) in module.named_parameters()
    )


    if (
        tuple(
            current_parameters.keys()
        )
        !=
        tuple(
            snapshot.keys()
        )
    ):

        return False


    return all(
        torch.equal(
            current_parameters[
                parameter_name
            ],
            snapshot[
                parameter_name
            ],
        )

        for parameter_name
        in snapshot
    )


def block_8_maximum_parameter_delta(
    module,
    snapshot,
):

    maximum_delta = (
        0.0
    )


    for (
        parameter_name,
        parameter,
    ) in module.named_parameters():

        delta = (
            parameter.detach()
            .cpu()
            -
            snapshot[
                parameter_name
            ]
        )


        parameter_delta = float(
            delta.abs()
            .max()
            .item()
        )


        maximum_delta = max(
            maximum_delta,
            parameter_delta,
        )


    return maximum_delta


# =============================================================================
# Pre-training snapshots
# =============================================================================

BLOCK_8_REPRESENTATION_SNAPSHOT = {
    module_name:
        block_8_snapshot_module(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_8_REPRESENTATION_MODULES.items()
}


BLOCK_8_TRANSFORMATIVE_INITIAL_SNAPSHOT = {
    pathway_name:
        block_8_snapshot_module(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRANSFORMATIVE_MODULES.items()
}


# =============================================================================
# Training-input contract
# =============================================================================
#
# Block 7 provides the supervised pathway-state tensors used here as the
# controlled current-state training inputs.
#
# The preceding conditions are strictly causal:
#
#     h_(t-1)
#
# and the targets are:
#
#     Delta_t = y_t - h_(t-1)
#
# The representation network itself is not re-executed or trained in this
# block.
# =============================================================================

BLOCK_8_CURRENT_INPUTS = {
    pathway_name:
        NOTEBOOK_08_TRANSFORMATIVE_CURRENT_SUPERVISION[
            pathway_name
        ].detach()

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
}


BLOCK_8_PRECEDING_INPUTS = {
    pathway_name:
        NOTEBOOK_08_TRANSFORMATIVE_PRECEDING_CONDITIONS[
            pathway_name
        ].detach()

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
}


BLOCK_8_TARGETS = {
    pathway_name:
        NOTEBOOK_08_TRANSFORMATIVE_TARGETS[
            pathway_name
        ].detach()

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
}


BLOCK_8_BASE_TARGET_MASKS = {
    pathway_name:
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_MASKS[
            pathway_name
        ].detach()

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
}


# =============================================================================
# Active-dimension masks
# =============================================================================

BLOCK_8_ACTIVE_DIMENSION_MASKS = {}

BLOCK_8_EFFECTIVE_TARGET_MASKS = {}


for pathway_name in BLOCK_8_ELIGIBLE_PATHWAYS:

    pathway_dim = (
        BLOCK_8_PATHWAY_DIMS[
            pathway_name
        ]
    )

    active_indices = (
        NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    if not active_indices:

        raise RuntimeError(
            f"No active dimensions are available for "
            f"{pathway_name} transformative training."
        )


    active_dimension_mask = torch.zeros(
        pathway_dim,
        dtype=
            torch.bool,
        device=
            DEVICE,
    )


    active_dimension_mask[
        list(
            active_indices
        )
    ] = True


    BLOCK_8_ACTIVE_DIMENSION_MASKS[
        pathway_name
    ] = active_dimension_mask


    effective_mask = (
        BLOCK_8_BASE_TARGET_MASKS[
            pathway_name
        ]
        &
        active_dimension_mask.unsqueeze(
            0
        )
    )


    BLOCK_8_EFFECTIVE_TARGET_MASKS[
        pathway_name
    ] = effective_mask


    if int(
        effective_mask.sum()
        .item()
    ) <= 0:

        raise RuntimeError(
            f"{pathway_name} has no mask-valid active "
            "transformative target elements."
        )


# =============================================================================
# Input / target dimensional validation
# =============================================================================

BLOCK_8_TRAINING_SHAPES_VALID = {}


for pathway_name in BLOCK_8_ELIGIBLE_PATHWAYS:

    pathway_dim = (
        BLOCK_8_PATHWAY_DIMS[
            pathway_name
        ]
    )

    current_shape = tuple(
        BLOCK_8_CURRENT_INPUTS[
            pathway_name
        ].shape
    )


    preceding_shape = tuple(
        BLOCK_8_PRECEDING_INPUTS[
            pathway_name
        ].shape
    )


    target_shape = tuple(
        BLOCK_8_TARGETS[
            pathway_name
        ].shape
    )


    mask_shape = tuple(
        BLOCK_8_EFFECTIVE_TARGET_MASKS[
            pathway_name
        ].shape
    )


    expected_shape = (
        target_shape[
            0
        ],
        pathway_dim,
    )


    BLOCK_8_TRAINING_SHAPES_VALID[
        pathway_name
    ] = all(
        [
            current_shape
            ==
            expected_shape,

            preceding_shape
            ==
            expected_shape,

            target_shape
            ==
            expected_shape,

            mask_shape
            ==
            expected_shape,
        ]
    )


if not all(
    BLOCK_8_TRAINING_SHAPES_VALID.values()
):

    raise RuntimeError(
        "One or more transformative training tensor shapes are invalid."
    )


# =============================================================================
# Numerical validation
# =============================================================================

BLOCK_8_TRAINING_INPUTS_FINITE = all(
    bool(
        torch.isfinite(
            BLOCK_8_CURRENT_INPUTS[
                pathway_name
            ]
        ).all()
        .item()
    )
    and
    bool(
        torch.isfinite(
            BLOCK_8_PRECEDING_INPUTS[
                pathway_name
            ]
        ).all()
        .item()
    )
    and
    bool(
        torch.isfinite(
            BLOCK_8_TARGETS[
                pathway_name
            ][
                BLOCK_8_EFFECTIVE_TARGET_MASKS[
                    pathway_name
                ]
            ]
        ).all()
        .item()
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


if not BLOCK_8_TRAINING_INPUTS_FINITE:

    raise RuntimeError(
        "Transformative training inputs or active targets "
        "contain non-finite values."
    )


# =============================================================================
# Mask-aware objective
# =============================================================================

def block_8_masked_mean_squared_error(
    prediction,
    target,
    mask,
):

    if prediction.shape != target.shape:

        raise RuntimeError(
            "Prediction and target shapes differ. "
            f"Prediction={tuple(prediction.shape)}, "
            f"target={tuple(target.shape)}."
        )


    if target.shape != mask.shape:

        raise RuntimeError(
            "Target and mask shapes differ."
        )


    observed_elements = int(
        mask.sum()
        .item()
    )


    if observed_elements <= 0:

        raise RuntimeError(
            "Masked transformative objective contains "
            "no observed elements."
        )


    squared_error = (
        prediction
        -
        target
    ) ** 2


    loss = (
        squared_error[
            mask
        ].mean()
    )


    if loss.ndim != 0:

        raise RuntimeError(
            "Transformative pathway loss must be scalar."
        )


    if not bool(
        torch.isfinite(
            loss
        ).item()
    ):

        raise RuntimeError(
            "Transformative pathway loss is non-finite."
        )


    return loss


# =============================================================================
# Transformative forward helper
# =============================================================================

def block_8_transformative_forward(
    module,
    current_state,
    preceding_state,
):

    output = module(
        current_state,
        preceding_state,
    )


    if not torch.is_tensor(
        output
    ):

        raise TypeError(
            "Transformative mechanism output must be a torch.Tensor."
        )


    return output


# =============================================================================
# Module execution modes
# =============================================================================

for module in BLOCK_8_REPRESENTATION_MODULES.values():

    module.eval()


for module in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values():

    module.eval()


for module in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values():

    module.train()


BLOCK_8_REPRESENTATION_EVAL_MODE = all(
    not module.training

    for module
    in BLOCK_8_REPRESENTATION_MODULES.values()
)


BLOCK_8_TRANSFORMATIVE_TRAIN_MODE = all(
    module.training

    for module
    in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values()
)


if not BLOCK_8_REPRESENTATION_EVAL_MODE:

    raise RuntimeError(
        "Inherited representation modules are not all in evaluation mode."
    )


if not BLOCK_8_TRANSFORMATIVE_TRAIN_MODE:

    raise RuntimeError(
        "Transformative modules are not all in training mode."
    )


# =============================================================================
# Optimiser parameter collection
# =============================================================================

BLOCK_8_TRAINABLE_TRANSFORMATIVE_PARAMETERS = [
    parameter

    for module
    in BLOCK_8_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()

    if parameter.requires_grad
]


BLOCK_8_TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel()

    for parameter
    in BLOCK_8_TRAINABLE_TRANSFORMATIVE_PARAMETERS
)


BLOCK_8_TRAINABLE_PARAMETER_COUNT_VALID = (
    BLOCK_8_TRAINABLE_PARAMETER_COUNT
    ==
    BLOCK_8_ELIGIBLE_TRANSFORMATIVE_PARAMETER_COUNT
)


if not BLOCK_8_TRAINABLE_PARAMETER_COUNT_VALID:

    raise RuntimeError(
        "Transformative trainable parameter count does not match "
        "the Block 7 eligible pathway scope."
    )


# =============================================================================
# Optimiser construction
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER = torch.optim.AdamW(
    BLOCK_8_TRAINABLE_TRANSFORMATIVE_PARAMETERS,
    lr=
        BLOCK_8_LEARNING_RATE,
    weight_decay=
        BLOCK_8_WEIGHT_DECAY,
)


NOTEBOOK_08_BLOCK_8_OPTIMIZER_CREATED = True


# =============================================================================
# Optimiser parameter validation
# =============================================================================

BLOCK_8_OPTIMIZER_PARAMETER_IDS = {
    id(
        parameter
    )

    for parameter_group
    in NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER.param_groups

    for parameter
    in parameter_group[
        "params"
    ]
}


BLOCK_8_OPTIMIZER_ONLY_ELIGIBLE_TRANSFORMATIVE = (
    BLOCK_8_OPTIMIZER_PARAMETER_IDS
    ==
    BLOCK_8_ELIGIBLE_TRANSFORMATIVE_PARAMETER_IDS
)


BLOCK_8_OPTIMIZER_EXCLUDES_REPRESENTATION = (
    BLOCK_8_OPTIMIZER_PARAMETER_IDS.isdisjoint(
        BLOCK_8_REPRESENTATION_PARAMETER_IDS
    )
)


if not BLOCK_8_OPTIMIZER_ONLY_ELIGIBLE_TRANSFORMATIVE:

    raise RuntimeError(
        "Optimizer parameter set does not exactly match "
        "the eligible transformative parameters."
    )


if not BLOCK_8_OPTIMIZER_EXCLUDES_REPRESENTATION:

    raise RuntimeError(
        "Optimizer incorrectly contains inherited representation parameters."
    )


# =============================================================================
# Initial deterministic objective
# =============================================================================
#
# Evaluation mode is used temporarily so dropout does not affect the baseline.
# =============================================================================

for module in BLOCK_8_TRANSFORMATIVE_MODULES.values():

    module.eval()


with torch.no_grad():

    BLOCK_8_INITIAL_PATHWAY_LOSSES = OrderedDict()


    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.items():

        prediction = (
            block_8_transformative_forward(
                module,
                BLOCK_8_CURRENT_INPUTS[
                    pathway_name
                ],
                BLOCK_8_PRECEDING_INPUTS[
                    pathway_name
                ],
            )
        )


        expected_shape = (
            BLOCK_8_TARGETS[
                pathway_name
            ].shape
        )


        if prediction.shape != expected_shape:

            raise RuntimeError(
                f"{pathway_name} transformative prediction shape "
                f"is invalid. Expected {tuple(expected_shape)}, "
                f"received {tuple(prediction.shape)}."
            )


        if not bool(
            torch.isfinite(
                prediction
            ).all()
            .item()
        ):

            raise RuntimeError(
                f"{pathway_name} initial transformative output "
                "contains non-finite values."
            )


        pathway_loss = (
            block_8_masked_mean_squared_error(
                prediction=
                    prediction,

                target=
                    BLOCK_8_TARGETS[
                        pathway_name
                    ],

                mask=
                    BLOCK_8_EFFECTIVE_TARGET_MASKS[
                        pathway_name
                    ],
            )
        )


        BLOCK_8_INITIAL_PATHWAY_LOSSES[
            pathway_name
        ] = pathway_loss


    BLOCK_8_INITIAL_TOTAL_OBJECTIVE = sum(
        BLOCK_8_INITIAL_PATHWAY_LOSSES[
            pathway_name
        ]
        *
        BLOCK_8_PATHWAY_LOSS_WEIGHTS[
            pathway_name
        ]

        for pathway_name
        in BLOCK_8_ELIGIBLE_PATHWAYS
    )


BLOCK_8_INITIAL_OBJECTIVE_FINITE = bool(
    torch.isfinite(
        BLOCK_8_INITIAL_TOTAL_OBJECTIVE
    ).item()
)


if not BLOCK_8_INITIAL_OBJECTIVE_FINITE:

    raise RuntimeError(
        "Initial transformative training objective is non-finite."
    )


# =============================================================================
# Controlled transformative training
# =============================================================================

for module in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.values():

    module.train()


BLOCK_8_EPOCH_HISTORY = []

BLOCK_8_MAXIMUM_OBSERVED_GRADIENT_NORM = (
    0.0
)

BLOCK_8_ALL_GRADIENTS_FINITE = True

NOTEBOOK_08_BLOCK_8_ZERO_GRAD_EXECUTED = False

NOTEBOOK_08_BLOCK_8_FORWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_8_LOSS_CALCULATED = False

NOTEBOOK_08_BLOCK_8_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_08_BLOCK_8_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_08_BLOCK_8_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_08_BLOCK_8_PARAMETER_UPDATE_EXECUTED = False


for epoch_index in range(
    BLOCK_8_EPOCHS
):

    # -------------------------------------------------------------------------
    # Deterministic dropout sequence
    # -------------------------------------------------------------------------

    epoch_seed = (
        SEED
        +
        epoch_index
    )


    torch.manual_seed(
        epoch_seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            epoch_seed
        )


    # -------------------------------------------------------------------------
    # Reset gradients
    # -------------------------------------------------------------------------

    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER.zero_grad(
        set_to_none=
            True
    )


    NOTEBOOK_08_BLOCK_8_ZERO_GRAD_EXECUTED = True


    # -------------------------------------------------------------------------
    # Pathway forward passes and losses
    # -------------------------------------------------------------------------

    epoch_pathway_losses = OrderedDict()


    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.items():

        prediction = (
            block_8_transformative_forward(
                module,
                BLOCK_8_CURRENT_INPUTS[
                    pathway_name
                ],
                BLOCK_8_PRECEDING_INPUTS[
                    pathway_name
                ],
            )
        )


        NOTEBOOK_08_BLOCK_8_FORWARD_PASS_EXECUTED = True


        if prediction.shape != (
            BLOCK_8_TARGETS[
                pathway_name
            ].shape
        ):

            raise RuntimeError(
                f"{pathway_name} prediction shape changed "
                "during training."
            )


        if not bool(
            torch.isfinite(
                prediction
            ).all()
            .item()
        ):

            raise RuntimeError(
                f"{pathway_name} transformative output became "
                "non-finite during training."
            )


        pathway_loss = (
            block_8_masked_mean_squared_error(
                prediction=
                    prediction,

                target=
                    BLOCK_8_TARGETS[
                        pathway_name
                    ],

                mask=
                    BLOCK_8_EFFECTIVE_TARGET_MASKS[
                        pathway_name
                    ],
            )
        )


        epoch_pathway_losses[
            pathway_name
        ] = pathway_loss


    NOTEBOOK_08_BLOCK_8_LOSS_CALCULATED = True


    # -------------------------------------------------------------------------
    # Combined objective
    # -------------------------------------------------------------------------

    total_objective = sum(
        epoch_pathway_losses[
            pathway_name
        ]
        *
        BLOCK_8_PATHWAY_LOSS_WEIGHTS[
            pathway_name
        ]

        for pathway_name
        in BLOCK_8_ELIGIBLE_PATHWAYS
    )


    if not bool(
        torch.isfinite(
            total_objective
        ).item()
    ):

        raise RuntimeError(
            f"Epoch {epoch_index + 1} total transformative objective "
            "is non-finite."
        )


    # -------------------------------------------------------------------------
    # Backward
    # -------------------------------------------------------------------------

    total_objective.backward()


    NOTEBOOK_08_BLOCK_8_BACKWARD_PASS_EXECUTED = True


    # -------------------------------------------------------------------------
    # Gradient finiteness validation
    # -------------------------------------------------------------------------

    epoch_gradients_finite = all(
        (
            parameter.grad is None
            or
            bool(
                torch.isfinite(
                    parameter.grad
                ).all()
                .item()
            )
        )

        for parameter
        in BLOCK_8_TRAINABLE_TRANSFORMATIVE_PARAMETERS
    )


    if not epoch_gradients_finite:

        BLOCK_8_ALL_GRADIENTS_FINITE = False

        raise RuntimeError(
            f"Epoch {epoch_index + 1} produced non-finite gradients."
        )


    # -------------------------------------------------------------------------
    # Gradient clipping
    # -------------------------------------------------------------------------

    preclip_gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            BLOCK_8_TRAINABLE_TRANSFORMATIVE_PARAMETERS,
            max_norm=
                BLOCK_8_MAXIMUM_GRADIENT_NORM,
        )
    )


    NOTEBOOK_08_BLOCK_8_GRADIENT_CLIPPING_EXECUTED = True


    preclip_gradient_norm_value = float(
        preclip_gradient_norm.detach()
        .cpu()
        .item()
        if torch.is_tensor(
            preclip_gradient_norm
        )
        else preclip_gradient_norm
    )


    if not np.isfinite(
        preclip_gradient_norm_value
    ):

        raise RuntimeError(
            f"Epoch {epoch_index + 1} gradient norm is non-finite."
        )


    BLOCK_8_MAXIMUM_OBSERVED_GRADIENT_NORM = max(
        BLOCK_8_MAXIMUM_OBSERVED_GRADIENT_NORM,
        preclip_gradient_norm_value,
    )


    # -------------------------------------------------------------------------
    # Optimiser update
    # -------------------------------------------------------------------------

    NOTEBOOK_08_TRANSFORMATIVE_OPTIMIZER.step()


    NOTEBOOK_08_BLOCK_8_OPTIMIZER_STEP_EXECUTED = True

    NOTEBOOK_08_BLOCK_8_PARAMETER_UPDATE_EXECUTED = True


    # -------------------------------------------------------------------------
    # Epoch record
    # -------------------------------------------------------------------------

    epoch_record = {
        "epoch":
            epoch_index
            +
            1,

        "pathway_losses":
            {
                pathway_name:
                    float(
                        pathway_loss.detach()
                        .cpu()
                        .item()
                    )

                for (
                    pathway_name,
                    pathway_loss,
                ) in epoch_pathway_losses.items()
            },

        "total_objective":
            float(
                total_objective.detach()
                .cpu()
                .item()
            ),

        "preclip_gradient_norm":
            preclip_gradient_norm_value,
    }


    epoch_record_values = (
        list(
            epoch_record[
                "pathway_losses"
            ].values()
        )
        +
        [
            epoch_record[
                "total_objective"
            ],
            epoch_record[
                "preclip_gradient_norm"
            ],
        ]
    )


    if not all(
        np.isfinite(
            float(
                value
            )
        )

        for value
        in epoch_record_values
    ):

        raise RuntimeError(
            f"Epoch {epoch_index + 1} history contains "
            "a non-finite value."
        )


    BLOCK_8_EPOCH_HISTORY.append(
        epoch_record
    )


# =============================================================================
# Final deterministic objective
# =============================================================================

for module in BLOCK_8_TRANSFORMATIVE_MODULES.values():

    module.eval()


with torch.no_grad():

    BLOCK_8_FINAL_PATHWAY_LOSSES = OrderedDict()

    BLOCK_8_FINAL_PREDICTIONS = {}


    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRAINING_TRANSFORMATIVE_MODULES.items():

        prediction = (
            block_8_transformative_forward(
                module,
                BLOCK_8_CURRENT_INPUTS[
                    pathway_name
                ],
                BLOCK_8_PRECEDING_INPUTS[
                    pathway_name
                ],
            )
        )


        BLOCK_8_FINAL_PREDICTIONS[
            pathway_name
        ] = (
            prediction.detach()
            .clone()
        )


        pathway_loss = (
            block_8_masked_mean_squared_error(
                prediction=
                    prediction,

                target=
                    BLOCK_8_TARGETS[
                        pathway_name
                    ],

                mask=
                    BLOCK_8_EFFECTIVE_TARGET_MASKS[
                        pathway_name
                    ],
            )
        )


        BLOCK_8_FINAL_PATHWAY_LOSSES[
            pathway_name
        ] = pathway_loss


    BLOCK_8_FINAL_TOTAL_OBJECTIVE = sum(
        BLOCK_8_FINAL_PATHWAY_LOSSES[
            pathway_name
        ]
        *
        BLOCK_8_PATHWAY_LOSS_WEIGHTS[
            pathway_name
        ]

        for pathway_name
        in BLOCK_8_ELIGIBLE_PATHWAYS
    )


BLOCK_8_FINAL_OBJECTIVE_FINITE = bool(
    torch.isfinite(
        BLOCK_8_FINAL_TOTAL_OBJECTIVE
    ).item()
)


if not BLOCK_8_FINAL_OBJECTIVE_FINITE:

    raise RuntimeError(
        "Final transformative objective is non-finite."
    )


# =============================================================================
# Objective diagnostics
# =============================================================================

BLOCK_8_INITIAL_TOTAL_OBJECTIVE_VALUE = float(
    BLOCK_8_INITIAL_TOTAL_OBJECTIVE.detach()
    .cpu()
    .item()
)


BLOCK_8_FINAL_TOTAL_OBJECTIVE_VALUE = float(
    BLOCK_8_FINAL_TOTAL_OBJECTIVE.detach()
    .cpu()
    .item()
)


BLOCK_8_OBJECTIVE_DELTA = (
    BLOCK_8_FINAL_TOTAL_OBJECTIVE_VALUE
    -
    BLOCK_8_INITIAL_TOTAL_OBJECTIVE_VALUE
)


BLOCK_8_OBJECTIVE_IMPROVED = (
    BLOCK_8_FINAL_TOTAL_OBJECTIVE_VALUE
    <
    BLOCK_8_INITIAL_TOTAL_OBJECTIVE_VALUE
)


# =============================================================================
# Representation-model immutability validation
# =============================================================================

BLOCK_8_REPRESENTATION_UNCHANGED_BY_MODULE = {
    module_name:
        block_8_module_matches_snapshot(
            module,
            BLOCK_8_REPRESENTATION_SNAPSHOT[
                module_name
            ],
        )

    for (
        module_name,
        module,
    ) in BLOCK_8_REPRESENTATION_MODULES.items()
}


BLOCK_8_REPRESENTATION_UNCHANGED = all(
    BLOCK_8_REPRESENTATION_UNCHANGED_BY_MODULE.values()
)


if not BLOCK_8_REPRESENTATION_UNCHANGED:

    changed_representation_modules = [
        module_name
        for (
            module_name,
            unchanged,
        ) in BLOCK_8_REPRESENTATION_UNCHANGED_BY_MODULE.items()
        if not unchanged
    ]


    raise RuntimeError(
        "Inherited representation parameters changed during "
        "transformative training. Changed modules: "
        f"{changed_representation_modules}"
    )


# =============================================================================
# Transformative parameter-update validation
# =============================================================================

BLOCK_8_TRANSFORMATIVE_MAX_PARAMETER_DELTAS = OrderedDict(
    (
        pathway_name,
        block_8_maximum_parameter_delta(
            module,
            BLOCK_8_TRANSFORMATIVE_INITIAL_SNAPSHOT[
                pathway_name
            ],
        ),
    )

    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRANSFORMATIVE_MODULES.items()
)


BLOCK_8_TRANSFORMATIVE_PARAMETERS_CHANGED = {
    pathway_name:
        (
            maximum_delta
            >
            0.0
        )

    for (
        pathway_name,
        maximum_delta,
    ) in BLOCK_8_TRANSFORMATIVE_MAX_PARAMETER_DELTAS.items()
}


BLOCK_8_ALL_TRANSFORMATIVE_PATHWAYS_UPDATED = all(
    BLOCK_8_TRANSFORMATIVE_PARAMETERS_CHANGED.values()
)


if not BLOCK_8_ALL_TRANSFORMATIVE_PATHWAYS_UPDATED:

    unchanged_pathways = [
        pathway_name
        for (
            pathway_name,
            changed,
        ) in BLOCK_8_TRANSFORMATIVE_PARAMETERS_CHANGED.items()
        if not changed
    ]


    raise RuntimeError(
        "One or more transformative pathways received no "
        "parameter update: "
        f"{unchanged_pathways}"
    )


BLOCK_8_MAXIMUM_TRANSFORMATIVE_PARAMETER_DELTA = max(
    BLOCK_8_TRANSFORMATIVE_MAX_PARAMETER_DELTAS.values()
)


# =============================================================================
# Final output validation
# =============================================================================

BLOCK_8_FINAL_OUTPUT_SHAPES_VALID = all(
    tuple(
        BLOCK_8_FINAL_PREDICTIONS[
            pathway_name
        ].shape
    )
    ==
    tuple(
        BLOCK_8_TARGETS[
            pathway_name
        ].shape
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


BLOCK_8_FINAL_OUTPUTS_FINITE = all(
    bool(
        torch.isfinite(
            BLOCK_8_FINAL_PREDICTIONS[
                pathway_name
            ]
        ).all()
        .item()
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


if not all(
    [
        BLOCK_8_FINAL_OUTPUT_SHAPES_VALID,
        BLOCK_8_FINAL_OUTPUTS_FINITE,
    ]
):

    raise RuntimeError(
        "Final transformative outputs failed dimensional "
        "or numerical validation."
    )


# =============================================================================
# Active-target accounting
# =============================================================================

BLOCK_8_ACTIVE_TARGET_ELEMENTS = OrderedDict(
    (
        pathway_name,
        int(
            BLOCK_8_EFFECTIVE_TARGET_MASKS[
                pathway_name
            ].sum()
            .item()
        ),
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


BLOCK_8_ACTIVE_DIMENSIONS = OrderedDict(
    (
        pathway_name,
        list(
            NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES[
                pathway_name
            ]
        ),
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


BLOCK_8_DEFERRED_DIMENSIONS = OrderedDict(
    (
        pathway_name,
        list(
            NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES[
                pathway_name
            ]
        ),
    )

    for pathway_name
    in BLOCK_8_ELIGIBLE_PATHWAYS
)


# =============================================================================
# Architectural-boundary flags
# =============================================================================

NOTEBOOK_08_BLOCK_8_TRANSFORMATIVE_WEIGHTS_CREATED = False

NOTEBOOK_08_BLOCK_8_RECURRENT_STATE_CREATED = False

NOTEBOOK_08_BLOCK_8_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_08_BLOCK_8_FUTURE_INFORMATION_USED = False

NOTEBOOK_08_BLOCK_8_REPRESENTATION_PARAMETER_UPDATE_EXECUTED = False


# =============================================================================
# Training-history validation
# =============================================================================

BLOCK_8_HISTORY_LENGTH_VALID = (
    len(
        BLOCK_8_EPOCH_HISTORY
    )
    ==
    BLOCK_8_EPOCHS
)


BLOCK_8_HISTORY_FINITE = all(
    all(
        np.isfinite(
            float(
                value
            )
        )

        for value
        in (
            list(
                epoch_record[
                    "pathway_losses"
                ].values()
            )
            +
            [
                epoch_record[
                    "total_objective"
                ],
                epoch_record[
                    "preclip_gradient_norm"
                ],
            ]
        )
    )

    for epoch_record
    in BLOCK_8_EPOCH_HISTORY
)


# =============================================================================
# Final training validation
# =============================================================================

NOTEBOOK_08_TRANSFORMATIVE_TRAINING_EXECUTED = True


NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID = all(
    [
        BLOCK_8_REPRESENTATION_FULLY_FROZEN,
        BLOCK_8_TRANSFORMATIVE_FULLY_TRAINABLE,
        BLOCK_8_PARAMETER_GROUPS_DISJOINT,

        BLOCK_8_OPTIMIZER_ONLY_ELIGIBLE_TRANSFORMATIVE,
        BLOCK_8_TRAINABLE_PARAMETER_COUNT_VALID,
        BLOCK_8_OPTIMISATION_POLICY_VALID,
        BLOCK_8_OPTIMIZER_EXCLUDES_REPRESENTATION,

        BLOCK_8_TRAINING_INPUTS_FINITE,
        all(
            BLOCK_8_TRAINING_SHAPES_VALID.values()
        ),

        BLOCK_8_INITIAL_OBJECTIVE_FINITE,
        BLOCK_8_FINAL_OBJECTIVE_FINITE,

        BLOCK_8_ALL_GRADIENTS_FINITE,
        BLOCK_8_HISTORY_LENGTH_VALID,
        BLOCK_8_HISTORY_FINITE,

        BLOCK_8_REPRESENTATION_UNCHANGED,
        BLOCK_8_ALL_TRANSFORMATIVE_PATHWAYS_UPDATED,

        BLOCK_8_FINAL_OUTPUT_SHAPES_VALID,
        BLOCK_8_FINAL_OUTPUTS_FINITE,

        NOTEBOOK_08_BLOCK_8_ZERO_GRAD_EXECUTED,
        NOTEBOOK_08_BLOCK_8_FORWARD_PASS_EXECUTED,
        NOTEBOOK_08_BLOCK_8_LOSS_CALCULATED,
        NOTEBOOK_08_BLOCK_8_BACKWARD_PASS_EXECUTED,
        NOTEBOOK_08_BLOCK_8_GRADIENT_CLIPPING_EXECUTED,
        NOTEBOOK_08_BLOCK_8_OPTIMIZER_STEP_EXECUTED,
        NOTEBOOK_08_BLOCK_8_PARAMETER_UPDATE_EXECUTED,

        not NOTEBOOK_08_BLOCK_8_TRANSFORMATIVE_WEIGHTS_CREATED,
        not NOTEBOOK_08_BLOCK_8_RECURRENT_STATE_CREATED,
        not NOTEBOOK_08_BLOCK_8_RECURRENT_UPDATE_EXECUTED,
        not NOTEBOOK_08_BLOCK_8_FUTURE_INFORMATION_USED,
        not NOTEBOOK_08_BLOCK_8_REPRESENTATION_PARAMETER_UPDATE_EXECUTED,
    ]
)


if not NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID:

    raise RuntimeError(
        "Notebook 08 Block 8 controlled transformative training "
        "failed methodological validation."
    )


# =============================================================================
# Block completion
# =============================================================================

NOTEBOOK_08_BLOCK_8_VALID = True

NOTEBOOK_08_BLOCK_8_COMPLETE = True


# =============================================================================
# Retained trained state
# =============================================================================

NOTEBOOK_08_TRAINED_TRANSFORMATIVE_MODULES = {
    pathway_name:
        module

    for (
        pathway_name,
        module,
    ) in BLOCK_8_TRANSFORMATIVE_MODULES.items()
}


NOTEBOOK_08_TRANSFORMATIVE_TRAINING_HISTORY = deepcopy(
    BLOCK_8_EPOCH_HISTORY
)


NOTEBOOK_08_TRANSFORMATIVE_INITIAL_OBJECTIVE = (
    BLOCK_8_INITIAL_TOTAL_OBJECTIVE_VALUE
)


NOTEBOOK_08_TRANSFORMATIVE_FINAL_OBJECTIVE = (
    BLOCK_8_FINAL_TOTAL_OBJECTIVE_VALUE
)


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_08_BLOCK_8_SUMMARY = {
    "block":
        NOTEBOOK_08_BLOCK_8,

    "block_name":
        NOTEBOOK_08_BLOCK_8_NAME,

    "version":
        NOTEBOOK_08_BLOCK_8_VERSION,

    "optimizer":
        BLOCK_8_OPTIMIZER_NAME,

    "learning_rate":
        BLOCK_8_LEARNING_RATE,

    "weight_decay":
        BLOCK_8_WEIGHT_DECAY,

    "maximum_gradient_norm":
        BLOCK_8_MAXIMUM_GRADIENT_NORM,

    "epochs":
        BLOCK_8_EPOCHS,

    "eligible_pathways":
        list(
            BLOCK_8_ELIGIBLE_PATHWAYS
        ),

    "pathway_loss_weights":
        dict(
            BLOCK_8_PATHWAY_LOSS_WEIGHTS
        ),

    "representation_parameters":
        BLOCK_8_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameters":
        BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNT,

    "active_target_elements":
        dict(
            BLOCK_8_ACTIVE_TARGET_ELEMENTS
        ),

    "active_dimensions":
        dict(
            BLOCK_8_ACTIVE_DIMENSIONS
        ),

    "deferred_dimensions":
        dict(
            BLOCK_8_DEFERRED_DIMENSIONS
        ),

    "initial_objective":
        BLOCK_8_INITIAL_TOTAL_OBJECTIVE_VALUE,

    "final_objective":
        BLOCK_8_FINAL_TOTAL_OBJECTIVE_VALUE,

    "objective_delta":
        BLOCK_8_OBJECTIVE_DELTA,

    "objective_improved":
        BLOCK_8_OBJECTIVE_IMPROVED,

    "maximum_observed_gradient_norm":
        BLOCK_8_MAXIMUM_OBSERVED_GRADIENT_NORM,

    "maximum_parameter_delta":
        BLOCK_8_MAXIMUM_TRANSFORMATIVE_PARAMETER_DELTA,

    "representation_unchanged":
        BLOCK_8_REPRESENTATION_UNCHANGED,

    "all_transformative_pathways_updated":
        BLOCK_8_ALL_TRANSFORMATIVE_PATHWAYS_UPDATED,

    "training_valid":
        NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID,

    "block_valid":
        NOTEBOOK_08_BLOCK_8_VALID,

    "block_complete":
        NOTEBOOK_08_BLOCK_8_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 8: "
    "Controlled Transformative-Mechanism Training "
    "and Parameter-Update Validation"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_8_VERSION}"
)

print("-" * 72)

print(
    "Optimisation policy"
)

print(
    f"Optimizer                   : "
    f"{BLOCK_8_OPTIMIZER_NAME}"
)

print(
    f"Learning rate               : "
    f"{BLOCK_8_LEARNING_RATE}"
)

print(
    f"Weight decay                : "
    f"{BLOCK_8_WEIGHT_DECAY}"
)

print(
    f"Maximum gradient norm       : "
    f"{BLOCK_8_MAXIMUM_GRADIENT_NORM}"
)

print(
    f"Epochs                      : "
    f"{BLOCK_8_EPOCHS}"
)

print(
    f"Sequence order preserved    : "
    f"{BLOCK_8_SEQUENCE_ORDER_PRESERVED}"
)

print(
    f"Batch shuffling used        : "
    f"{BLOCK_8_BATCH_SHUFFLING_USED}"
)

print(
    f"Future context allowed      : "
    f"{BLOCK_8_FUTURE_CONTEXT_ALLOWED}"
)


print(
    f"Eligible pathways           : "
    f"{list(BLOCK_8_ELIGIBLE_PATHWAYS)}"
)

print(
    f"Pathway loss weights        : "
    f"{dict(BLOCK_8_PATHWAY_LOSS_WEIGHTS)}"
)

print("-" * 72)

print(
    "Parameter scope"
)

print(
    f"Representation parameters   : "
    f"{BLOCK_8_REPRESENTATION_PARAMETER_COUNT:,}"
)

for (
    pathway_name,
    parameter_count,
) in BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNTS.items():

    print(
        f"{pathway_name:<14} transform params   : "
        f"{parameter_count:,}"
    )

print(
    f"Total transformative params : "
    f"{BLOCK_8_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Optimizer parameter count   : "
    f"{BLOCK_8_TRAINABLE_PARAMETER_COUNT:,}"
)

print("-" * 72)

print(
    "Active transformative supervision"
)

for pathway_name in BLOCK_8_ELIGIBLE_PATHWAYS:

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{BLOCK_8_ACTIVE_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{BLOCK_8_DEFERRED_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} target elements    : "
        f"{BLOCK_8_ACTIVE_TARGET_ELEMENTS[pathway_name]}"
    )

print("-" * 72)

print(
    "Training objective"
)

print(
    f"Initial total objective     : "
    f"{BLOCK_8_INITIAL_TOTAL_OBJECTIVE_VALUE:.8f}"
)

print(
    f"Final total objective       : "
    f"{BLOCK_8_FINAL_TOTAL_OBJECTIVE_VALUE:.8f}"
)

print(
    f"Objective delta             : "
    f"{BLOCK_8_OBJECTIVE_DELTA:.8f}"
)

print(
    f"Objective improved          : "
    f"{BLOCK_8_OBJECTIVE_IMPROVED}"
)

print("-" * 72)

print(
    "Final pathway losses"
)

for (
    pathway_name,
    pathway_loss,
) in BLOCK_8_FINAL_PATHWAY_LOSSES.items():

    print(
        f"{pathway_name:<14} masked MSE         : "
        f"{pathway_loss.detach().cpu().item():.8f}"
    )

print("-" * 72)

print(
    "Gradient and parameter validation"
)

print(
    f"All gradients finite        : "
    f"{BLOCK_8_ALL_GRADIENTS_FINITE}"
)

print(
    f"Maximum observed grad norm  : "
    f"{BLOCK_8_MAXIMUM_OBSERVED_GRADIENT_NORM:.8f}"
)

for (
    pathway_name,
    maximum_delta,
) in BLOCK_8_TRANSFORMATIVE_MAX_PARAMETER_DELTAS.items():

    print(
        f"{pathway_name:<14} max param delta    : "
        f"{maximum_delta:.10f}"
    )

print(
    f"All transform paths updated : "
    f"{BLOCK_8_ALL_TRANSFORMATIVE_PATHWAYS_UPDATED}"
)

print(
    f"Representation unchanged    : "
    f"{BLOCK_8_REPRESENTATION_UNCHANGED}"
)

print("-" * 72)

print(
    "Execution state"
)

print(
    f"Optimizer created           : "
    f"{NOTEBOOK_08_BLOCK_8_OPTIMIZER_CREATED}"
)

print(
    f"Zero grad executed          : "
    f"{NOTEBOOK_08_BLOCK_8_ZERO_GRAD_EXECUTED}"
)

print(
    f"Transform forward executed  : "
    f"{NOTEBOOK_08_BLOCK_8_FORWARD_PASS_EXECUTED}"
)

print(
    f"Loss calculated             : "
    f"{NOTEBOOK_08_BLOCK_8_LOSS_CALCULATED}"
)

print(
    f"Backward pass executed      : "
    f"{NOTEBOOK_08_BLOCK_8_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Gradient clipping executed  : "
    f"{NOTEBOOK_08_BLOCK_8_GRADIENT_CLIPPING_EXECUTED}"
)

print(
    f"Optimizer step executed     : "
    f"{NOTEBOOK_08_BLOCK_8_OPTIMIZER_STEP_EXECUTED}"
)

print(
    f"Parameter update executed   : "
    f"{NOTEBOOK_08_BLOCK_8_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Transform weights created   : "
    f"{NOTEBOOK_08_BLOCK_8_TRANSFORMATIVE_WEIGHTS_CREATED}"
)

print(
    f"Recurrent state created     : "
    f"{NOTEBOOK_08_BLOCK_8_RECURRENT_STATE_CREATED}"
)

print(
    f"Recurrent update executed   : "
    f"{NOTEBOOK_08_BLOCK_8_RECURRENT_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Transform training valid    : "
    f"{NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_8_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_8_COMPLETE}"
)

print("=" * 72)

print(
    f"{len(BLOCK_8_ELIGIBLE_PATHWAYS)} eligible transformative pathway(s) "
    "were trained successfully under the validated mask-aware objective."
)

print(
    f"Only the {BLOCK_8_TRAINABLE_PARAMETER_COUNT:,} parameters belonging "
    "to Block 7-eligible transformative pathways were included in the "
    "optimiser."
)

print(
    f"The inherited {BLOCK_8_REPRESENTATION_PARAMETER_COUNT:,}-parameter "
    "representation model remained frozen and exactly unchanged."
)

print(
    "Only mask-valid and activation-eligible derived transformation "
    "targets contributed to the objective."
)

print(
    "Training preserved canonical sequence order and used no future "
    "context."
)

print(
    "The trained outputs remain candidate transformations rather than "
    "recurrent-state updates."
)

print(
    "No Transformative Weight, recurrent state or recurrent update "
    "mechanism was created."
)

print(
    "Notebook 08 may now proceed to final transformative-mechanism "
    "validation and persistent Notebook 08 → Notebook 09 handover."
)

print("=" * 72)

Media AI — Notebook 08, Block 8: Controlled Transformative-Mechanism Training and Parameter-Update Validation
Block version               : 1.1
------------------------------------------------------------------------
Optimisation policy
Optimizer                   : AdamW
Learning rate               : 0.001
Weight decay                : 0.0001
Maximum gradient norm       : 1.0
Epochs                      : 10
Sequence order preserved    : True
Batch shuffling used        : False
Future context allowed      : False
Eligible pathways           : ['factual', 'psychological', 'social']
Pathway loss weights        : {'factual': 0.3333333333333333, 'psychological': 0.3333333333333333, 'social': 0.3333333333333333}
------------------------------------------------------------------------
Parameter scope
Representation parameters   : 894,003
factual        transform params   : 670
psychological  transform params   : 7,174
social         transform params   : 343
Total transformative params : 8,1

## Block 9 — Final Transformative-Mechanism Validation, Persistence and Notebook 08 → 09 Handover

This block completes Notebook 08 by validating the trained transformative mechanisms, preserving their learned parameter state, and constructing the persistent handover required by Notebook 09.

The preceding blocks established three independent transformative pathways for the factual, psychological, and social representation spaces. Each pathway receives the current representation together with a same-dimensional preceding condition and produces a **candidate transformation**. Block 8 then trained these mechanisms against the validated, mask-aware transformation targets constructed in Block 7 while keeping the complete inherited representation model frozen.

The purpose of the present block is therefore not to introduce another learning stage. Instead, it establishes that the trained transformative mechanisms constitute a stable and reproducible architectural state that can safely be inherited by the next notebook.

### Final transformative-mechanism validation

Let the three trained transformative mechanisms be

$$
T_F:\mathbb{R}^{20}\rightarrow\mathbb{R}^{10},
$$

$$
T_P:\mathbb{R}^{68}\rightarrow\mathbb{R}^{34},
$$

and

$$
T_S:\mathbb{R}^{14}\rightarrow\mathbb{R}^{7}.
$$

For pathway \(k\in\{F,P,S\}\), the mechanism receives the concatenation of the current representation \(r_t^{(k)}\) and the preceding condition \(h_{t-1}^{(k)}\):

$$
c_t^{(k)} =
T_k\!\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

The resulting vector \(c_t^{(k)}\) is the learned **candidate transformation** for the pathway.

The final validation must confirm that the trained mechanisms preserve the dimensional contracts established earlier in Notebook 08:

$$
c_t^{(F)}\in\mathbb{R}^{10},
\qquad
c_t^{(P)}\in\mathbb{R}^{34},
\qquad
c_t^{(S)}\in\mathbb{R}^{7}.
$$

The mechanisms are placed in evaluation mode and evaluated under deterministic conditions. Repeated evaluation of identical validated inputs must therefore reproduce identical outputs, subject to the numerical precision of the execution environment.

All resulting candidate transformations must also be finite.

This post-training forward validation is strictly an architectural and numerical verification step. It does not calculate a training loss, perform backpropagation, execute an optimiser step, or modify any learned parameter.

### Preservation of the representation-model boundary

The representation model inherited from Notebook 07 contains **894,003 parameters** distributed across the backbone, global confluent module, and factual, psychological, and social representation heads.

These parameters remained frozen throughout transformative training and must remain exactly unchanged at Notebook 08 completion.

The final validation therefore confirms the separation

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

where \(\theta_R\) denotes the complete inherited representation-model parameter state.

The trained transformative parameters

$$
\theta_T
=
\{
\theta_F,\theta_P,\theta_S
\},
$$

with

$$
|\theta_F|=670,
\qquad
|\theta_P|=7{,}174,
\qquad
|\theta_S|=343,
$$

form a separate learned parameter set containing

$$
|\theta_T|=8{,}187
$$

parameters in total.

Following successful training, these parameters are no longer modified in Notebook 08. They are placed in evaluation mode and treated as the validated transformative-mechanism state to be inherited by Notebook 09.

### Candidate transformations remain distinct from Transformative Weights

A central methodological boundary is preserved during persistence.

The outputs learned in Notebook 08 are **candidate transformations**. They are not themselves Transformative Weights and they do not directly modify the recurrent state.

For each pathway,

$$
c_t^{(k)}
\neq
TW_k,
$$

where \(TW_k\) denotes the future Transformative Weight associated with pathway \(k\).

The role of a Transformative Weight is conceptually different. It will determine how strongly a candidate transformation contributes to the subsequent state transition. That mechanism has deliberately not been introduced in Notebook 08.

Accordingly, this block does **not** create

$$
TW_F,\qquad TW_P,\qquad TW_S.
$$

It also does not infer them from the optimisation-loss coefficients used during transformative training. The factual, psychological, and social loss coefficients defined in Block 6 are optimisation parameters only and must not be interpreted as Transformative Weights.

### No recurrent state is created

The preceding conditions used for target construction and transformative training were controlled causal training conditions. They must not be reinterpreted retrospectively as learned recurrent states.

Notebook 08 therefore ends without creating a persistent state variable such as

$$
h_t^{(k)}.
$$

In particular, the notebook does not yet execute a recurrence of the form

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\,c_t^{(k)}.
$$

This equation represents the conceptual boundary beyond Notebook 08 rather than an operation performed here.

The definition and validation of Transformative Weights, the recurrent-state contract, and the actual state-update mechanism belong to Notebook 09.

### Persistent transformative-mechanism checkpoint

After successful final validation, the trained transformative parameter state is serialised into a dedicated Notebook 08 checkpoint.

The checkpoint preserves the state dictionaries of the three transformative mechanisms together with the information required to reconstruct their architecture unambiguously:

- factual transformative mechanism: \(20\rightarrow20\rightarrow10\);
- psychological transformative mechanism: \(68\rightarrow68\rightarrow34\);
- social transformative mechanism: \(14\rightarrow14\rightarrow7\);
- GELU activation;
- dropout probability \(0.10\);
- hidden-layer LayerNorm;
- pathway dimensions;
- parameter counts;
- active and deferred transformative dimensions;
- training and target-contract provenance;
- Notebook 07 representation-model handover references;
- Notebook 08 block and version information.

The persisted state represents the **trained transformative mechanisms**, not a recurrent model.

### Persistent metadata and provenance

A separate metadata artefact records the methodological and architectural state at the Notebook 08 boundary.

The metadata identifies Notebook 08 as the source notebook and Notebook 09 as the intended consumer. It records that:

- the inherited representation model is validated and frozen;
- the factual, psychological, and social transformative mechanisms have been trained;
- the transformative target contract is valid;
- only mask-valid and activation-eligible targets contributed to training;
- sequence order was preserved;
- future context was not used;
- the transformative mechanisms passed post-training validation;
- candidate transformations are available;
- Transformative Weights have not been created;
- recurrent state has not been created;
- no recurrent update has been executed.

This makes the Notebook 08 → Notebook 09 boundary explicit and machine-verifiable rather than dependent on transient notebook state.

### Persistence read-back validation

Writing an artefact to persistent storage is not by itself sufficient evidence of a valid handover.

The checkpoint and metadata are therefore read back after persistence. Their structure, identity, architecture, parameter counts, and methodological flags are checked against the validated in-memory state.

For every persisted transformative parameter tensor \(\theta_i\), the read-back state must reproduce the state that existed immediately before persistence:

$$
\theta_i^{\mathrm{readback}}
=
\theta_i^{\mathrm{validated}}.
$$

This ensures that Notebook 09 will inherit the same trained transformative mechanisms that were validated at the end of Notebook 08.

### Notebook 08 completion boundary

Notebook 08 is complete only when all of the following conditions hold:

- the inherited representation model remains frozen and unchanged;
- all three transformative mechanisms retain the trained state produced by Block 8;
- deterministic post-training forward validation succeeds;
- candidate outputs have the required factual, psychological, and social dimensions;
- all validated outputs are finite;
- no additional parameter update occurs during final validation;
- the transformative checkpoint is persisted successfully;
- the metadata artefact is persisted successfully;
- both persisted artefacts pass read-back validation;
- no Transformative Weight exists;
- no recurrent state exists;
- no recurrent-state update has been executed.

At this boundary, the Media AI architecture contains two validated learned layers of computation:

$$
\text{prepared input}
\rightarrow
\text{representation model}
\rightarrow
\text{candidate transformative mechanism}.
$$

The next architectural stage is intentionally absent:

$$
\text{candidate transformation}
\rightarrow
\text{Transformative Weight}
\rightarrow
\text{recurrent state update}.
$$

That stage is reserved for Notebook 09.

Consequently, the Notebook 08 handover provides Notebook 09 with a frozen validated representation model and three trained, independently parameterised transformative mechanisms, while leaving the weighting and recurrent-state dynamics to be defined explicitly in the subsequent notebook.

In [15]:
# =============================================================================
# Media AI — Notebook 08, Block 9
# Final Transformative-Mechanism Validation, Persistence
# and Notebook 08 → Notebook 09 Handover
# =============================================================================

from copy import deepcopy
from datetime import datetime, timezone

import hashlib
import io
import json

import numpy as np
import torch

from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaIoBaseUpload,
)


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_08_BLOCK_9 = 9

NOTEBOOK_08_BLOCK_9_NAME = (
    "Final Transformative-Mechanism Validation, Persistence "
    "and Notebook 08 → Notebook 09 Handover"
)

NOTEBOOK_08_BLOCK_9_VERSION = "1.1"

BLOCK_9_CREATED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# =============================================================================
# Required inherited runtime contract
# =============================================================================

required_block_9_objects = [
    # -------------------------------------------------------------------------
    # Drive / runtime
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_DRIVE_SERVICE",
    "SEED",
    "DEVICE",
    "DEFAULT_DTYPE",

    # -------------------------------------------------------------------------
    # Representation model
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_RESTORED_BACKBONE",
    "NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT",
    "NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD",
    "NOTEBOOK_08_RESTORED_FACTUAL_HEAD",
    "NOTEBOOK_08_RESTORED_SOCIAL_HEAD",
    "NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED",

    # -------------------------------------------------------------------------
    # Transformative mechanisms
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM",
    "NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM",

    # -------------------------------------------------------------------------
    # Block 7 target contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_7_COMPLETE",
    "NOTEBOOK_08_BLOCK_7_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS",
    "NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Block 8 training
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_BLOCK_8_COMPLETE",
    "NOTEBOOK_08_BLOCK_8_VALID",
    "NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID",

    # -------------------------------------------------------------------------
    # Dimensions
    # -------------------------------------------------------------------------
    "NOTEBOOK_08_FACTUAL_TRANSFORM_DIM",
    "NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM",
    "NOTEBOOK_08_SOCIAL_TRANSFORM_DIM",
]


missing_block_9_objects = [
    object_name
    for object_name
    in required_block_9_objects
    if object_name not in globals()
]


if missing_block_9_objects:

    raise NameError(
        "Notebook 08 Block 9 prerequisites are not initialised. "
        f"Missing: {missing_block_9_objects}"
    )


if not all(
    [
        NOTEBOOK_08_REPRESENTATION_MODEL_RESTORED,

        NOTEBOOK_08_BLOCK_7_COMPLETE,
        NOTEBOOK_08_BLOCK_7_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID,

        NOTEBOOK_08_BLOCK_8_COMPLETE,
        NOTEBOOK_08_BLOCK_8_VALID,
        NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID,
    ]
):

    raise RuntimeError(
        "Notebook 08 validated representation, target, and training "
        "contracts must be complete before final persistence."
    )


# =============================================================================
# Canonical module collections
# =============================================================================

BLOCK_9_REPRESENTATION_MODULES = {
    "backbone":
        NOTEBOOK_08_RESTORED_BACKBONE,

    "global_confluent":
        NOTEBOOK_08_RESTORED_GLOBAL_CONFLUENT,

    "psychological_head":
        NOTEBOOK_08_RESTORED_PSYCHOLOGICAL_HEAD,

    "factual_head":
        NOTEBOOK_08_RESTORED_FACTUAL_HEAD,

    "social_head":
        NOTEBOOK_08_RESTORED_SOCIAL_HEAD,
}


BLOCK_9_TRANSFORMATIVE_MODULES = {
    "factual":
        NOTEBOOK_08_FACTUAL_TRANSFORMATIVE_MECHANISM,

    "psychological":
        NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,

    "social":
        NOTEBOOK_08_SOCIAL_TRANSFORMATIVE_MECHANISM,
}


BLOCK_9_PATHWAY_DIMS = {
    "factual":
        int(
            NOTEBOOK_08_FACTUAL_TRANSFORM_DIM
        ),

    "psychological":
        int(
            NOTEBOOK_08_PSYCHOLOGICAL_TRANSFORM_DIM
        ),

    "social":
        int(
            NOTEBOOK_08_SOCIAL_TRANSFORM_DIM
        ),
}


BLOCK_9_PATHWAY_DIMENSIONS_VALID = all(
    [
        set(
            BLOCK_9_PATHWAY_DIMS.keys()
        )
        ==
        set(
            BLOCK_9_TRANSFORMATIVE_MODULES.keys()
        ),

        all(
            pathway_dim > 0

            for pathway_dim
            in BLOCK_9_PATHWAY_DIMS.values()
        ),

        set(
            NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS
        ).issubset(
            set(
                BLOCK_9_PATHWAY_DIMS.keys()
            )
        ),
    ]
)


if not BLOCK_9_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 08 final transformative dimensions are invalid."
    )


# =============================================================================
# Parameter counts
# =============================================================================

BLOCK_9_REPRESENTATION_PARAMETER_COUNTS = {
    module_name:
        int(
            sum(
                parameter.numel()
                for parameter
                in module.parameters()
            )
        )

    for (
        module_name,
        module,
    ) in BLOCK_9_REPRESENTATION_MODULES.items()
}


BLOCK_9_TRANSFORMATIVE_PARAMETER_COUNTS = {
    pathway_name:
        int(
            sum(
                parameter.numel()
                for parameter
                in module.parameters()
            )
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_9_TRANSFORMATIVE_MODULES.items()
}


BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS = int(
    sum(
        BLOCK_9_REPRESENTATION_PARAMETER_COUNTS.values()
    )
)


BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS = int(
    sum(
        BLOCK_9_TRANSFORMATIVE_PARAMETER_COUNTS.values()
    )
)


BLOCK_9_COMBINED_PARAMETER_COUNT = (
    BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS
    +
    BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS
)


BLOCK_9_PARAMETER_COUNTS_VALID = all(
    [
        BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS > 0,
        BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS > 0,
        BLOCK_9_COMBINED_PARAMETER_COUNT > 0,

        all(
            parameter_count > 0

            for parameter_count
            in BLOCK_9_REPRESENTATION_PARAMETER_COUNTS.values()
        ),

        all(
            parameter_count > 0

            for parameter_count
            in BLOCK_9_TRANSFORMATIVE_PARAMETER_COUNTS.values()
        ),

        BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS
        ==
        sum(
            BLOCK_9_REPRESENTATION_PARAMETER_COUNTS.values()
        ),

        BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS
        ==
        sum(
            BLOCK_9_TRANSFORMATIVE_PARAMETER_COUNTS.values()
        ),
    ]
)


if not BLOCK_9_PARAMETER_COUNTS_VALID:

    raise RuntimeError(
        "Notebook 08 final dynamic parameter accounting is invalid."
    )


# =============================================================================
# Snapshot representation state before final validation
# =============================================================================

BLOCK_9_REPRESENTATION_STATE_BEFORE = {
    module_name:
        {
            key:
                value.detach().cpu().clone()

            for (
                key,
                value,
            ) in module.state_dict().items()
        }

    for (
        module_name,
        module,
    ) in BLOCK_9_REPRESENTATION_MODULES.items()
}


BLOCK_9_TRANSFORMATIVE_STATE_BEFORE = {
    pathway_name:
        {
            key:
                value.detach().cpu().clone()

            for (
                key,
                value,
            ) in module.state_dict().items()
        }

    for (
        pathway_name,
        module,
    ) in BLOCK_9_TRANSFORMATIVE_MODULES.items()
}


# =============================================================================
# Freeze final inherited architecture
# =============================================================================

for module in BLOCK_9_REPRESENTATION_MODULES.values():

    module.eval()

    for parameter in module.parameters():

        parameter.requires_grad_(
            False
        )


for module in BLOCK_9_TRANSFORMATIVE_MODULES.values():

    module.eval()

    for parameter in module.parameters():

        parameter.requires_grad_(
            False
        )


BLOCK_9_REPRESENTATION_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_9_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_TRANSFORMATIVE_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_9_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_ALL_MODULES_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_9_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_9_TRANSFORMATIVE_MODULES.values()
        )
    )
)


# =============================================================================
# Controlled deterministic post-training forward validation
# =============================================================================
#
# This validation exercises only the transformative mechanisms.
#
# It does not execute the representation model.
# It does not calculate a loss.
# It does not create an optimiser.
# It does not perform a parameter update.
# =============================================================================

BLOCK_9_PROBE_GENERATOR = torch.Generator(
    device=
        "cpu"
)


BLOCK_9_PROBE_GENERATOR.manual_seed(
    SEED
)


BLOCK_9_FORWARD_VALIDATION = {}


with torch.no_grad():

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_9_PATHWAY_DIMS.items():

        module = (
            BLOCK_9_TRANSFORMATIVE_MODULES[
                pathway_name
            ]
        )


        current_probe_cpu = torch.randn(
            (
                1,
                pathway_dim,
            ),
            generator=
                BLOCK_9_PROBE_GENERATOR,
            dtype=
                DEFAULT_DTYPE,
            device=
                "cpu",
        )


        preceding_probe_cpu = torch.randn(
            (
                1,
                pathway_dim,
            ),
            generator=
                BLOCK_9_PROBE_GENERATOR,
            dtype=
                DEFAULT_DTYPE,
            device=
                "cpu",
        )


        current_probe = (
            current_probe_cpu.to(
                DEVICE
            )
        )


        preceding_probe = (
            preceding_probe_cpu.to(
                DEVICE
            )
        )


        output_1 = module(
            current_probe,
            preceding_probe,
        )


        output_2 = module(
            current_probe,
            preceding_probe,
        )


        expected_shape = (
            1,
            pathway_dim,
        )


        shape_valid = (
            tuple(
                output_1.shape
            )
            ==
            expected_shape
        )


        finite = bool(
            torch.isfinite(
                output_1
            ).all().item()
        )


        deterministic = bool(
            torch.equal(
                output_1,
                output_2,
            )
        )


        BLOCK_9_FORWARD_VALIDATION[
            pathway_name
        ] = {
            "expected_shape":
                expected_shape,

            "observed_shape":
                tuple(
                    output_1.shape
                ),

            "shape_valid":
                shape_valid,

            "finite":
                finite,

            "deterministic":
                deterministic,
        }


BLOCK_9_FORWARD_PATH_VALID = all(
    all(
        [
            validation[
                "shape_valid"
            ],

            validation[
                "finite"
            ],

            validation[
                "deterministic"
            ],
        ]
    )

    for validation
    in BLOCK_9_FORWARD_VALIDATION.values()
)


if not BLOCK_9_FORWARD_PATH_VALID:

    raise RuntimeError(
        "Final post-training transformative forward validation failed."
    )


# =============================================================================
# Verify that validation changed no parameter
# =============================================================================

BLOCK_9_REPRESENTATION_STATE_AFTER = {
    module_name:
        {
            key:
                value.detach().cpu().clone()

            for (
                key,
                value,
            ) in module.state_dict().items()
        }

    for (
        module_name,
        module,
    ) in BLOCK_9_REPRESENTATION_MODULES.items()
}


BLOCK_9_TRANSFORMATIVE_STATE_AFTER = {
    pathway_name:
        {
            key:
                value.detach().cpu().clone()

            for (
                key,
                value,
            ) in module.state_dict().items()
        }

    for (
        pathway_name,
        module,
    ) in BLOCK_9_TRANSFORMATIVE_MODULES.items()
}


def block_9_nested_state_exact(
    state_a,
    state_b,
):

    if state_a.keys() != state_b.keys():

        return False


    for group_name in state_a:

        if (
            state_a[
                group_name
            ].keys()
            !=
            state_b[
                group_name
            ].keys()
        ):

            return False


        for tensor_name in state_a[
            group_name
        ]:

            if not torch.equal(
                state_a[
                    group_name
                ][
                    tensor_name
                ],
                state_b[
                    group_name
                ][
                    tensor_name
                ],
            ):

                return False


    return True


BLOCK_9_REPRESENTATION_UNCHANGED = (
    block_9_nested_state_exact(
        BLOCK_9_REPRESENTATION_STATE_BEFORE,
        BLOCK_9_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_9_TRANSFORMATIVE_UNCHANGED_DURING_VALIDATION = (
    block_9_nested_state_exact(
        BLOCK_9_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_9_TRANSFORMATIVE_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_9_REPRESENTATION_UNCHANGED,
        BLOCK_9_TRANSFORMATIVE_UNCHANGED_DURING_VALIDATION,
    ]
):

    raise RuntimeError(
        "Final validation unexpectedly modified model parameters."
    )


# =============================================================================
# Explicit Notebook 08 completion boundary
# =============================================================================

BLOCK_9_TRANSFORMATIVE_WEIGHTS_CREATED = False

BLOCK_9_RECURRENT_STATE_CREATED = False

BLOCK_9_RECURRENT_UPDATE_EXECUTED = False

BLOCK_9_LOSS_CALCULATED = False

BLOCK_9_OPTIMIZER_CREATED = False

BLOCK_9_BACKWARD_PASS_EXECUTED = False

BLOCK_9_PARAMETER_UPDATE_EXECUTED = False

BLOCK_9_REPRESENTATION_FORWARD_PASS_EXECUTED = False

BLOCK_9_TRANSFORM_FORWARD_PASS_EXECUTED = True


# =============================================================================
# CPU-safe persistence helpers
# =============================================================================

def block_9_cpu_state_dict(
    module,
):

    return {
        key:
            value.detach().cpu().clone()

        for (
            key,
            value,
        ) in module.state_dict().items()
    }


def block_9_sha256(
    payload,
):

    return hashlib.sha256(
        payload
    ).hexdigest()


def block_9_torch_serialise(
    value,
):

    buffer = io.BytesIO()


    torch.save(
        value,
        buffer,
    )


    return buffer.getvalue()


def block_9_torch_deserialise(
    payload,
):

    buffer = io.BytesIO(
        payload
    )


    return torch.load(
        buffer,
        map_location=
            "cpu",
        weights_only=
            False,
    )


# =============================================================================
# Runtime architecture contract helper
# =============================================================================
#
# Architecture descriptions are derived from the actual instantiated modules.
# This avoids persisting stale hand-written activation, dropout or dimension
# strings that could disagree with the executable model.
# =============================================================================

def block_9_runtime_module_contract(
    module,
):

    return {
        "class":
            module.__class__.__name__,

        "repr":
            repr(
                module
            ),

        "state_shapes":
            {
                key:
                    list(
                        value.shape
                    )

                for (
                    key,
                    value,
                ) in module.state_dict().items()
            },
    }


BLOCK_9_REPRESENTATION_ARCHITECTURE_CONTRACT = {
    module_name:
        block_9_runtime_module_contract(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_9_REPRESENTATION_MODULES.items()
}


BLOCK_9_TRANSFORMATIVE_ARCHITECTURE_CONTRACT = {
    pathway_name:
        {
            **block_9_runtime_module_contract(
                module
            ),

            "pathway_dimension":
                BLOCK_9_PATHWAY_DIMS[
                    pathway_name
                ],
        }

    for (
        pathway_name,
        module,
    ) in BLOCK_9_TRANSFORMATIVE_MODULES.items()
}


BLOCK_9_ELIGIBLE_PATHWAYS = tuple(
    NOTEBOOK_08_TRANSFORMATIVE_ELIGIBLE_PATHWAYS
)


BLOCK_9_TRAINED_PATHWAYS = tuple(
    pathway_name

    for pathway_name
    in BLOCK_9_ELIGIBLE_PATHWAYS

    if pathway_name
    in BLOCK_9_TRANSFORMATIVE_MODULES
)


BLOCK_9_TRAINED_PATHWAY_CONTRACT_VALID = all(
    [
        len(
            BLOCK_9_TRAINED_PATHWAYS
        )
        >
        0,

        set(
            BLOCK_9_TRAINED_PATHWAYS
        )
        ==
        set(
            BLOCK_9_ELIGIBLE_PATHWAYS
        ),
    ]
)


if not BLOCK_9_TRAINED_PATHWAY_CONTRACT_VALID:

    raise RuntimeError(
        "Final trained-pathway contract does not reproduce "
        "Block 7 eligibility."
    )


# =============================================================================
# Final self-contained Notebook 08 → Notebook 09 checkpoint
# =============================================================================
#
# Notebook 09 should need only this checkpoint plus the metadata artefact.
#
# The checkpoint deliberately contains:
#
#   1. complete frozen representation state;
#   2. complete trained transformative state;
#   3. exact architecture contract;
#   4. active/deferred dimensions;
#   5. explicit Notebook 09 boundary flags.
#
# No Transformative Weight or recurrent state is persisted.
# =============================================================================

BLOCK_9_REPRESENTATION_STATE_DICT = {
    module_name:
        block_9_cpu_state_dict(
            module
        )

    for (
        module_name,
        module,
    ) in BLOCK_9_REPRESENTATION_MODULES.items()
}


BLOCK_9_TRANSFORMATIVE_STATE_DICT = {
    pathway_name:
        block_9_cpu_state_dict(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_9_TRANSFORMATIVE_MODULES.items()
}


NOTEBOOK_08_TO_09_CHECKPOINT = {
    # -------------------------------------------------------------------------
    # Identity
    # -------------------------------------------------------------------------
    "checkpoint_type":
        "validated_transformative_model_handover",

    "checkpoint_version":
        "1.0",

    "source_notebook":
        "08_transformative_mechanism",

    "target_notebook":
        "09_transformative_weight_and_recurrent_state",

    "source_block":
        NOTEBOOK_08_BLOCK_9,

    "source_block_version":
        NOTEBOOK_08_BLOCK_9_VERSION,

    "created_at_utc":
        BLOCK_9_CREATED_AT_UTC,

    # -------------------------------------------------------------------------
    # Representation architecture
    # -------------------------------------------------------------------------
    "representation_architecture":
        deepcopy(
            BLOCK_9_REPRESENTATION_ARCHITECTURE_CONTRACT
        ),

    # -------------------------------------------------------------------------
    # Transformative architecture
    # -------------------------------------------------------------------------
    "transformative_architecture":
        deepcopy(
            BLOCK_9_TRANSFORMATIVE_ARCHITECTURE_CONTRACT
        ),

    # -------------------------------------------------------------------------
    # State dictionaries
    # -------------------------------------------------------------------------
    "representation_state_dict":
        BLOCK_9_REPRESENTATION_STATE_DICT,

    "transformative_state_dict":
        BLOCK_9_TRANSFORMATIVE_STATE_DICT,

    # -------------------------------------------------------------------------
    # Parameter contract
    # -------------------------------------------------------------------------
    "parameter_counts":
        {
            "representation":
                deepcopy(
                    BLOCK_9_REPRESENTATION_PARAMETER_COUNTS
                ),

            "representation_total":
                BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS,

            "transformative":
                deepcopy(
                    BLOCK_9_TRANSFORMATIVE_PARAMETER_COUNTS
                ),

            "transformative_total":
                BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS,

            "combined_total":
                (
                    BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS
                    +
                    BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS
                ),
        },

    # -------------------------------------------------------------------------
    # Pathway contract
    # -------------------------------------------------------------------------
    "pathway_dimensions":
        deepcopy(
            BLOCK_9_PATHWAY_DIMS
        ),

    "eligible_pathways":
        list(
            BLOCK_9_ELIGIBLE_PATHWAYS
        ),

    "trained_pathways":
        list(
            BLOCK_9_TRAINED_PATHWAYS
        ),

    "active_dimension_indices":
        deepcopy(
            NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES
        ),

    "deferred_dimension_indices":
        deepcopy(
            NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES
        ),

    # -------------------------------------------------------------------------
    # Validation state
    # -------------------------------------------------------------------------
    "validation":
        {
            "representation_model_restored":
                True,

            "representation_model_frozen":
                BLOCK_9_REPRESENTATION_FROZEN,

            "representation_state_unchanged":
                BLOCK_9_REPRESENTATION_UNCHANGED,

            "transformative_training_valid":
                NOTEBOOK_08_TRANSFORMATIVE_TRAINING_VALID,

            "transformative_target_contract_valid":
                NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID,

            "transformative_post_training_forward_valid":
                BLOCK_9_FORWARD_PATH_VALID,

            "transformative_state_unchanged_during_final_validation":
                BLOCK_9_TRANSFORMATIVE_UNCHANGED_DURING_VALIDATION,
        },

    # -------------------------------------------------------------------------
    # Methodological boundary
    # -------------------------------------------------------------------------
    "methodological_boundary":
        {
            "candidate_transformations_available":
                True,

            "candidate_transformations_are_transformative_weights":
                False,

            "transformative_weights_created":
                False,

            "recurrent_state_created":
                False,

            "recurrent_update_executed":
                False,

            "future_context_used":
                False,

            "notebook_09_may_define_transformative_weights":
                True,

            "notebook_09_may_define_recurrent_state":
                True,
        },
}


# =============================================================================
# Compact Notebook 08 → Notebook 09 metadata contract
# =============================================================================

NOTEBOOK_08_TO_09_METADATA = {
    "artifact_type":
        "media_ai_notebook_08_to_09_handover_metadata",

    "artifact_version":
        "1.0",

    "created_at_utc":
        BLOCK_9_CREATED_AT_UTC,

    "source_notebook":
        "08_transformative_mechanism",

    "target_notebook":
        "09_transformative_weight_and_recurrent_state",

    "checkpoint_type":
        "validated_transformative_model_handover",

    "checkpoint_filename":
        "notebook_08_final_checkpoint.pt",

    "metadata_filename":
        "notebook_08_final_metadata.json",

    "representation":
        {
            "parameter_count":
                BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS,

            "frozen":
                BLOCK_9_REPRESENTATION_FROZEN,

            "unchanged":
                BLOCK_9_REPRESENTATION_UNCHANGED,

            "spaces":
                [
                    "factual",
                    "psychological",
                    "social",
                ],
        },

    "transformative_mechanisms":
        {
            "parameter_count":
                BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS,

            "architecturally_available":
                list(
                    BLOCK_9_TRANSFORMATIVE_MODULES.keys()
                ),

            "eligible_pathways":
                list(
                    BLOCK_9_ELIGIBLE_PATHWAYS
                ),

            "trained_pathways":
                list(
                    BLOCK_9_TRAINED_PATHWAYS
                ),

            "trained":
                bool(
                    BLOCK_9_TRAINED_PATHWAYS
                ),

            "frozen_for_handover":
                BLOCK_9_TRANSFORMATIVE_FROZEN,

            "post_training_forward_valid":
                BLOCK_9_FORWARD_PATH_VALID,

            "pathway_dimensions":
                deepcopy(
                    BLOCK_9_PATHWAY_DIMS
                ),

            "active_dimension_indices":
                {
                    pathway_name:
                        list(
                            indices
                        )

                    for (
                        pathway_name,
                        indices,
                    ) in (
                        NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES.items()
                    )
                },

            "deferred_dimension_indices":
                {
                    pathway_name:
                        list(
                            indices
                        )

                    for (
                        pathway_name,
                        indices,
                    ) in (
                        NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES.items()
                    )
                },
        },

    "target_contract":
        {
            "valid":
                NOTEBOOK_08_TRANSFORMATIVE_TARGET_CONTRACT_VALID,

            "mask_aware":
                True,

            "signed_targets":
                True,

            "sequence_order_preserved":
                True,

            "future_context_used":
                False,

            "initial_state_policy":
                "deterministic_zero_architectural_initial_condition_per_article",

            "preceding_condition_policy":
                "previous_supervised_state_within_article",
        },

    "notebook_09_boundary":
        {
            "candidate_transformations_available":
                True,

            "transformative_weights_available":
                False,

            "recurrent_state_available":
                False,

            "recurrent_update_available":
                False,

            "next_required_stage":
                (
                    "define_and_validate_transformative_weights_"
                    "and_recurrent_state_update"
                ),
        },

    "notebook_09_ready":
        True,
}


# =============================================================================
# Serialise checkpoint and metadata
# =============================================================================

BLOCK_9_CHECKPOINT_FILENAME = (
    "notebook_08_final_checkpoint.pt"
)

BLOCK_9_METADATA_FILENAME = (
    "notebook_08_final_metadata.json"
)


BLOCK_9_CHECKPOINT_BYTES = (
    block_9_torch_serialise(
        NOTEBOOK_08_TO_09_CHECKPOINT
    )
)


BLOCK_9_METADATA_BYTES = json.dumps(
    NOTEBOOK_08_TO_09_METADATA,
    indent=
        2,
    sort_keys=
        True,
    ensure_ascii=
        False,
).encode(
    "utf-8"
)


BLOCK_9_CHECKPOINT_SHA256 = (
    block_9_sha256(
        BLOCK_9_CHECKPOINT_BYTES
    )
)


BLOCK_9_METADATA_SHA256 = (
    block_9_sha256(
        BLOCK_9_METADATA_BYTES
    )
)


# =============================================================================
# Drive read-back helper
# =============================================================================

def block_9_download_drive_bytes(
    drive_service,
    file_id,
):

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    complete = False


    while not complete:

        _, complete = (
            downloader.next_chunk()
        )


    buffer.seek(
        0
    )


    return buffer.read()


# =============================================================================
# Drive persistence helper
# =============================================================================
#
# Uses only the already-authorised Notebook 08 Drive API service.
#
# No Google Drive filesystem mount is used.
# No new authentication flow is started.
# =============================================================================

def block_9_upsert_drive_bytes(
    drive_service,
    filename,
    payload,
    mime_type,
):

    escaped_filename = filename.replace(
        "'",
        "\\'",
    )


    existing = (
        drive_service.files()
        .list(
            q=(
                f"name = '{escaped_filename}' "
                "and trashed = false"
            ),
            spaces=
                "drive",
            fields=
                "files(id,name,modifiedTime)",
            orderBy=
                "modifiedTime desc",
            pageSize=
                20,
        )
        .execute()
        .get(
            "files",
            [],
        )
    )


    media = MediaIoBaseUpload(
        io.BytesIO(
            payload
        ),
        mimetype=
            mime_type,
        resumable=
            False,
    )


    if existing:

        file_id = (
            existing[
                0
            ][
                "id"
            ]
        )


        result = (
            drive_service.files()
            .update(
                fileId=
                    file_id,
                media_body=
                    media,
                fields=
                    "id,name,size,modifiedTime",
            )
            .execute()
        )


        operation = (
            "updated"
        )


    else:

        result = (
            drive_service.files()
            .create(
                body=
                    {
                        "name":
                            filename,
                    },
                media_body=
                    media,
                fields=
                    "id,name,size,modifiedTime",
            )
            .execute()
        )


        file_id = (
            result[
                "id"
            ]
        )


        operation = (
            "created"
        )


    return {
        "file_id":
            file_id,

        "operation":
            operation,

        "result":
            result,
    }


# =============================================================================
# Persist checkpoint
# =============================================================================

BLOCK_9_CHECKPOINT_DRIVE_RESULT = (
    block_9_upsert_drive_bytes(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_9_CHECKPOINT_FILENAME,
        BLOCK_9_CHECKPOINT_BYTES,
        "application/octet-stream",
    )
)


BLOCK_9_CHECKPOINT_DRIVE_FILE_ID = (
    BLOCK_9_CHECKPOINT_DRIVE_RESULT[
        "file_id"
    ]
)


BLOCK_9_CHECKPOINT_DRIVE_OPERATION = (
    BLOCK_9_CHECKPOINT_DRIVE_RESULT[
        "operation"
    ]
)


# =============================================================================
# Add checkpoint identity to metadata before metadata persistence
# =============================================================================

NOTEBOOK_08_TO_09_METADATA[
    "checkpoint_drive_file_id"
] = BLOCK_9_CHECKPOINT_DRIVE_FILE_ID


NOTEBOOK_08_TO_09_METADATA[
    "checkpoint_sha256"
] = BLOCK_9_CHECKPOINT_SHA256


NOTEBOOK_08_TO_09_METADATA[
    "checkpoint_bytes"
] = len(
    BLOCK_9_CHECKPOINT_BYTES
)


BLOCK_9_METADATA_BYTES = json.dumps(
    NOTEBOOK_08_TO_09_METADATA,
    indent=
        2,
    sort_keys=
        True,
    ensure_ascii=
        False,
).encode(
    "utf-8"
)


BLOCK_9_METADATA_SHA256 = (
    block_9_sha256(
        BLOCK_9_METADATA_BYTES
    )
)


# =============================================================================
# Persist metadata
# =============================================================================

BLOCK_9_METADATA_DRIVE_RESULT = (
    block_9_upsert_drive_bytes(
        NOTEBOOK_08_DRIVE_SERVICE,
        BLOCK_9_METADATA_FILENAME,
        BLOCK_9_METADATA_BYTES,
        "application/json",
    )
)


BLOCK_9_METADATA_DRIVE_FILE_ID = (
    BLOCK_9_METADATA_DRIVE_RESULT[
        "file_id"
    ]
)


BLOCK_9_METADATA_DRIVE_OPERATION = (
    BLOCK_9_METADATA_DRIVE_RESULT[
        "operation"
    ]
)


# =============================================================================
# Stable Notebook 09 bootstrap identifiers
# =============================================================================
#
# These are the only two Drive identifiers Notebook 09 needs in order to
# retrieve the complete validated Notebook 08 handover.
# =============================================================================

NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID = (
    BLOCK_9_CHECKPOINT_DRIVE_FILE_ID
)


NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID = (
    BLOCK_9_METADATA_DRIVE_FILE_ID
)


# =============================================================================
# Read persisted checkpoint and metadata back from Drive
# =============================================================================

BLOCK_9_CHECKPOINT_READBACK_BYTES = (
    block_9_download_drive_bytes(
        NOTEBOOK_08_DRIVE_SERVICE,
        NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID,
    )
)


BLOCK_9_METADATA_READBACK_BYTES = (
    block_9_download_drive_bytes(
        NOTEBOOK_08_DRIVE_SERVICE,
        NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID,
    )
)


BLOCK_9_CHECKPOINT_READBACK_SHA256 = (
    block_9_sha256(
        BLOCK_9_CHECKPOINT_READBACK_BYTES
    )
)


BLOCK_9_METADATA_READBACK_SHA256 = (
    block_9_sha256(
        BLOCK_9_METADATA_READBACK_BYTES
    )
)


BLOCK_9_CHECKPOINT_BYTE_EXACT = (
    BLOCK_9_CHECKPOINT_READBACK_SHA256
    ==
    BLOCK_9_CHECKPOINT_SHA256
)


BLOCK_9_METADATA_BYTE_EXACT = (
    BLOCK_9_METADATA_READBACK_SHA256
    ==
    BLOCK_9_METADATA_SHA256
)


if not all(
    [
        BLOCK_9_CHECKPOINT_BYTE_EXACT,
        BLOCK_9_METADATA_BYTE_EXACT,
    ]
):

    raise RuntimeError(
        "Notebook 08 persisted artefact hash validation failed."
    )


# =============================================================================
# Deserialize read-back artefacts
# =============================================================================

BLOCK_9_CHECKPOINT_READBACK = (
    block_9_torch_deserialise(
        BLOCK_9_CHECKPOINT_READBACK_BYTES
    )
)


BLOCK_9_METADATA_READBACK = json.loads(
    BLOCK_9_METADATA_READBACK_BYTES.decode(
        "utf-8"
    )
)


# =============================================================================
# State-dictionary exactness helper
# =============================================================================

def block_9_state_dict_exact(
    expected_state,
    observed_state,
):

    if expected_state.keys() != observed_state.keys():

        return False


    for key in expected_state:

        expected = expected_state[
            key
        ]


        observed = observed_state[
            key
        ]


        if not (
            torch.is_tensor(
                expected
            )
            and
            torch.is_tensor(
                observed
            )
        ):

            return False


        if not torch.equal(
            expected,
            observed,
        ):

            return False


    return True


# =============================================================================
# Read-back representation state validation
# =============================================================================

BLOCK_9_REPRESENTATION_READBACK_EXACT = all(
    block_9_state_dict_exact(
        BLOCK_9_REPRESENTATION_STATE_DICT[
            module_name
        ],
        BLOCK_9_CHECKPOINT_READBACK[
            "representation_state_dict"
        ][
            module_name
        ],
    )

    for module_name
    in BLOCK_9_REPRESENTATION_STATE_DICT
)


# =============================================================================
# Read-back transformative state validation
# =============================================================================

BLOCK_9_TRANSFORMATIVE_READBACK_EXACT = all(
    block_9_state_dict_exact(
        BLOCK_9_TRANSFORMATIVE_STATE_DICT[
            pathway_name
        ],
        BLOCK_9_CHECKPOINT_READBACK[
            "transformative_state_dict"
        ][
            pathway_name
        ],
    )

    for pathway_name
    in BLOCK_9_TRANSFORMATIVE_STATE_DICT
)


# =============================================================================
# Read-back checkpoint contract validation
# =============================================================================

BLOCK_9_CHECKPOINT_CONTRACT_VALID = all(
    [
        isinstance(
            BLOCK_9_CHECKPOINT_READBACK,
            dict,
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "checkpoint_type"
            )
            ==
            "validated_transformative_model_handover"
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "source_notebook"
            )
            ==
            "08_transformative_mechanism"
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "target_notebook"
            )
            ==
            "09_transformative_weight_and_recurrent_state"
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "parameter_counts",
                {},
            ).get(
                "representation_total"
            )
            ==
            BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "parameter_counts",
                {},
            ).get(
                "transformative_total"
            )
            ==
            BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "pathway_dimensions"
            )
            ==
            BLOCK_9_PATHWAY_DIMS
        ),


        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "representation_architecture"
            )
            ==
            BLOCK_9_REPRESENTATION_ARCHITECTURE_CONTRACT
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "transformative_architecture"
            )
            ==
            BLOCK_9_TRANSFORMATIVE_ARCHITECTURE_CONTRACT
        ),

        (
            tuple(
                BLOCK_9_CHECKPOINT_READBACK.get(
                    "eligible_pathways",
                    [],
                )
            )
            ==
            BLOCK_9_ELIGIBLE_PATHWAYS
        ),

        (
            tuple(
                BLOCK_9_CHECKPOINT_READBACK.get(
                    "trained_pathways",
                    [],
                )
            )
            ==
            BLOCK_9_TRAINED_PATHWAYS
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "methodological_boundary",
                {},
            ).get(
                "candidate_transformations_available"
            )
            is True
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "methodological_boundary",
                {},
            ).get(
                "transformative_weights_created"
            )
            is False
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "methodological_boundary",
                {},
            ).get(
                "recurrent_state_created"
            )
            is False
        ),

        (
            BLOCK_9_CHECKPOINT_READBACK.get(
                "methodological_boundary",
                {},
            ).get(
                "recurrent_update_executed"
            )
            is False
        ),

        BLOCK_9_REPRESENTATION_READBACK_EXACT,
        BLOCK_9_TRANSFORMATIVE_READBACK_EXACT,
    ]
)


# =============================================================================
# Read-back metadata validation
# =============================================================================

BLOCK_9_METADATA_CONTRACT_VALID = all(
    [
        isinstance(
            BLOCK_9_METADATA_READBACK,
            dict,
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "artifact_type"
            )
            ==
            "media_ai_notebook_08_to_09_handover_metadata"
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "source_notebook"
            )
            ==
            "08_transformative_mechanism"
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "target_notebook"
            )
            ==
            "09_transformative_weight_and_recurrent_state"
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "checkpoint_drive_file_id"
            )
            ==
            NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "checkpoint_sha256"
            )
            ==
            BLOCK_9_CHECKPOINT_SHA256
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "representation",
                {},
            ).get(
                "parameter_count"
            )
            ==
            BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "transformative_mechanisms",
                {},
            ).get(
                "parameter_count"
            )
            ==
            BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS
        ),

        (
            tuple(
                BLOCK_9_METADATA_READBACK.get(
                    "transformative_mechanisms",
                    {},
                ).get(
                    "eligible_pathways",
                    [],
                )
            )
            ==
            BLOCK_9_ELIGIBLE_PATHWAYS
        ),

        (
            tuple(
                BLOCK_9_METADATA_READBACK.get(
                    "transformative_mechanisms",
                    {},
                ).get(
                    "trained_pathways",
                    [],
                )
            )
            ==
            BLOCK_9_TRAINED_PATHWAYS
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "notebook_09_boundary",
                {},
            ).get(
                "candidate_transformations_available"
            )
            is True
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "notebook_09_boundary",
                {},
            ).get(
                "transformative_weights_available"
            )
            is False
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "notebook_09_boundary",
                {},
            ).get(
                "recurrent_state_available"
            )
            is False
        ),

        (
            BLOCK_9_METADATA_READBACK.get(
                "notebook_09_ready"
            )
            is True
        ),
    ]
)


# =============================================================================
# Cross-artefact validation
# =============================================================================

BLOCK_9_CROSS_ARTEFACT_VALID = all(
    [
        (
            BLOCK_9_METADATA_READBACK[
                "checkpoint_type"
            ]
            ==
            BLOCK_9_CHECKPOINT_READBACK[
                "checkpoint_type"
            ]
        ),

        (
            BLOCK_9_METADATA_READBACK[
                "source_notebook"
            ]
            ==
            BLOCK_9_CHECKPOINT_READBACK[
                "source_notebook"
            ]
        ),

        (
            BLOCK_9_METADATA_READBACK[
                "target_notebook"
            ]
            ==
            BLOCK_9_CHECKPOINT_READBACK[
                "target_notebook"
            ]
        ),

        (
            BLOCK_9_METADATA_READBACK[
                "representation"
            ][
                "parameter_count"
            ]
            ==
            BLOCK_9_CHECKPOINT_READBACK[
                "parameter_counts"
            ][
                "representation_total"
            ]
        ),

        (
            BLOCK_9_METADATA_READBACK[
                "transformative_mechanisms"
            ][
                "parameter_count"
            ]
            ==
            BLOCK_9_CHECKPOINT_READBACK[
                "parameter_counts"
            ][
                "transformative_total"
            ]
        ),
    ]
)


# =============================================================================
# Notebook 09 bootstrap contract
# =============================================================================
#
# This compact runtime object may be copied directly into Notebook 09 during
# development, while the two Drive file IDs remain the persistent source of
# truth.
# =============================================================================

NOTEBOOK_08_TO_09_HANDOVER = {
    "handover_type":
        "validated_transformative_model_handover",

    "source_notebook":
        "08_transformative_mechanism",

    "target_notebook":
        "09_transformative_weight_and_recurrent_state",

    "checkpoint":
        {
            "filename":
                BLOCK_9_CHECKPOINT_FILENAME,

            "drive_file_id":
                NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID,

            "sha256":
                BLOCK_9_CHECKPOINT_SHA256,

            "bytes":
                len(
                    BLOCK_9_CHECKPOINT_BYTES
                ),
        },

    "metadata":
        {
            "filename":
                BLOCK_9_METADATA_FILENAME,

            "drive_file_id":
                NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID,

            "sha256":
                BLOCK_9_METADATA_SHA256,

            "bytes":
                len(
                    BLOCK_9_METADATA_BYTES
                ),
        },

    "representation":
        {
            "available":
                True,

            "frozen":
                True,

            "parameter_count":
                BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS,
        },

    "transformative_mechanisms":
        {
            "available":
                True,

            "trained":
                bool(
                    BLOCK_9_TRAINED_PATHWAYS
                ),

            "eligible_pathways":
                list(
                    BLOCK_9_ELIGIBLE_PATHWAYS
                ),

            "trained_pathways":
                list(
                    BLOCK_9_TRAINED_PATHWAYS
                ),

            "frozen":
                True,

            "parameter_count":
                BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS,

            "pathway_dimensions":
                deepcopy(
                    BLOCK_9_PATHWAY_DIMS
                ),
        },

    "notebook_09_boundary":
        {
            "candidate_transformations_available":
                True,

            "transformative_weights_created":
                False,

            "recurrent_state_created":
                False,

            "recurrent_update_executed":
                False,
        },

    "notebook_09_ready":
        True,
}


# =============================================================================
# Final Notebook 08 completion validation
# =============================================================================

NOTEBOOK_08_TO_09_HANDOVER_VALID = all(
    [
        BLOCK_9_PATHWAY_DIMENSIONS_VALID,
        BLOCK_9_PARAMETER_COUNTS_VALID,
        BLOCK_9_TRAINED_PATHWAY_CONTRACT_VALID,

        BLOCK_9_REPRESENTATION_FROZEN,
        BLOCK_9_TRANSFORMATIVE_FROZEN,
        BLOCK_9_ALL_MODULES_EVAL,

        BLOCK_9_FORWARD_PATH_VALID,

        BLOCK_9_REPRESENTATION_UNCHANGED,
        BLOCK_9_TRANSFORMATIVE_UNCHANGED_DURING_VALIDATION,

        BLOCK_9_CHECKPOINT_BYTE_EXACT,
        BLOCK_9_METADATA_BYTE_EXACT,

        BLOCK_9_CHECKPOINT_CONTRACT_VALID,
        BLOCK_9_METADATA_CONTRACT_VALID,
        BLOCK_9_CROSS_ARTEFACT_VALID,

        BLOCK_9_REPRESENTATION_READBACK_EXACT,
        BLOCK_9_TRANSFORMATIVE_READBACK_EXACT,

        not BLOCK_9_TRANSFORMATIVE_WEIGHTS_CREATED,
        not BLOCK_9_RECURRENT_STATE_CREATED,
        not BLOCK_9_RECURRENT_UPDATE_EXECUTED,

        not BLOCK_9_LOSS_CALCULATED,
        not BLOCK_9_OPTIMIZER_CREATED,
        not BLOCK_9_BACKWARD_PASS_EXECUTED,
        not BLOCK_9_PARAMETER_UPDATE_EXECUTED,
        not BLOCK_9_REPRESENTATION_FORWARD_PASS_EXECUTED,

        BLOCK_9_TRANSFORM_FORWARD_PASS_EXECUTED,
    ]
)


NOTEBOOK_09_READY = (
    NOTEBOOK_08_TO_09_HANDOVER_VALID
)


NOTEBOOK_08_BLOCK_9_VALID = (
    NOTEBOOK_08_TO_09_HANDOVER_VALID
)


NOTEBOOK_08_BLOCK_9_COMPLETE = (
    NOTEBOOK_08_BLOCK_9_VALID
)


NOTEBOOK_08_COMPLETE = (
    NOTEBOOK_08_BLOCK_9_COMPLETE
)


if not NOTEBOOK_08_COMPLETE:

    raise RuntimeError(
        "Notebook 08 final validation or persistent "
        "Notebook 08 → Notebook 09 handover failed."
    )


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 08, Block 9: "
    "Final Transformative-Mechanism Validation and Handover"
)

print("=" * 72)

print(
    f"Block version               : "
    f"{NOTEBOOK_08_BLOCK_9_VERSION}"
)

print("-" * 72)

print(
    "Final architecture"
)

print(
    f"Representation parameters   : "
    f"{BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS:,}"
)

print(
    f"Transformative parameters   : "
    f"{BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS:,}"
)

print(
    f"Combined persisted params   : "
    f"{BLOCK_9_COMBINED_PARAMETER_COUNT:,}"
)

print("-" * 72)

print(
    "Transformative pathways"
)


print(
    f"Eligible pathways           : "
    f"{list(BLOCK_9_ELIGIBLE_PATHWAYS)}"
)

print(
    f"Trained pathways            : "
    f"{list(BLOCK_9_TRAINED_PATHWAYS)}"
)

for pathway_name in BLOCK_9_PATHWAY_DIMS:

    print(
        f"{pathway_name:<14} dimension          : "
        f"{BLOCK_9_PATHWAY_DIMS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} parameters         : "
        f"{BLOCK_9_TRANSFORMATIVE_PARAMETER_COUNTS[pathway_name]:,}"
    )

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{list(NOTEBOOK_08_TRANSFORMATIVE_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{list(NOTEBOOK_08_TRANSFORMATIVE_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

print("-" * 72)

print(
    "Post-training validation"
)

for pathway_name in BLOCK_9_PATHWAY_DIMS:

    validation = (
        BLOCK_9_FORWARD_VALIDATION[
            pathway_name
        ]
    )

    print(
        f"{pathway_name:<14} shape valid        : "
        f"{validation['shape_valid']}"
    )

    print(
        f"{pathway_name:<14} finite             : "
        f"{validation['finite']}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{validation['deterministic']}"
    )

print("-" * 72)

print(
    f"Representation frozen       : "
    f"{BLOCK_9_REPRESENTATION_FROZEN}"
)

print(
    f"Transformative frozen       : "
    f"{BLOCK_9_TRANSFORMATIVE_FROZEN}"
)

print(
    f"All modules in eval mode    : "
    f"{BLOCK_9_ALL_MODULES_EVAL}"
)

print(
    f"Representation unchanged    : "
    f"{BLOCK_9_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transform state unchanged   : "
    f"{BLOCK_9_TRANSFORMATIVE_UNCHANGED_DURING_VALIDATION}"
)

print("-" * 72)

print(
    "Persistent handover"
)

print(
    f"Checkpoint filename         : "
    f"{BLOCK_9_CHECKPOINT_FILENAME}"
)

print(
    f"Checkpoint Drive operation  : "
    f"{BLOCK_9_CHECKPOINT_DRIVE_OPERATION}"
)

print(
    f"Checkpoint Drive file ID    : "
    f"{NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID}"
)

print(
    f"Checkpoint size             : "
    f"{len(BLOCK_9_CHECKPOINT_BYTES) / (1024 ** 2):.3f} MB"
)

print(
    f"Checkpoint SHA-256 valid    : "
    f"{BLOCK_9_CHECKPOINT_BYTE_EXACT}"
)

print("-" * 72)

print(
    f"Metadata filename           : "
    f"{BLOCK_9_METADATA_FILENAME}"
)

print(
    f"Metadata Drive operation    : "
    f"{BLOCK_9_METADATA_DRIVE_OPERATION}"
)

print(
    f"Metadata Drive file ID      : "
    f"{NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID}"
)

print(
    f"Metadata size               : "
    f"{len(BLOCK_9_METADATA_BYTES) / 1024:.3f} KB"
)

print(
    f"Metadata SHA-256 valid      : "
    f"{BLOCK_9_METADATA_BYTE_EXACT}"
)

print("-" * 72)

print(
    f"Representation read-back    : "
    f"{BLOCK_9_REPRESENTATION_READBACK_EXACT}"
)

print(
    f"Transform read-back         : "
    f"{BLOCK_9_TRANSFORMATIVE_READBACK_EXACT}"
)

print(
    f"Checkpoint contract valid   : "
    f"{BLOCK_9_CHECKPOINT_CONTRACT_VALID}"
)

print(
    f"Metadata contract valid     : "
    f"{BLOCK_9_METADATA_CONTRACT_VALID}"
)

print(
    f"Cross-artefact valid        : "
    f"{BLOCK_9_CROSS_ARTEFACT_VALID}"
)

print("-" * 72)

print(
    f"Transform weights created   : "
    f"{BLOCK_9_TRANSFORMATIVE_WEIGHTS_CREATED}"
)

print(
    f"Recurrent state created     : "
    f"{BLOCK_9_RECURRENT_STATE_CREATED}"
)

print(
    f"Recurrent update executed   : "
    f"{BLOCK_9_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Loss calculated             : "
    f"{BLOCK_9_LOSS_CALCULATED}"
)

print(
    f"Optimizer created           : "
    f"{BLOCK_9_OPTIMIZER_CREATED}"
)

print(
    f"Parameter update executed   : "
    f"{BLOCK_9_PARAMETER_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Notebook 08 → 09 valid      : "
    f"{NOTEBOOK_08_TO_09_HANDOVER_VALID}"
)

print(
    f"Notebook 09 ready           : "
    f"{NOTEBOOK_09_READY}"
)

print(
    f"Block valid                 : "
    f"{NOTEBOOK_08_BLOCK_9_VALID}"
)

print(
    f"Block complete              : "
    f"{NOTEBOOK_08_BLOCK_9_COMPLETE}"
)

print(
    f"Notebook 08 complete        : "
    f"{NOTEBOOK_08_COMPLETE}"
)

print("=" * 72)

print(
    "The validated representation model and the Block 7-eligible trained "
    "transformative mechanisms were persisted successfully as a self-contained "
    "Notebook 08 → Notebook 09 handover."
)

print(
    f"The persisted checkpoint reproduces all "
    f"{BLOCK_9_TOTAL_REPRESENTATION_PARAMETERS:,} representation parameters "
    f"and all {BLOCK_9_TOTAL_TRANSFORMATIVE_PARAMETERS:,} transformative "
    "parameters exactly."
)

print(
    "The checkpoint and metadata were read back from Google Drive and "
    "validated against the in-memory Notebook 08 completion state."
)

print(
    "The Notebook 09 bootstrap contract contains direct Drive file IDs, "
    "so the next notebook does not need to rediscover upstream artefacts."
)

print(
    "Candidate transformations remain distinct from Transformative Weights."
)

print(
    "No Transformative Weight, recurrent state, recurrent update, loss, "
    "optimiser operation, backward pass or parameter update was introduced "
    "during final validation and persistence."
)

print(
    "Notebook 09 may initialise directly from the persisted Notebook 08 "
    "handover and begin the controlled Transformative-Weight and recurrent-"
    "state stage."
)

print("=" * 72)

Media AI — Notebook 08, Block 9: Final Transformative-Mechanism Validation and Handover
Block version               : 1.1
------------------------------------------------------------------------
Final architecture
Representation parameters   : 894,003
Transformative parameters   : 8,187
Combined persisted params   : 902,190
------------------------------------------------------------------------
Transformative pathways
Eligible pathways           : ['factual', 'psychological', 'social']
Trained pathways            : ['factual', 'psychological', 'social']
factual        dimension          : 10
factual        parameters         : 670
factual        active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
factual        deferred dimensions: []
psychological  dimension          : 34
psychological  parameters         : 7,174
psychological  active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
psychologi